##DOWNLOAD THE VTUAD INTO THE INPUTS FOLDER
##CREATE A CUSTOM ONC DATASET AND ALSO PLACE INTO INPUTS FOLDER

##PROJECT PATHS — SET ONCE, RUN FIRST

In [ ]:
from pathlib import Path

# =============== SET YOUR PATHS ONCE (run this cell before any other) ===============
BASE_DIR = Path("/Users/mandeepwalia/Downloads/Vessel-Classification-Representations-Architectures-and-Hyperparameters-main")  #CHANGE THIS: your project root

INPUTS = BASE_DIR / "Inputs"    # VTUAD range folders (2000_4000, ...) + ONC data live here
OUTPUTS = BASE_DIR / "Outputs"  # everything the pipeline produces is written here

# ONC classified-clips folder (holds metadata_1s.csv; clip paths are relative to it)
ONC_METADATA_ROOT = INPUTS / "ONC" / "07b_classified_wav_files" / "inclusion_2000_exclusion_4000"
# ====================================================================================

print(f"BASE_DIR = {BASE_DIR}")
print(f"INPUTS   = {INPUTS}   (exists: {INPUTS.is_dir()})")
print(f"OUTPUTS  = {OUTPUTS}")


##ONC DATASET SETTINGS

In [ ]:
#......

##2000_4000 processing

In [ ]:
# This script constructs a split within each class of each ship ID to prevent leakage


import csv
import shutil
from collections import defaultdict
from pathlib import Path


RANGE = "2000_4000"                        #CHANGE THIS to 3000_5000 / 4000_6000 for the other ranges
ROOT = INPUTS / RANGE                      # VTUAD range folder (from the paths cell at the top)
OUT = OUTPUTS / RANGE / f"{RANGE}_splits"  # ship-id folders written here
SPLITS = ["train", "validation", "test"]
MOVE = False
SPLIT_ORDER = {"train": 0, "validation": 1, "test": 2}


def mmsi_to_folder(mmsi_raw): #Normalize MMSI to integer for clean folder name
    s = str(mmsi_raw).strip()
    try:
        f = float(s)
        if f.is_integer():
            return str(int(f))
    except ValueError:
        pass
    return s.replace("/", "_")


def main():
    if not ROOT.is_dir():
        raise SystemExit(f"root not found: {ROOT}")

    # Key: Obtain (MMSI, Class)
    # Value: list of tuples: (split order, file index, and source_wav_path)
    groups = defaultdict(list)
    total_rows = 0
    missing = []

    for split in SPLITS:
        csv_path = ROOT / split / f"metadata_{split}.csv"
        audio_dir = ROOT / split / "audio"
        if not csv_path.is_file():
            raise SystemExit(f"metadata not found: {csv_path}")
        with open(csv_path, newline="") as f:
            reader = csv.DictReader(f)
            for row in reader:
                total_rows += 1
                cls = row["label"].strip()
                mmsi = mmsi_to_folder(row["MMSI"])
                fidx = row["file_index"].strip()
                src = audio_dir / cls / f"{fidx}.wav"
                if not src.is_file():
                    missing.append(str(src))
                    continue
                try:
                    order_key = (SPLIT_ORDER[split], int(fidx))
                except ValueError:
                    order_key = (SPLIT_ORDER[split], fidx)
                groups[(cls, mmsi)].append((order_key, src))

    print(f"read {total_rows} rows across {len(SPLITS)} splits")
    print(f"found {len(groups)} (class, ship) groups")
    if missing:
        print(f"WARNING: {len(missing)} rows had no matching .wav on disk "
              f"(first few): {missing[:5]}")

    OUT.mkdir(parents=True, exist_ok=True)


    per_class_ships = defaultdict(int)
    per_class_clips = defaultdict(int)
    grand_copied = 0

    for (cls, mmsi), items in sorted(groups.items()):
        items.sort(key=lambda x: x[0])
        dest_dir = OUT / f"{cls}_ship_ids" / f"ship_id_{mmsi}"
        dest_dir.mkdir(parents=True, exist_ok=True)
        for new_idx, (_, src) in enumerate(items):
            dest = dest_dir / f"{mmsi}_{new_idx}.wav"
            if MOVE:
                shutil.move(str(src), str(dest))
            else:
                shutil.copy2(str(src), str(dest))
            grand_copied += 1
        per_class_ships[cls] += 1
        per_class_clips[cls] += len(items)

    print("\n=== summary (per class) ===")
    print(f"{'class':16}{'ships':>7}{'clips':>9}")
    for cls in sorted(per_class_clips):
        print(f"{cls:16}{per_class_ships[cls]:>7}{per_class_clips[cls]:>9}")
    print("-" * 32)
    print(f"{'TOTAL':16}{sum(per_class_ships.values()):>7}{grand_copied:>9}")
    print(f"\n{'moved' if MOVE else 'copied'} {grand_copied} wav files -> {OUT}")


if __name__ == "__main__":
    main()

##Create Train 2000_4000

In [ ]:
#This script partitions ship ids within each class into a training dataset, ensuring no leakage into the validation or test sets.

import random
import shutil
from pathlib import Path


RANGE = "2000_4000"                        #CHANGE THIS to 3000_5000 / 4000_6000 for the other ranges
OUT = OUTPUTS / RANGE / f"{RANGE}_splits"  # step 1 output
TRAIN_DIR = OUTPUTS / RANGE / "train"
SEED = 42

TARGETS = {
    "background": 688,
    "cargo": 688,
    "passengership": 688,
    "tanker": 688,
    "tug": 688,
}
BACKGROUND_CLASS = "background"


def list_ship_dirs(class_dir):
    if not class_dir.is_dir():
        return []
    ships = [d for d in sorted(class_dir.iterdir())
             if d.is_dir() and d.name.startswith("ship_id_") and not d.name.endswith("_used")]
    return ships


def wavs_in(ship_dir):
    return sorted(p for p in ship_dir.iterdir()
                  if p.suffix.lower() == ".wav" and not p.stem.endswith("_used"))


def mark_file_used(p):
    target = p.with_name(p.stem + "_used.wav")
    if not target.exists():
        p.rename(target)


def build_background(class_dir, dest_dir, target, rng):
    ships = list_ship_dirs(class_dir)
    if not ships:
        print(f"[background] no available ship folders in {class_dir} -> skip", flush=True)
        return 0

    all_wavs = []
    for s in ships:
        all_wavs.extend(wavs_in(s))
    if not all_wavs:
        print(f"[background] no unused wavs in {class_dir} -> skip", flush=True)
        return 0

    pool_size = len(all_wavs) // 3          # 1/3 for train, 1/3 val, 1/3 test
    if pool_size < 1:
        pool_size = 1
    pool = rng.sample(all_wavs, pool_size)
    rng.shuffle(pool)

    print(f"[background] {len(ships)} ship id(s), {len(all_wavs)} unused clips -> "
          f"file pool floor({len(all_wavs)}/3)={pool_size}, target {target}", flush=True)

    take = min(target, len(pool))
    for p in pool[:take]:
        shutil.copy2(str(p), str(dest_dir / p.name))


    for p in pool:
        mark_file_used(p)

    status = "OK" if take == target else f"SHORT by {target - take}"
    print(f"    copied {take}/{target} clips from the file pool "
          f"({pool_size} files marked _used) [{status}]", flush=True)
    if take < target:
        print(f"    !! the 1/3 file pool holds only {pool_size} clips (< {target})", flush=True)
    return take


def main():
    rng = random.Random(SEED)
    if not OUT.is_dir():
        print(f"source outputs dir not found: {OUT}. Creating automatically")
        OUT.mkdir(parents=True, exist_ok=True)

    TRAIN_DIR.mkdir(parents=True, exist_ok=True)
    print(f"SEED = {SEED}\n", flush=True)

    grand_total = 0
    for cls, target in TARGETS.items():
        class_dir = OUT / f"{cls}_ship_ids"
        dest_dir = TRAIN_DIR / cls
        dest_dir.mkdir(parents=True, exist_ok=True)


        if cls == BACKGROUND_CLASS:
            grand_total += build_background(class_dir, dest_dir, target, rng)
            continue

        ships = list_ship_dirs(class_dir)
        if not ships:
            print(f"[{cls}] no available ship folders in {class_dir} -> skip", flush=True)
            continue


        n_ships = len(ships) // 3 #partition by 3, 1/3 for train, 1/3 for validation, 1/3 for test
        if n_ships < 1:
            n_ships = 1
        n_ships = min(n_ships, len(ships))

        chosen = rng.sample(ships, n_ships)            # random ship selection using random seed
        quota = target // n_ships

        # pre-load and shuffle each chosen ship's clips
        pool = {}
        for s in chosen:
            w = wavs_in(s)
            rng.shuffle(w)
            pool[s] = w

        print(f"[{cls}] {len(ships)} ships available -> using {n_ships} "
              f"(floor quota {quota}/ship), target {target}", flush=True)

        copied = 0
        taken = {s: 0 for s in chosen}

        # pass 1: take the floor quota from each chosen ship
        for s in chosen:
            avail = pool[s]
            take = min(quota, len(avail), target - copied)
            for p in avail[taken[s]: taken[s] + take]:
                shutil.copy2(str(p), str(dest_dir / p.name))
            taken[s] += take
            copied += take
            if copied >= target:
                break

        #We take whatever is left in other ship ids until target is met
        if copied < target:
            for s in chosen:
                if copied >= target:
                    break
                avail = pool[s]
                remaining_in_ship = avail[taken[s]:]
                need = target - copied
                take = min(need, len(remaining_in_ship))
                for p in remaining_in_ship[:take]:
                    shutil.copy2(str(p), str(dest_dir / p.name))
                taken[s] += take
                copied += take

        # mark each consumed ship folder as used
        for s in chosen:
            used_name = s.parent / f"{s.name}_used"
            if not used_name.exists():
                s.rename(used_name)

        status = "OK" if copied == target else f"SHORT by {target - copied}"
        print(f"    copied {copied}/{target} clips from {n_ships} ships "
              f"[{status}]", flush=True)
        if copied < target:
            print(f"    !! not enough clips across the {n_ships} selected ships to "
                  f"reach {target}; consider allocating more ships to this class", flush=True)
        grand_total += copied

    print(f"\nTRAIN dataset written to {TRAIN_DIR}")
    print(f"total clips copied: {grand_total}")


if __name__ == "__main__":
    main()

Val for 2000_4000

In [ ]:
#This script partitions ship ids within each class into a validation dataset, drawing only
#from ships/files not already consumed by the train build, so there is no
#leakage between train, validation, and test.

import random
import shutil
from pathlib import Path


RANGE = "2000_4000"                        #CHANGE THIS to 3000_5000 / 4000_6000 for the other ranges
OUT = OUTPUTS / RANGE / f"{RANGE}_splits"  # step 1 output
VAL_DIR = OUTPUTS / RANGE / "validation"
SEED = 42
DIVISOR = 2

TARGETS = {                        # required clip count per class in the validation set
    "background": 72,
    "cargo": 72,
    "passengership": 72,
    "tanker": 72,
    "tug": 72,
}
BACKGROUND_CLASS = "background"    # one ship id


def list_ship_dirs(class_dir):
    if not class_dir.is_dir():
        return []
    ships = [d for d in sorted(class_dir.iterdir())
             if d.is_dir() and d.name.startswith("ship_id_") and not d.name.endswith("_used")]
    return ships


def wavs_in(ship_dir):
    return sorted(p for p in ship_dir.iterdir()
                  if p.suffix.lower() == ".wav" and not p.stem.endswith("_used"))


def mark_file_used(p):
    target = p.with_name(p.stem + "_used.wav")
    if not target.exists():
        p.rename(target)


def build_background(class_dir, dest_dir, target, rng):
    ships = list_ship_dirs(class_dir)
    if not ships:
        print(f"[background] no available ship folders in {class_dir} -> skip", flush=True)
        return 0

    all_wavs = []
    for s in ships:
        all_wavs.extend(wavs_in(s))          # excludes train's _used files
    if not all_wavs:
        print(f"[background] no unused wavs left in {class_dir} -> skip", flush=True)
        return 0

    pool_size = len(all_wavs) // DIVISOR
    if pool_size < 1:
        pool_size = 1
    pool = rng.sample(all_wavs, pool_size)
    rng.shuffle(pool)

    print(f"[background] {len(ships)} ship id(s), {len(all_wavs)} unused clips -> "
          f"file pool floor({len(all_wavs)}/{DIVISOR})={pool_size}, target {target}", flush=True)

    take = min(target, len(pool))
    for p in pool[:take]:
        shutil.copy2(str(p), str(dest_dir / p.name))

    for p in pool:                            # mark the half used
        mark_file_used(p)

    status = "OK" if take == target else f"SHORT by {target - take}"
    print(f"    copied {take}/{target} clips from the file pool "
          f"({pool_size} files marked _used) [{status}]", flush=True)
    if take < target:
        print(f"    !! the 1/{DIVISOR} file pool holds only {pool_size} clips "
              f"(< {target})", flush=True)
    return take


def main():
    rng = random.Random(SEED)
    if not OUT.is_dir():
        raise SystemExit(f"source outputs dir not found: {OUT}")
    VAL_DIR.mkdir(parents=True, exist_ok=True)
    print(f"SEED = {SEED}\n", flush=True)

    grand_total = 0
    for cls, target in TARGETS.items():
        class_dir = OUT / f"{cls}_ship_ids"
        dest_dir = VAL_DIR / cls
        dest_dir.mkdir(parents=True, exist_ok=True)

        # background: single ship id
        if cls == BACKGROUND_CLASS:
            grand_total += build_background(class_dir, dest_dir, target, rng)
            continue

        ships = list_ship_dirs(class_dir)      # excludes train's _used ships
        if not ships:
            print(f"[{cls}] no available ship folders in {class_dir} -> skip", flush=True)
            continue

        # how many ships to use for validation
        n_ships = len(ships) // DIVISOR
        if n_ships < 1:
            n_ships = 1
        n_ships = min(n_ships, len(ships))

        chosen = rng.sample(ships, n_ships)            # random ship selection using random seed
        quota = target // n_ships

        # pre-load and shuffle each chosen ship's clips
        pool = {}
        for s in chosen:
            w = wavs_in(s)
            rng.shuffle(w)
            pool[s] = w

        print(f"[{cls}] {len(ships)} unused ships -> using {n_ships} "
              f"(floor quota {quota}/ship), target {target}", flush=True)

        copied = 0
        taken = {s: 0 for s in chosen}

        # pass 1: take the floor quota from each chosen ship
        for s in chosen:
            avail = pool[s]
            take = min(quota, len(avail), target - copied)
            for p in avail[taken[s]: taken[s] + take]:
                shutil.copy2(str(p), str(dest_dir / p.name))
            taken[s] += take
            copied += take
            if copied >= target:
                break

        # pass 2: fill the remaining shortfall from the other chosen ships,
        #We take whatever is left in other ship ids until target is met
        if copied < target:
            for s in chosen:
                if copied >= target:
                    break
                avail = pool[s]
                remaining_in_ship = avail[taken[s]:]
                need = target - copied
                take = min(need, len(remaining_in_ship))
                for p in remaining_in_ship[:take]:
                    shutil.copy2(str(p), str(dest_dir / p.name))
                taken[s] += take
                copied += take

        # mark each consumed ship folder as used
        for s in chosen:
            used_name = s.parent / f"{s.name}_used"
            if not used_name.exists():
                s.rename(used_name)

        status = "OK" if copied == target else f"SHORT by {target - copied}"
        print(f"    copied {copied}/{target} clips from {n_ships} ships "
              f"[{status}]", flush=True)
        if copied < target:
            print(f"    !! not enough clips across the {n_ships} selected ships to "
                  f"reach {target}; consider allocating more ships to this class", flush=True)
        grand_total += copied

    print(f"\nVALIDATION dataset written to {VAL_DIR}")
    print(f"total clips copied: {grand_total}")


if __name__ == "__main__":
    main()

##TEST for 2000_4000

In [ ]:
#This script partitions the remaining ship ids within each class into a test dataset. It draws
#only from ships/files not already consumed by the train and validation builds,
#completing a leakage-free three-way split.

import random
import shutil
from pathlib import Path


RANGE = "2000_4000"                        #CHANGE THIS to 3000_5000 / 4000_6000 for the other ranges
OUT = OUTPUTS / RANGE / f"{RANGE}_splits"  # step 1 output
TEST_DIR = OUTPUTS / RANGE / "test"
SEED = 42
DIVISOR = 1                        # final split

TARGETS = {                        # required clip count per class in the test set
    "background": 40,
    "cargo": 40,
    "passengership": 40,
    "tanker": 40,
    "tug": 40,
}
BACKGROUND_CLASS = "background"    # one ship id


def list_ship_dirs(class_dir):
    """Unused ship_id_* folders in a class folder (skips ones already marked _used)."""
    if not class_dir.is_dir():
        return []
    ships = [d for d in sorted(class_dir.iterdir())
             if d.is_dir() and d.name.startswith("ship_id_") and not d.name.endswith("_used")]
    return ships


def wavs_in(ship_dir):
    """Unused wavs in a ship folder (skips files already marked _used)."""
    return sorted(p for p in ship_dir.iterdir()
                  if p.suffix.lower() == ".wav" and not p.stem.endswith("_used"))


def mark_file_used(p):
    """Rename a wav to <stem>_used.wav (records what this split consumed)."""
    target = p.with_name(p.stem + "_used.wav")
    if not target.exists():
        p.rename(target)


def build_background(class_dir, dest_dir, target, rng):
    """Background has a single ship id, so its FILES are partitioned instead of its ships.
    Train and validation already marked their thirds/halves _used; test uses ALL remaining
    files, draws the target from them, and marks them _used."""
    ships = list_ship_dirs(class_dir)
    if not ships:
        print(f"[background] no available ship folders in {class_dir} -> skip", flush=True)
        return 0

    all_wavs = []
    for s in ships:
        all_wavs.extend(wavs_in(s))
    if not all_wavs:
        print(f"[background] no unused wavs left in {class_dir} -> skip", flush=True)
        return 0

    pool_size = len(all_wavs) // DIVISOR
    if pool_size < 1:
        pool_size = 1
    pool = rng.sample(all_wavs, pool_size)
    rng.shuffle(pool)

    print(f"[background] {len(ships)} ship id(s), {len(all_wavs)} unused clips -> "
          f"file pool (all) {pool_size}, target {target}", flush=True)

    take = min(target, len(pool))
    for p in pool[:take]:
        shutil.copy2(str(p), str(dest_dir / p.name))

    for p in pool:
        mark_file_used(p)

    status = "OK" if take == target else f"SHORT by {target - take}"
    print(f"    copied {take}/{target} clips from the file pool "
          f"({pool_size} files marked _used) [{status}]", flush=True)
    if take < target:
        print(f"    !! only {pool_size} clips remained (< {target}); this was the last split",
              flush=True)
    return take


def main():
    rng = random.Random(SEED)
    if not OUT.is_dir():
        raise SystemExit(f"source outputs dir not found: {OUT}")
    TEST_DIR.mkdir(parents=True, exist_ok=True)
    print(f"SEED = {SEED}\n", flush=True)

    grand_total = 0
    for cls, target in TARGETS.items():
        class_dir = OUT / f"{cls}_ship_ids"
        dest_dir = TEST_DIR / cls
        dest_dir.mkdir(parents=True, exist_ok=True)

        # background: single ship id
        if cls == BACKGROUND_CLASS:
            grand_total += build_background(class_dir, dest_dir, target, rng)
            continue

        ships = list_ship_dirs(class_dir)
        if not ships:
            print(f"[{cls}] no available ship folders in {class_dir} -> skip", flush=True)
            continue

        # how many ships to use for test
        n_ships = len(ships) // DIVISOR
        if n_ships < 1:
            n_ships = 1
        n_ships = min(n_ships, len(ships))

        chosen = rng.sample(ships, n_ships)            # random ship selection using random seed
        quota = target // n_ships

        # pre-load and shuffle each chosen ship's clips
        pool = {}
        for s in chosen:
            w = wavs_in(s)
            rng.shuffle(w)
            pool[s] = w

        print(f"[{cls}] {len(ships)} unused ships -> using {n_ships} "
              f"(floor quota {quota}/ship), target {target}", flush=True)

        copied = 0
        taken = {s: 0 for s in chosen}

        # pass 1: take the floor quota from each chosen ship
        for s in chosen:
            avail = pool[s]
            take = min(quota, len(avail), target - copied)
            for p in avail[taken[s]: taken[s] + take]:
                shutil.copy2(str(p), str(dest_dir / p.name))
            taken[s] += take
            copied += take
            if copied >= target:
                break

        # pass 2: fill the remaining shortfall from the other chosen ships,
        #We take whatever is left in other ship ids until target is met
        if copied < target:
            for s in chosen:
                if copied >= target:
                    break
                avail = pool[s]
                remaining_in_ship = avail[taken[s]:]
                need = target - copied
                take = min(need, len(remaining_in_ship))
                for p in remaining_in_ship[:take]:
                    shutil.copy2(str(p), str(dest_dir / p.name))
                taken[s] += take
                copied += take

        # mark each consumed ship folder as used
        for s in chosen:
            used_name = s.parent / f"{s.name}_used"
            if not used_name.exists():
                s.rename(used_name)

        status = "OK" if copied == target else f"SHORT by {target - copied}"
        print(f"    copied {copied}/{target} clips from {n_ships} ships "
              f"[{status}]", flush=True)
        if copied < target:
            print(f"    !! not enough clips across the {n_ships} remaining ships to "
                  f"reach {target}; this was the last split", flush=True)
        grand_total += copied

    print(f"\nTEST dataset written to {TEST_DIR}")
    print(f"total clips copied: {grand_total}")


if __name__ == "__main__":
    main()

##REPEAT THE STEPS FOR 3000_5000 AND 4000_6000, JUST MAKE SURE TO CHANGE THE FILE PATHS!

#ONC SETTINGS: (config.py)



First csv file:

begin,end,latitude,longitude,depth,location
2016-05-02T15:24:11.000Z,2016-07-24T08:33:27.000Z,49.080927,-123.338713,141.0,LSBBL

In [ ]:
MAX_INCLUSION_RADIUS=15000.0
INCLUSION_RADIUS=2000

UNIQUE_SCENARIOS=True

METADATA_SECONDS=1
METADATA_FILE="metadata"
METADATA_VAL_SPLIT=0.09
METADATA_TEST_SPLIT=0.05

# Pacific - Salish Sea - Strait of Georgia - Fraser River Delta (49.080927,-123.338713)
AIS_CODE = "DIGITALYACHTAISNET1302-0097-01"
WAV_DEVICES = [{"deviceCode": "ICLISTENAF2523"}, {"deviceCode": "ICLISTENAF2556"}]
CTD_DEVICE = "SBECTD19p6935"

# Define if the metadata will include ctd information. Only needed for step 10.
USE_CTD=False

##SPLIT SHIP IDS FOR ONC DATASET

In [ ]:
#This script groups the classified clips into class/MMSI folders and writes a
#manifest + summary CSV describing what was placed where.

import os
import shutil
import wave

import pandas as pd

from tqdm import tqdm


# ======================= EDIT THESE =======================
METADATA_ROOT = str(ONC_METADATA_ROOT)             # from the paths cell at the top
METADATA_FILE = "metadata_1s.csv"                  # metadata CSV file name inside METADATA_ROOT
OUTPUT_DIR = str(OUTPUTS / "ONC" / "ship_splits")  # class/MMSI folders written here

UNIT = "file"       # "file" = place whole WAV files, "segment" = cut into fixed length segments
MODE = "hardlink"   # "hardlink", "symlink" or "copy" — only used when UNIT is "file"
SECONDS = 1         # duration of each segment — only used when UNIT is "segment"
# ==========================================================


BACKGROUND_LABEL = "background"
# The background class has no vessel identity, so every background clip is
# stored under a single MMSI folder.
BACKGROUND_MMSI = 0


class bcolors:
    HEADER = "\033[95m"
    WARNING = "\033[93m"
    ENDC = "\033[0m"


def create_dir(parent, name):
    directory = os.path.join(parent, name)
    os.makedirs(directory, exist_ok=True)
    return directory


def get_mmsi_folder_name(mmsi):
    return str(int(mmsi))


def get_clips_from_metadata(metadata_file):
    """Categorise every clip in the metadata by its class and MMSI.

    Returns one row per WAV file with the label, the MMSI folder name it
    belongs to and the 1 second offsets that the metadata lists for it.
    """
    metadata = pd.read_csv(metadata_file)

    clips = []
    for path, rows in metadata.groupby("path"):
        labels = rows.label.unique()
        mmsis = rows.MMSI.unique()

        if len(labels) > 1:
            print(f"{bcolors.WARNING}Skipping {path}: more than one label {labels}{bcolors.ENDC}")
            continue
        if len(mmsis) > 1:
            print(f"{bcolors.WARNING}Skipping {path}: more than one MMSI {mmsis}{bcolors.ENDC}")
            continue

        label = labels[0]
        mmsi = BACKGROUND_MMSI if label == BACKGROUND_LABEL else mmsis[0]

        clips.append(
            {
                "label": label,
                "mmsi": get_mmsi_folder_name(mmsi),
                "path": path,
                "sub_inits": sorted(rows.sub_init.tolist()),
            }
        )

    return clips


def place_wav_file(source_file, destination_file, mode):
    if os.path.exists(destination_file):
        os.remove(destination_file)

    if mode == "hardlink":
        os.link(source_file, destination_file)
    elif mode == "symlink":
        os.symlink(os.path.abspath(source_file), destination_file)
    else:
        shutil.copyfile(source_file, destination_file)


def save_wav_segments(source_file, destination_directory, sub_inits, seconds):
    """Cut the clip into the fixed length segments listed in the metadata."""
    saved = []
    file_name = os.path.splitext(os.path.basename(source_file))[0]

    with wave.open(source_file, "rb") as source_wav:
        frame_rate = source_wav.getframerate()
        segment_frames = frame_rate * seconds
        total_frames = source_wav.getnframes()

        for sub_init in sub_inits:
            start_frame = sub_init * frame_rate
            if start_frame + segment_frames > total_frames:
                print(
                    f"{bcolors.WARNING}Skipping {source_file} at {sub_init}s: "
                    f"beyond the end of the file{bcolors.ENDC}"
                )
                continue

            source_wav.setpos(start_frame)
            frames = source_wav.readframes(segment_frames)

            destination_file = os.path.join(
                destination_directory, f"{file_name}_{sub_init:05d}.wav"
            )
            with wave.open(destination_file, "wb") as destination_wav:
                destination_wav.setnchannels(source_wav.getnchannels())
                destination_wav.setsampwidth(source_wav.getsampwidth())
                destination_wav.setframerate(frame_rate)
                destination_wav.writeframes(frames)

            saved.append((destination_file, sub_init))

    return saved


def group_clips_by_mmsi(metadata_root, metadata_file, output_directory, unit, mode, seconds):
    clips = get_clips_from_metadata(os.path.join(metadata_root, metadata_file))
    print(f"Categorised {len(clips)} clips by class and MMSI")

    manifest = []
    for clip in tqdm(clips, total=len(clips)):
        # The metadata stores the clip paths relative to the metadata folder.
        source_file = os.path.join(metadata_root, clip["path"])
        if not os.path.exists(source_file):
            print(f"{bcolors.WARNING}Skipping {clip['path']}: file not found{bcolors.ENDC}")
            continue

        # Every MMSI folder lives inside the folder of its own class.
        label_directory = create_dir(output_directory, clip["label"])
        mmsi_directory = create_dir(label_directory, clip["mmsi"])

        if unit == "segment":
            saved = save_wav_segments(source_file, mmsi_directory, clip["sub_inits"], seconds)
        else:
            destination_file = os.path.join(mmsi_directory, os.path.basename(source_file))
            place_wav_file(source_file, destination_file, mode)
            saved = [(destination_file, None)]

        for destination_file, sub_init in saved:
            manifest.append(
                {
                    "label": clip["label"],
                    "mmsi": clip["mmsi"],
                    "group_id": f"{clip['label']}/{clip['mmsi']}",
                    "source_path": clip["path"],
                    "sub_init": sub_init,
                    "path": os.path.relpath(destination_file, output_directory),
                }
            )

    return pd.DataFrame(manifest)


def save_manifest(manifest, output_directory):
    manifest_file = os.path.join(output_directory, "grouped_manifest.csv")
    manifest.to_csv(manifest_file, index=False)

    summary = (
        manifest.groupby("label")
        .agg(mmsi_folders=("mmsi", "nunique"), clips=("path", "count"))
        .sort_values("clips", ascending=False)
    )
    summary_file = os.path.join(output_directory, "grouped_summary.csv")
    summary.to_csv(summary_file)

    print(f"\n{summary.to_string()}")
    print(f"\nTotal MMSI folders: {manifest.group_id.nunique()}")
    print(f"Manifest saved to {manifest_file}")
    print(f"Summary saved to {summary_file}")


def main():
    if not os.path.isdir(METADATA_ROOT):
        raise SystemExit(f"metadata root not found: {METADATA_ROOT}")
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print(f"\n{bcolors.HEADER}Grouping clips by class and MMSI{bcolors.ENDC}")
    print(f"Reading {os.path.join(METADATA_ROOT, METADATA_FILE)}")
    print(f"Writing to {OUTPUT_DIR}")

    manifest = group_clips_by_mmsi(
        METADATA_ROOT,
        METADATA_FILE,
        OUTPUT_DIR,
        UNIT,
        MODE,
        SECONDS,
    )
    save_manifest(manifest, OUTPUT_DIR)


if __name__ == "__main__":
    main()


##CREATE TRAIN-VAL-TEST SPLIT

In [ ]:
#This script builds one train/validation/test split from the class/MMSI grouped
#clips. The units it draws from are marked _used so the splits built afterwards
#never reuse the same ship. Run it once per split, changing SPLIT each time
#(train first, then validation, then test).

import os
import wave

import pandas as pd


# ======================= EDIT THESE =======================
GROUPED_DIR = str(OUTPUTS / "ONC" / "ship_splits")       # the previous cell's OUTPUT_DIR
METADATA_CSV = str(ONC_METADATA_ROOT / "metadata_1s.csv")
SPLITS_DIR = str(OUTPUTS / "ONC" / "dataset_splits")     # split folders + manifests written here

SPLIT = "train"     # which split to build: "train", "validation" or "test". TYPE IN ALL THREE SPLITS
PARTITIONS = None   # how many partitions the available units are divided into; None = the default for the chosen split (train 3, validation 2, test 1)
SECONDS = 1         # duration of each segment
# ==========================================================


BACKGROUND_LABEL = "background"
USED_MARKER = "_used"

# The number of segments that each class contributes to every split.
SPLIT_QUOTAS = {
    "train": {
        "background": 1376,
        "cargo": 688,
        "passengership": 688,
        "tanker": 688,
        "tug": 688,
    },
    "validation": {
        "background": 144,
        "cargo": 72,
        "passengership": 72,
        "tanker": 72,
        "tug": 72,
    },
    "test": {
        "background": 80,
        "cargo": 40,
        "passengership": 40,
        "tanker": 40,
        "tug": 40,
    },
}

# How many partitions the available units are divided into for every split. The
# split is drawn from the first partition and the rest is kept for the splits
# that are built afterwards.
SPLIT_PARTITIONS = {
    "train": 3,
    "validation": 2,
    # The test split is the last one, so there is nothing left to reserve.
    "test": 1,
}


class bcolors:
    HEADER = "\033[95m"
    OKBLUE = "\033[94m"
    WARNING = "\033[93m"
    FAIL = "\033[91m"
    ENDC = "\033[0m"


def create_dir(parent, name):
    directory = os.path.join(parent, name)
    os.makedirs(directory, exist_ok=True)
    return directory


def is_used(name):
    # The marker sits on the folder name for vessels and before the extension
    # for the background clips.
    return os.path.splitext(name)[0].endswith(USED_MARKER)


def get_sort_key(unit):
    """Sort the units numerically so the partitions are reproducible."""
    stem = os.path.splitext(os.path.basename(unit))[0]
    return (0, int(stem)) if stem.isdigit() else (1, unit)


def get_units_from_manifest(manifest, metadata):
    """Build the pool of drawable units for every class.

    The vessel classes are drawn from their MMSI folders. The background class
    has a single MMSI folder, so it is partitioned by clip instead.
    """
    segments = (
        metadata.sort_values(["path", "sub_init"])
        .groupby("path")
        .sub_init.apply(list)
        .to_dict()
    )

    units = {}
    for label, rows in manifest.groupby("label"):
        pool = {}
        for row in rows.itertuples(index=False):
            if any(is_used(part) for part in row.path.split(os.sep)):
                continue

            unit = os.path.basename(row.path) if label == BACKGROUND_LABEL else row.mmsi
            for sub_init in segments.get(row.source_path, []):
                pool.setdefault(str(unit), []).append((row.path, sub_init))

        units[label] = dict(sorted(pool.items(), key=lambda item: get_sort_key(item[0])))

    return units


def draw_evenly(pool, ordered_units, quota, cursors, drawn):
    """Take one segment at a time from each unit in turn until the quota is met."""
    while len(drawn) < quota:
        progressed = False
        for unit in ordered_units:
            if len(drawn) >= quota:
                break
            cursor = cursors[unit]
            if cursor < len(pool[unit]):
                path, sub_init = pool[unit][cursor]
                drawn.append((unit, path, sub_init))
                cursors[unit] = cursor + 1
                progressed = True
        if not progressed:
            break

    return drawn


def draw_from_class(pool, quota, partition_count):
    """Draw the quota from the first partition of units, borrowing if needed."""
    names = list(pool.keys())
    size = len(names) // partition_count
    partitions = [names[index * size : (index + 1) * size] for index in range(partition_count - 1)]
    partitions.append(names[(partition_count - 1) * size :])

    cursors = {unit: 0 for unit in names}
    drawn = []
    ordered = []
    for index, partition in enumerate(partitions):
        if not partition:
            continue
        ordered = ordered + partition
        draw_evenly(pool, ordered, quota, cursors, drawn)
        if len(drawn) >= quota:
            break
        if index < len(partitions) - 1:
            print(
                f"{bcolors.WARNING}  quota not met from partition {index + 1}, "
                f"borrowing from partition {index + 2}{bcolors.ENDC}"
            )

    return drawn, size


def save_segments(grouped_directory, output_directory, label, drawn, seconds):
    """Cut and save every drawn segment, opening each source clip only once."""
    by_clip = {}
    for unit, path, sub_init in drawn:
        by_clip.setdefault((unit, path), []).append(sub_init)

    saved = []
    for (unit, path), sub_inits in by_clip.items():
        source_file = os.path.join(grouped_directory, path)
        destination_directory = create_dir(
            create_dir(output_directory, label), unit.replace(".wav", "")
        )
        file_name = os.path.splitext(os.path.basename(path))[0]

        with wave.open(source_file, "rb") as source_wav:
            frame_rate = source_wav.getframerate()
            segment_frames = frame_rate * seconds

            for sub_init in sorted(sub_inits):
                source_wav.setpos(sub_init * frame_rate)
                frames = source_wav.readframes(segment_frames)

                destination_file = os.path.join(
                    destination_directory, f"{file_name}_{sub_init:05d}.wav"
                )
                with wave.open(destination_file, "wb") as destination_wav:
                    destination_wav.setnchannels(source_wav.getnchannels())
                    destination_wav.setsampwidth(source_wav.getsampwidth())
                    destination_wav.setframerate(frame_rate)
                    destination_wav.writeframes(frames)

                saved.append(
                    {
                        "label": label,
                        "unit": unit,
                        "source_path": path,
                        "sub_init": sub_init,
                        "path": os.path.relpath(destination_file, output_directory),
                    }
                )

    return saved


def mark_used(grouped_directory, label, used_units):
    """Rename every unit that was drawn from so it is never reused."""
    renamed = {}
    for unit in used_units:
        if label == BACKGROUND_LABEL:
            matches = [
                os.path.join(root, unit)
                for root, _, files in os.walk(os.path.join(grouped_directory, label))
                if unit in files
            ]
            if not matches:
                continue
            current = matches[0]
            stem, extension = os.path.splitext(current)
            target = f"{stem}{USED_MARKER}{extension}"
        else:
            current = os.path.join(grouped_directory, label, unit)
            target = f"{current}{USED_MARKER}"

        if os.path.exists(current) and not os.path.exists(target):
            os.rename(current, target)
        renamed[os.path.relpath(current, grouped_directory)] = os.path.relpath(
            target, grouped_directory
        )

    return renamed


def build_split(
    grouped_directory, metadata_file, output_directory, quotas, partition_count, seconds
):
    manifest = pd.read_csv(os.path.join(grouped_directory, "grouped_manifest.csv"))
    manifest["mmsi"] = manifest.mmsi.astype(str)
    metadata = pd.read_csv(metadata_file)

    units = get_units_from_manifest(manifest, metadata)

    split = []
    renamed = {}
    report = []
    for label, quota in quotas.items():
        pool = units.get(label, {})
        if not pool:
            print(f"{bcolors.FAIL}No units available for {label}{bcolors.ENDC}")
            continue

        print(f"\n{bcolors.OKBLUE}{label}{bcolors.ENDC}: {len(pool)} units, quota {quota}")
        drawn, partition_size = draw_from_class(pool, quota, partition_count)
        if len(drawn) < quota:
            print(
                f"{bcolors.FAIL}  only {len(drawn)} of {quota} segments available"
                f"{bcolors.ENDC}"
            )

        used_units = sorted({unit for unit, _, _ in drawn}, key=get_sort_key)
        saved = save_segments(grouped_directory, output_directory, label, drawn, seconds)
        split.extend(saved)
        renamed.update(mark_used(grouped_directory, label, used_units))

        borrowed = len(used_units) - min(len(used_units), partition_size)
        print(
            f"  drew {len(drawn)} segments from {len(used_units)} units "
            f"(first partition = {partition_size}, borrowed = {borrowed})"
        )
        report.append(
            {
                "label": label,
                "quota": quota,
                "segments": len(drawn),
                "units_total": len(pool),
                "units_in_first_partition": partition_size,
                "units_used": len(used_units),
                "units_borrowed": borrowed,
                "segments_per_unit_min": min(
                    (sum(1 for u, _, _ in drawn if u == unit) for unit in used_units),
                    default=0,
                ),
                "segments_per_unit_max": max(
                    (sum(1 for u, _, _ in drawn if u == unit) for unit in used_units),
                    default=0,
                ),
            }
        )

    # Keep the grouped manifest pointing at the renamed folders.
    if renamed:
        manifest["path"] = [
            next((v for k, v in renamed.items() if p == k or p.startswith(k + os.sep)), p)
            for p in manifest.path
        ]
        manifest.to_csv(os.path.join(grouped_directory, "grouped_manifest.csv"), index=False)

    return pd.DataFrame(split), pd.DataFrame(report)


def main():
    if SPLIT not in SPLIT_QUOTAS:
        raise SystemExit(f"SPLIT must be one of {list(SPLIT_QUOTAS)}, got {SPLIT!r}")
    if not os.path.isdir(GROUPED_DIR):
        raise SystemExit(f"grouped clips dir not found: {GROUPED_DIR}")
    if not os.path.isfile(METADATA_CSV):
        raise SystemExit(f"metadata csv not found: {METADATA_CSV}")

    os.makedirs(SPLITS_DIR, exist_ok=True)
    output_directory = create_dir(SPLITS_DIR, SPLIT)

    partition_count = PARTITIONS or SPLIT_PARTITIONS[SPLIT]

    print(f"\n{bcolors.HEADER}Building the {SPLIT} split{bcolors.ENDC}")
    print(f"Reading {GROUPED_DIR}")
    print(f"Writing to {output_directory}")
    print(f"Dividing the available units into {partition_count} partitions")

    split, report = build_split(
        GROUPED_DIR,
        METADATA_CSV,
        output_directory,
        SPLIT_QUOTAS[SPLIT],
        partition_count,
        SECONDS,
    )

    manifest_file = os.path.join(SPLITS_DIR, f"{SPLIT}_manifest.csv")
    split.to_csv(manifest_file, index=False)
    report.to_csv(os.path.join(SPLITS_DIR, f"{SPLIT}_report.csv"), index=False)

    print(f"\n{report.to_string(index=False)}")
    print(f"\nTotal {SPLIT} segments: {len(split)}")
    print(f"Manifest saved to {manifest_file}")


if __name__ == "__main__":
    main()


##CREATE CUMULATIVE DATASET

In [ ]:
import os
import shutil

SOURCE_DATASETS = ['2000_4000', '3000_5000', '4000_6000', 'ONC']


def create_cumulative_dataset(source_root, output_root=None):
    output_root = source_root + "/cumulative_dataset"
    splits = ['train', 'test', 'validation']

    for split in splits:
        os.makedirs(os.path.join(output_root, split), exist_ok=True)

    dataset_folders = [d for d in SOURCE_DATASETS
                       if os.path.isdir(os.path.join(source_root, d))]
    missing = [d for d in SOURCE_DATASETS if d not in dataset_folders]
    if missing:
        print(f"WARNING: source folders not found: {missing}")
    if not dataset_folders:
        raise SystemExit(f"none of {SOURCE_DATASETS} found under {source_root}")

    print(f"Processing datasets: {dataset_folders}")

    total_copied = 0
    per_dataset = {}
    per_class = {}

    for dataset_idx, dataset_name in enumerate(dataset_folders):
        print(f"  Processing dataset {dataset_idx + 1}/{len(dataset_folders)}: "
              f"{dataset_name}")
        dataset_path = os.path.join(source_root, dataset_name)
        dataset_total = 0

        for split in splits:
            split_path = os.path.join(dataset_path, split)
            if not os.path.exists(split_path):
                print(f"    Skipping split '{split}': path does not exist.")
                continue
            print(f"    Processing split: {split}")

            try:
                class_names = sorted(os.listdir(split_path))
            except Exception as e:
                print(f"      ERROR: could not list {split_path}: {e}")
                continue

            for class_name in class_names:
                class_dir = os.path.join(split_path, class_name)
                if not os.path.isdir(class_dir):
                    continue

                dest_class_path = os.path.join(output_root, split, class_name)
                os.makedirs(dest_class_path, exist_ok=True)

                try:
                    files_in_class = sorted(
                        f for f in os.listdir(class_dir)
                        if os.path.isfile(os.path.join(class_dir, f))
                        and os.path.splitext(f)[1].lower() == '.wav')
                except Exception as e:
                    print(f"        ERROR: could not list {class_dir}: {e}")
                    continue

                print(f"      Processing class: {class_name} "
                      f"({len(files_in_class)} files)")

                for file_index, filename in enumerate(files_in_class):
                    src_file = os.path.join(class_dir, filename)

                    if file_index % 500 == 0 and file_index > 0:
                        print(f"        {file_index}/{len(files_in_class)}")

                    clip_index = 0
                    extension = os.path.splitext(filename)[1]
                    new_name = (f"{dataset_name}_{split}_{class_name}_"
                                f"{file_index}_{clip_index}{extension}")
                    dst_file = os.path.join(dest_class_path, new_name)

                    if os.path.exists(dst_file) and os.path.getsize(dst_file) > 0:
                        continue
                    shutil.copy2(src_file, dst_file)

                n = len(files_in_class)
                dataset_total += n
                total_copied += n
                per_class[(split, class_name)] = per_class.get(
                    (split, class_name), 0) + n

        per_dataset[dataset_name] = dataset_total

    print()
    print("Files per dataset:")
    for name in dataset_folders:
        print(f"  {name:<12} {per_dataset.get(name, 0):>7}")

    classes = sorted({c for _, c in per_class})
    print()
    print(f"{'split':<12}" + "".join(f"{c:>16}" for c in classes) + f"{'total':>9}")
    for split in splits:
        row = "".join(f"{per_class.get((split, c), 0):>16}" for c in classes)
        print(f"{split:<12}{row}"
              f"{sum(per_class.get((split, c), 0) for c in classes):>9}")
    print(f"{'TOTAL':<12}"
          + "".join(f"{sum(per_class.get((s, c), 0) for s in splits):>16}"
                    for c in classes)
          + f"{total_copied:>9}")

    print()
    print(f"Successfully created cumulative dataset at: "
          f"{os.path.abspath(output_root)}")


root_source = str(OUTPUTS)   # from the paths cell at the top
create_cumulative_dataset(root_source)


##CREATE MEL DATASET

In [ ]:
#This script converts the wav dataset into mel spectrogram .npy arrays, mirroring
#the <split>/<class> folder structure: INPUT_DIR/<split>/<class>/*.wav ->
#OUTPUT_DIR/<split>/<class>/*.npy

import sys
import time
from multiprocessing import Pool, cpu_count
from pathlib import Path

import numpy as np

# ======================= EDIT THESE =======================
INPUT_DIR = OUTPUTS / "cumulative_dataset"   # folder holding <split>/<class>/*.wav
OUTPUT_DIR = OUTPUTS / "mel_dataset"         # where the mirrored .npy folders will be written

SPLITS = ["train", "validation", "test"]

SR = 16000
N_FFT = 1024
WIN_LENGTH = 1024
HOP_LENGTH = 128
N_MELS = 128
FMIN = 0
FMAX = SR // 2
CLIP_SAMPLES = SR          # 1 second clips

LOG_SCALE = True
MINMAX_01 = True
DTYPE = np.float32

WORKERS = max(1, cpu_count() - 1)   # set to 1 if multiprocessing gives trouble in the notebook
# ==========================================================


def mel_from_wav(wav_path):
    import librosa

    y, _ = librosa.load(str(wav_path), sr=SR, mono=True)
    if len(y) < CLIP_SAMPLES:
        y = np.pad(y, (0, CLIP_SAMPLES - len(y)))
    else:
        y = y[:CLIP_SAMPLES]

    S = librosa.feature.melspectrogram(
        y=y, sr=SR, n_fft=N_FFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH,
        window="hann", center=True, power=2.0,
        n_mels=N_MELS, fmin=FMIN, fmax=FMAX,
    )

    if LOG_SCALE:
        S = librosa.power_to_db(S, ref=np.max)

    if MINMAX_01:
        lo, hi = float(S.min()), float(S.max())
        S = (S - lo) / (hi - lo) if hi > lo else np.zeros_like(S)

    return S.astype(DTYPE)


def process_one(job):
    src, dst = job
    try:
        arr = mel_from_wav(src)
        np.save(dst, arr)
        return True, arr.shape
    except Exception as exc:
        return False, f"{src}: {exc}"


def collect_jobs():
    jobs = []
    per_split = {}
    for split in SPLITS:
        split_dir = INPUT_DIR / split
        if not split_dir.is_dir():
            print(f"  !! split folder missing: {split_dir}", flush=True)
            continue
        counts = {}
        for class_dir in sorted(p for p in split_dir.iterdir() if p.is_dir()):
            out_dir = OUTPUT_DIR / split / class_dir.name
            out_dir.mkdir(parents=True, exist_ok=True)
            wavs = sorted(class_dir.glob("*.wav"))
            for w in wavs:
                jobs.append((w, out_dir / f"{w.stem}.npy"))
            counts[class_dir.name] = len(wavs)
        per_split[split] = counts
    return jobs, per_split


def main():
    try:
        import librosa
    except ImportError:
        sys.exit("this script needs librosa:  pip install librosa soundfile")

    if not INPUT_DIR.is_dir():
        raise SystemExit(f"input dir not found: {INPUT_DIR}")

    print(f"mel config: n_fft {N_FFT} | win {WIN_LENGTH} | hop {HOP_LENGTH} | "
          f"n_mels {N_MELS} | sr {SR}", flush=True)
    print(f"  -> {1 + CLIP_SAMPLES // HOP_LENGTH} frames per 1s clip "
          f"(arrays are {N_MELS} x {1 + CLIP_SAMPLES // HOP_LENGTH})", flush=True)
    print(f"  log_scale={LOG_SCALE}  minmax01={MINMAX_01}  dtype={np.dtype(DTYPE).name}\n",
          flush=True)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    jobs, per_split = collect_jobs()
    if not jobs:
        raise SystemExit(f"no .wav files found under {INPUT_DIR}")

    print("source clips:")
    for split, counts in per_split.items():
        total = sum(counts.values())
        detail = "  ".join(f"{c}={n}" for c, n in sorted(counts.items()))
        print(f"  {split:11} {total:>6}   {detail}", flush=True)
    print(f"  {'TOTAL':11} {len(jobs):>6}\n", flush=True)

    print(f"generating {len(jobs)} mel arrays on {WORKERS} workers...", flush=True)
    t0 = time.time()
    done, failed, shapes = 0, [], set()

    def handle(ok, info):
        if ok:
            shapes.add(info)
        else:
            failed.append(info)

    if WORKERS == 1:
        for job in jobs:
            ok, info = process_one(job)
            handle(ok, info)
            done += 1
            if done % 1000 == 0 or done == len(jobs):
                rate = done / max(time.time() - t0, 1e-9)
                print(f"  {done}/{len(jobs)}  ({rate:.0f} clips/s)", flush=True)
    else:
        with Pool(WORKERS) as pool:
            for ok, info in pool.imap_unordered(process_one, jobs, chunksize=32):
                handle(ok, info)
                done += 1
                if done % 1000 == 0 or done == len(jobs):
                    rate = done / max(time.time() - t0, 1e-9)
                    print(f"  {done}/{len(jobs)}  ({rate:.0f} clips/s)", flush=True)
    print(f"  finished in {time.time() - t0:.0f}s", flush=True)

    print(f"\narray shapes produced: {sorted(shapes)}", flush=True)
    if len(shapes) > 1:
        print("  !! more than one shape - check for clips that were not 1s", flush=True)
    if failed:
        print(f"  !! {len(failed)} clip(s) failed:", flush=True)
        for msg in failed[:10]:
            print(f"     {msg}", flush=True)

    n_npy = sum(1 for _ in OUTPUT_DIR.rglob("*.npy"))
    size_mb = sum(p.stat().st_size for p in OUTPUT_DIR.rglob("*.npy")) / 1e6
    print(f"\nwrote {n_npy} .npy files ({size_mb:.0f} MB) under {OUTPUT_DIR}", flush=True)
    print(f"  -> {OUTPUT_DIR}/<split>/<class>/*.npy")


if __name__ == "__main__":
    main()


##CREATE MCG DATASET

In [ ]:
#This script converts the wav dataset into 3-channel mel/CQT/gammatone .npy
#arrays (CHW), mirroring the <split>/<class> folder structure:
#INPUT_DIR/<split>/<class>/*.wav -> OUTPUT_DIR/<split>/<class>/*.npy

import sys
import time
from multiprocessing import Pool, cpu_count
from pathlib import Path

import numpy as np

# ======================= EDIT THESE =======================
INPUT_DIR = OUTPUTS / "cumulative_dataset"   # folder holding <split>/<class>/*.wav
OUTPUT_DIR = OUTPUTS / "mcg_dataset"         # where the mirrored .npy folders will be written

SPLITS = ["train", "validation", "test"]

SR = 16000
N_FFT = 1024
WIN_LENGTH = 1024
HOP_LENGTH = 128
CLIP_SAMPLES = SR                # 1 second clips
N_FRAMES = 1 + CLIP_SAMPLES // HOP_LENGTH

N_MELS = 128
MEL_FMIN, MEL_FMAX = 0, SR // 2

CQT_N_BINS = 128
CQT_BINS_PER_OCTAVE = 16
CQT_FMIN = 30.0

N_GAMMA = 128
GAMMA_FMIN, GAMMA_FMAX = 30.0, SR // 2
GAMMA_ORDER = 4

TOP_DB = 80.0
DTYPE = np.float32

WORKERS = max(1, cpu_count() - 1)   # set to 1 if multiprocessing gives trouble in the notebook
# ==========================================================


def _erb(f):
    return 24.7 * (4.37 * f / 1000.0 + 1.0)


def _erb_space(fmin, fmax, n):
    ear_q, min_bw = 9.26449, 24.7
    idx = np.arange(1, n + 1)
    cf = -(ear_q * min_bw) + np.exp(
        idx * (-np.log(fmax + ear_q * min_bw) + np.log(fmin + ear_q * min_bw)) / n
    ) * (fmax + ear_q * min_bw)
    return cf[::-1]


def gammatone_weights():
    freqs = np.fft.rfftfreq(N_FFT, 1.0 / SR)
    cf = _erb_space(GAMMA_FMIN, GAMMA_FMAX, N_GAMMA)
    bw = 1.019 * _erb(cf)
    w = (1.0 + ((freqs[None, :] - cf[:, None]) / bw[:, None]) ** 2) ** (-GAMMA_ORDER / 2.0)
    w /= np.maximum(w.max(axis=1, keepdims=True), 1e-10)
    return w


_GAMMA_W = None


def _gamma_w():
    global _GAMMA_W
    if _GAMMA_W is None:
        _GAMMA_W = gammatone_weights()
    return _GAMMA_W


def to_unit_db(power, ref_max=True):
    import librosa
    db = librosa.power_to_db(np.maximum(power, 1e-10),
                             ref=np.max if ref_max else 1.0, top_db=TOP_DB)
    lo, hi = float(db.min()), float(db.max())
    if hi <= lo:
        return np.zeros_like(db)
    return (db - lo) / (hi - lo)


def mcg_from_wav(wav_path):
    import librosa

    y, _ = librosa.load(str(wav_path), sr=SR, mono=True)
    if len(y) < CLIP_SAMPLES:
        y = np.pad(y, (0, CLIP_SAMPLES - len(y)))
    else:
        y = y[:CLIP_SAMPLES]

    stft = librosa.stft(y, n_fft=N_FFT, hop_length=HOP_LENGTH,
                        win_length=WIN_LENGTH, window="hann", center=True)
    power = np.abs(stft) ** 2

    mel = librosa.feature.melspectrogram(S=power, sr=SR, n_mels=N_MELS,
                                         fmin=MEL_FMIN, fmax=MEL_FMAX)
    gam = _gamma_w() @ power

    cqt = np.abs(librosa.cqt(y, sr=SR, hop_length=HOP_LENGTH, fmin=CQT_FMIN,
                             n_bins=CQT_N_BINS,
                             bins_per_octave=CQT_BINS_PER_OCTAVE)) ** 2

    t = min(mel.shape[1], cqt.shape[1], gam.shape[1])
    mel, cqt, gam = mel[:, :t], cqt[:, :t], gam[:, :t]

    chans = [to_unit_db(mel), to_unit_db(cqt), to_unit_db(gam)]
    return np.stack(chans, axis=0).astype(DTYPE)


def process_one(job):
    src, dst = job
    try:
        arr = mcg_from_wav(src)
        np.save(dst, arr)
        return True, arr.shape
    except Exception as exc:
        return False, f"{src}: {exc}"


def collect_jobs():
    jobs, per_split = [], {}
    for split in SPLITS:
        split_dir = INPUT_DIR / split
        if not split_dir.is_dir():
            print(f"  !! split folder missing: {split_dir}", flush=True)
            continue
        counts = {}
        for class_dir in sorted(p for p in split_dir.iterdir() if p.is_dir()):
            out_dir = OUTPUT_DIR / split / class_dir.name
            out_dir.mkdir(parents=True, exist_ok=True)
            wavs = sorted(class_dir.glob("*.wav"))
            for w in wavs:
                jobs.append((w, out_dir / f"{w.stem}.npy"))
            counts[class_dir.name] = len(wavs)
        per_split[split] = counts
    return jobs, per_split


def main():
    try:
        import librosa
    except ImportError:
        sys.exit("this script needs librosa:  pip install librosa soundfile")

    if not INPUT_DIR.is_dir():
        raise SystemExit(f"input dir not found: {INPUT_DIR}")

    print(f"STFT: n_fft {N_FFT} | win {WIN_LENGTH} | hop {HOP_LENGTH} | sr {SR}", flush=True)
    print(f"  ch0 mel   : {N_MELS} bands, {MEL_FMIN}-{MEL_FMAX} Hz", flush=True)
    print(f"  ch1 cqt   : {CQT_N_BINS} bins, {CQT_BINS_PER_OCTAVE}/octave from "
          f"{CQT_FMIN:.0f} Hz -> top {CQT_FMIN * 2 ** (CQT_N_BINS / CQT_BINS_PER_OCTAVE):.0f} Hz",
          flush=True)
    print(f"  ch2 gamma : {N_GAMMA} ERB filters, {GAMMA_FMIN:.0f}-{GAMMA_FMAX} Hz", flush=True)
    print(f"  -> arrays are (3, {N_MELS}, {N_FRAMES}) CHW, {np.dtype(DTYPE).name}, "
          f"each channel dB + min-max to [0,1]\n", flush=True)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    jobs, per_split = collect_jobs()
    if not jobs:
        raise SystemExit(f"no .wav files found under {INPUT_DIR}")

    print("source clips:")
    for split, counts in per_split.items():
        detail = "  ".join(f"{c}={n}" for c, n in sorted(counts.items()))
        print(f"  {split:11} {sum(counts.values()):>6}   {detail}", flush=True)
    print(f"  {'TOTAL':11} {len(jobs):>6}\n", flush=True)

    est_gb = len(jobs) * 3 * N_MELS * N_FRAMES * np.dtype(DTYPE).itemsize / 1e9
    print(f"generating {len(jobs)} arrays on {WORKERS} workers "
          f"(~{est_gb:.2f} GB expected; CQT is the slow channel)...", flush=True)
    t0 = time.time()
    done, failed, shapes = 0, [], set()

    def handle(ok, info):
        if ok:
            shapes.add(info)
        else:
            failed.append(info)

    def progress():
        el = time.time() - t0
        rate = done / max(el, 1e-9)
        eta = (len(jobs) - done) / max(rate, 1e-9)
        print(f"  {done}/{len(jobs)}  ({rate:.1f} clips/s, eta {eta/60:.1f} min)",
              flush=True)

    if WORKERS == 1:
        for job in jobs:
            ok, info = process_one(job)
            handle(ok, info)
            done += 1
            if done % 500 == 0 or done == len(jobs):
                progress()
    else:
        with Pool(WORKERS) as pool:
            for ok, info in pool.imap_unordered(process_one, jobs, chunksize=16):
                handle(ok, info)
                done += 1
                if done % 500 == 0 or done == len(jobs):
                    progress()
    print(f"  finished in {(time.time() - t0)/60:.1f} min", flush=True)

    print(f"\narray shapes produced: {sorted(shapes)}", flush=True)
    if len(shapes) > 1:
        print("  !! more than one shape - inspect before training", flush=True)
    if failed:
        print(f"  !! {len(failed)} clip(s) failed:", flush=True)
        for msg in failed[:10]:
            print(f"     {msg}", flush=True)

    n_npy = sum(1 for _ in OUTPUT_DIR.rglob("*.npy"))
    size_gb = sum(p.stat().st_size for p in OUTPUT_DIR.rglob("*.npy")) / 1e9
    print(f"\nwrote {n_npy} .npy files ({size_gb:.2f} GB) under {OUTPUT_DIR}", flush=True)
    print(f"  -> {OUTPUT_DIR}/<split>/<class>/*.npy   (3, {N_MELS}, {N_FRAMES}) CHW")


if __name__ == "__main__":
    main()


##CREATE MFCC DATASET

In [ ]:
#This script converts the wav dataset into standardized MFCC feature vectors
#(mean + std over time per clip). Reads INPUT_DIR/<split>/<class>/*.wav and
#writes X/y .npy arrays plus the scaler and metadata into OUTPUT_DIR.

import json
import sys
import time
from multiprocessing import Pool, cpu_count
from pathlib import Path

import numpy as np

# ======================= EDIT THESE =======================
INPUT_DIR = OUTPUTS / "cumulative_dataset"   # folder holding <split>/<class>/*.wav
OUTPUT_DIR = OUTPUTS / "mfcc_dataset"        # where the X/y arrays and metadata will be written

SPLITS = ["train", "validation", "test"]
FIT_SPLIT = "train"      # the split the scaler is fit on

SR = 16000
N_FFT = 1024
WIN_LENGTH = 1024
HOP_LENGTH = 128
CLIP_SAMPLES = SR        # 1 second clips

N_MELS = 128
FMIN, FMAX = 0, 8000

N_MFCC = 20
DCT_TYPE = 2
DCT_NORM = "ortho"
LIFTER = 0
DROP_C0 = False

WORKERS = max(1, cpu_count() - 1)   # set to 1 if multiprocessing gives trouble in the notebook
# ==========================================================


def find_split_root(base):
    base = Path(base)
    if not base.exists():
        return None
    for d in [base] + sorted(p for p in base.iterdir() if p.is_dir()):
        if (d / "train").is_dir() and (d / "validation").is_dir() \
                and next((d / "validation").rglob("*.wav"), None) is not None:
            return d
    return None


def mfcc_features(wav_path):
    import librosa

    y, _ = librosa.load(str(wav_path), sr=SR, mono=True)
    if len(y) < CLIP_SAMPLES:
        y = np.pad(y, (0, CLIP_SAMPLES - len(y)))
    else:
        y = y[:CLIP_SAMPLES]

    m = librosa.feature.mfcc(
        y=y, sr=SR, n_mfcc=N_MFCC, dct_type=DCT_TYPE, norm=DCT_NORM, lifter=LIFTER,
        n_fft=N_FFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH,
        window="hann", center=True, n_mels=N_MELS, fmin=FMIN, fmax=FMAX,
    )

    if DROP_C0:
        m = m[1:]
    return np.concatenate([m.mean(axis=1), m.std(axis=1)]).astype(np.float32)


def process_one(job):
    path, label = job
    try:
        return True, (mfcc_features(path), label, path.name)
    except Exception as exc:
        return False, f"{path}: {exc}"


def collect_jobs(split_dir, class_to_idx):
    jobs, counts = [], {}
    for cls, idx in class_to_idx.items():
        cdir = split_dir / cls
        wavs = sorted(cdir.glob("*.wav")) if cdir.is_dir() else []
        if not cdir.is_dir():
            print(f"  !! {split_dir.name}: no folder for class '{cls}'", flush=True)
        for w in wavs:
            jobs.append((w, idx))
        counts[cls] = len(wavs)
    return jobs, counts


def feature_names():
    idx = range(1, N_MFCC) if DROP_C0 else range(N_MFCC)
    return [f"mfcc{i}_mean" for i in idx] + [f"mfcc{i}_std" for i in idx]


def main():
    try:
        import librosa
    except ImportError:
        sys.exit("this script needs librosa:  pip install librosa soundfile")

    n_feat = 2 * (N_MFCC - 1 if DROP_C0 else N_MFCC)
    print(f"MFCC: n_mfcc {N_MFCC} (C0 {'dropped' if DROP_C0 else 'kept'}) | "
          f"dct {DCT_TYPE}/{DCT_NORM} | lifter {LIFTER}", flush=True)
    print(f"STFT: n_fft {N_FFT} | win {WIN_LENGTH} | hop {HOP_LENGTH} | sr {SR}", flush=True)
    print(f"filterbank: {N_MELS} mels, {FMIN}-{FMAX} Hz", flush=True)
    print(f"aggregation: mean + std over time -> {n_feat} features per clip\n", flush=True)

    root = find_split_root(INPUT_DIR)
    if root is None:
        raise SystemExit(f"could not find train/validation wav splits under {INPUT_DIR}")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    classes = sorted(d.name for d in (root / "train").iterdir() if d.is_dir())
    class_to_idx = {c: i for i, c in enumerate(classes)}
    print(f"classes: {classes}\n", flush=True)

    raw, labels, names = {}, {}, {}
    for split in SPLITS:
        sdir = root / split
        if not sdir.is_dir():
            print(f"  !! split folder missing: {sdir}", flush=True)
            continue
        jobs, counts = collect_jobs(sdir, class_to_idx)
        if not jobs:
            print(f"  !! no wavs in {sdir}", flush=True)
            continue

        detail = "  ".join(f"{c}={n}" for c, n in sorted(counts.items()))
        print(f"[{split}] {len(jobs)} clips   {detail}", flush=True)

        t0, feats, labs, fnames, failed = time.time(), [], [], [], []

        def handle(ok, info):
            if ok:
                v, lab, nm = info
                feats.append(v); labs.append(lab); fnames.append(nm)
            else:
                failed.append(info)

        if WORKERS == 1:
            for job in jobs:
                ok, info = process_one(job)
                handle(ok, info)
        else:
            with Pool(WORKERS) as pool:
                for ok, info in pool.imap(process_one, jobs, chunksize=64):
                    handle(ok, info)

        raw[split] = np.stack(feats).astype(np.float32)
        labels[split] = np.asarray(labs, dtype=np.int64)
        names[split] = fnames
        print(f"    -> {raw[split].shape} in {time.time() - t0:.0f}s", flush=True)
        if failed:
            print(f"    !! {len(failed)} clip(s) failed:", flush=True)
            for msg in failed[:5]:
                print(f"       {msg}", flush=True)

    if FIT_SPLIT not in raw:
        raise SystemExit(f"'{FIT_SPLIT}' split produced no features; cannot fit scaler")

    mean = raw[FIT_SPLIT].mean(axis=0)
    scale = raw[FIT_SPLIT].std(axis=0)
    scale[scale < 1e-8] = 1.0                            # guard constant features
    print(f"\nscaler fit on '{FIT_SPLIT}' ({raw[FIT_SPLIT].shape[0]} clips) "
          f"and applied to {', '.join(raw)}", flush=True)

    for split in raw:
        X = ((raw[split] - mean) / scale).astype(np.float32)
        np.save(OUTPUT_DIR / f"X_{split}.npy", X)
        np.save(OUTPUT_DIR / f"X_{split}_raw.npy", raw[split])
        np.save(OUTPUT_DIR / f"y_{split}.npy", labels[split])
        (OUTPUT_DIR / f"files_{split}.json").write_text(json.dumps(names[split]))
        print(f"  {split:11} X {X.shape}  mean {X.mean():+.3f}  std {X.std():.3f}",
              flush=True)

    np.save(OUTPUT_DIR / "scaler_mean.npy", mean.astype(np.float32))
    np.save(OUTPUT_DIR / "scaler_scale.npy", scale.astype(np.float32))
    (OUTPUT_DIR / "classes.json").write_text(json.dumps(classes))
    (OUTPUT_DIR / "metadata.json").write_text(json.dumps({
        "sr": SR, "n_fft": N_FFT, "win_length": WIN_LENGTH, "hop_length": HOP_LENGTH,
        "n_mels": N_MELS, "fmin": FMIN, "fmax": FMAX,
        "n_mfcc": N_MFCC, "dct_type": DCT_TYPE, "norm": DCT_NORM, "lifter": LIFTER,
        "drop_c0": DROP_C0, "aggregation": "mean+std over time",
        "n_features": n_feat, "feature_names": feature_names(),
        "classes": classes, "scaler_fit_split": FIT_SPLIT,
        "counts": {s: int(raw[s].shape[0]) for s in raw},
    }, indent=2))

    print(f"\ndone. features written to {OUTPUT_DIR}")
    print(f"  X = np.load('{OUTPUT_DIR}/X_train.npy')   # already standardized")
    print(f"  y = np.load('{OUTPUT_DIR}/y_train.npy')")


if __name__ == "__main__":
    main()


##MCG + RESNET50 (train)

In [ ]:
#This script trains ResNet50 on the 3-channel MCG dataset over a grid of
#learning rates and epoch counts. Reads DATA_DIR/<split>/<class>/*.npy, writes
#the accuracy grid to OUTPUT_DIR/results_grid.csv and the best weights to
#OUTPUT_DIR/resnet50_mcg_best.pt

import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as VT
from transformers import AutoModelForImageClassification

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **k):
        return x


# ======================= EDIT THESE =======================
DATA_DIR = OUTPUTS / "mcg_dataset"       # folder holding <split>/<class>/*.npy (the MCG cell's OUTPUT_DIR)
OUTPUT_DIR = OUTPUTS / "resnet50_mcg"    # where the results grid CSV and best weights will be written

MODEL_NAME = "microsoft/resnet-50"

OPTIMIZER = "adamw"
BATCH = 64
WEIGHT_DECAY = 1e-5
NO_AUGMENTATION = True

GRID_LRS = [0.05, 0.01, 0.005, 0.001, 0.0001, 0.00001, 0.000001]
GRID_EPOCHS = [10, 20, 30, 40, 50]

# Resume point: the first grid cell that will actually run. Cells before it are
# skipped, so set these to pick up where an earlier session stopped.
START_LR = 0.05
START_EPOCHS = 10

# Accuracies from earlier sessions, e.g. {(0.05, 10): 0.7212}. Used to seed the
# best cell so a resumed session only saves weights that beat it.
PRIOR_RESULTS = {}

IMG_SIZE = 224
NUM_WORKERS = 2      # set to 0 if the DataLoader gives trouble in the notebook
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
# ==========================================================

RESULTS_CSV_NAME = "results_grid.csv"

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)


def lr_tag(lr):
    return f"{lr:.0e}"


def find_split_root(base):
    base = Path(base)
    if not base.exists():
        return None
    for d in [base] + sorted(p for p in base.iterdir() if p.is_dir()):
        if (d / "train").is_dir() and (d / "validation").is_dir() \
                and next((d / "validation").rglob("*.npy"), None) is not None:
            return d
    return None


class McgDataset(Dataset):

    def __init__(self, split_dir, class_to_idx):
        self.samples = []
        for cls, idx in class_to_idx.items():
            cdir = split_dir / cls
            if not cdir.is_dir():
                print(f"  !! {split_dir.name}: no folder for class '{cls}'", flush=True)
                continue
            for npy in sorted(cdir.glob("*.npy")):
                self.samples.append((npy, idx))
        self.resize = VT.Resize((IMG_SIZE, IMG_SIZE), antialias=True)
        self.normalize = VT.Normalize(IMAGENET_MEAN, IMAGENET_STD)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        arr = np.load(path).astype(np.float32)

        if arr.ndim == 3 and arr.shape[2] == 3 and arr.shape[0] != 3:
            arr = np.transpose(arr, (2, 0, 1))
        arr = np.ascontiguousarray(arr)
        t = torch.from_numpy(arr)

        if t.ndim == 2:
            t = t.unsqueeze(0).repeat(3, 1, 1)
        elif t.shape[0] == 1:
            t = t.repeat(3, 1, 1)

        return self.normalize(self.resize(t)), label


def build_loaders():
    root = find_split_root(DATA_DIR)
    if root is None:
        raise SystemExit(f"could not find train/validation splits under {DATA_DIR}")

    classes = sorted(d.name for d in (root / "train").iterdir() if d.is_dir())
    class_to_idx = {c: i for i, c in enumerate(classes)}

    train_ds = McgDataset(root / "train", class_to_idx)
    val_ds = McgDataset(root / "validation", class_to_idx)
    print(f"classes: {classes}", flush=True)
    print(f"train clips {len(train_ds)} | validation clips {len(val_ds)}", flush=True)

    if len(train_ds):
        probe = np.load(train_ds.samples[0][0])
        print(f"sample array shape: {probe.shape}  "
              f"(expect (3, 128, F) CHW)\n", flush=True)

    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"),
                              drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
    return train_loader, val_loader, classes


def build_model(num_classes):
    model = AutoModelForImageClassification.from_pretrained(
        MODEL_NAME, num_labels=num_classes, ignore_mismatched_sizes=True)
    return model.to(DEVICE)


def forward(model, x):
    return model(pixel_values=x).logits


@torch.no_grad()
def validate_accuracy(model, loader):
    model.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
            logits = forward(model, x)
        correct += (logits.argmax(1) == y).sum().item()
        total += y.numel()
    return correct / max(total, 1)


def train_one_config(train_loader, val_loader, lr, epochs, num_classes):
    model = build_model(num_classes)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()                       # no label smoothing
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

    acc, state = 0.0, None
    try:
        for epoch in range(epochs):
            model.train()
            run_loss, seen = 0.0, 0
            for x, y in train_loader:
                x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
                    loss = criterion(forward(model, x), y)
                if not torch.isfinite(loss):
                    continue
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                run_loss += loss.item() * x.size(0)
                seen += x.size(0)

            acc = validate_accuracy(model, val_loader)
            print(f"      epoch {epoch + 1:>3}/{epochs}  "
                  f"train_loss {run_loss / max(seen, 1):.4f}  val_acc {acc * 100:.2f}%",
                  flush=True)

        state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    finally:
        del model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    return acc, state


def new_grid():
    return [["" for _ in GRID_EPOCHS] for _ in GRID_LRS]


def save_grid(grid):
    """Rewrite the results CSV: one row per LR, one column per epoch count."""
    lines = ["Epochs / LR," + ",".join(str(e) for e in GRID_EPOCHS)]
    for lr, row in zip(GRID_LRS, grid):
        lines.append(f"{lr}," + ",".join(str(v) for v in row))
    (OUTPUT_DIR / RESULTS_CSV_NAME).write_text("\n".join(lines) + "\n")


def save_best(state, classes, lr, epochs, acc):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    path = OUTPUT_DIR / "resnet50_mcg_best.pt"
    torch.save({"model": "resnet50", "model_name": MODEL_NAME,
                "representation": "mcg", "channels": ["mel", "cqt", "gammatone"],
                "classes": classes,
                "normalization": {"mean": IMAGENET_MEAN, "std": IMAGENET_STD},
                "hyperparameters": {"lr": lr, "epochs": epochs, "batch": BATCH,
                                    "optimizer": OPTIMIZER, "weight_decay": WEIGHT_DECAY,
                                    "augmentation": not NO_AUGMENTATION},
                "val_accuracy": acc, "model_state": state}, path)
    return path


def main():
    print(f"Device: {DEVICE}", flush=True)
    if DEVICE.type == "cuda":
        print(f"  GPU: {torch.cuda.get_device_name(0)}", flush=True)
    print(f"ResNet50 ({MODEL_NAME}) + 3-channel (Mel, CQT, Gamma)", flush=True)
    print(f"optimizer {OPTIMIZER} | batch {BATCH} | wd {WEIGHT_DECAY:.0e} | "
          f"no augmentation | no early stopping", flush=True)
    print(f"normalisation: ImageNet (matches the ResNet50 mel grid)", flush=True)
    total_epochs = len(GRID_LRS) * sum(GRID_EPOCHS)
    print(f"grid: {len(GRID_LRS)} LRs x {GRID_EPOCHS} epochs = "
          f"{len(GRID_LRS) * len(GRID_EPOCHS)} cells, {total_epochs} epochs total\n",
          flush=True)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    grid = new_grid()
    for (lr, epochs), acc in PRIOR_RESULTS.items():
        if lr in GRID_LRS and epochs in GRID_EPOCHS:
            grid[GRID_LRS.index(lr)][GRID_EPOCHS.index(epochs)] = round(acc, 4)
    save_grid(grid)
    print(f"results grid -> {OUTPUT_DIR / RESULTS_CSV_NAME}\n", flush=True)

    train_loader, val_loader, classes = build_loaders()
    num_classes = len(classes)

    if PRIOR_RESULTS:
        pk = max(PRIOR_RESULTS, key=PRIOR_RESULTS.get)
        best = {"acc": PRIOR_RESULTS[pk], "lr": pk[0], "epochs": pk[1], "state": None}
        print(f"seeded best from PRIOR_RESULTS: lr {best['lr']:g}, "
              f"epochs {best['epochs']}, val_acc {best['acc']:.4f}\n", flush=True)
    else:
        best = {"acc": -1.0, "lr": None, "epochs": 0, "state": None}

    print(f"resuming at lr {START_LR:g}, epochs {START_EPOCHS}\n", flush=True)
    started = False
    grid_t0 = time.time()

    for li, lr in enumerate(GRID_LRS):
        for ej, epochs in enumerate(GRID_EPOCHS):
            if not started:
                if abs(lr - START_LR) <= 1e-12 and epochs == START_EPOCHS:
                    started = True
                else:
                    continue
            print("#" * 72, flush=True)
            print(f"lr {lr:g} | epochs {epochs}  (fresh model)", flush=True)
            print("#" * 72, flush=True)

            t0 = time.time()
            acc, state = train_one_config(train_loader, val_loader, lr, epochs, num_classes)
            mins = (time.time() - t0) / 60

            grid[li][ej] = round(acc, 4)
            save_grid(grid)
            print(f"  >> val_acc {acc * 100:.2f}%  ({mins:.1f} min)  "
                  f"-> grid row {li + 2}, col {ej + 2}", flush=True)

            if state is not None and acc > best["acc"]:
                best = {"acc": acc, "lr": lr, "epochs": epochs, "state": state}
                path = save_best(state, classes, lr, epochs, acc)
                print(f"  >> NEW BEST -> {path}", flush=True)

            elapsed = (time.time() - grid_t0) / 3600
            print(f"  >> elapsed this session: {elapsed:.2f} h\n", flush=True)

    if not started:
        print(f"!! resume point (lr {START_LR:g}, epochs {START_EPOCHS}) is not in the "
              f"grid - nothing ran", flush=True)
        return

    print("=" * 60, flush=True)
    if best["lr"] is None:
        print("no successful cell; nothing saved", flush=True)
    elif best["state"] is not None:
        print(f"BEST: lr {best['lr']:g}, epochs {best['epochs']}, "
              f"val_acc {best['acc'] * 100:.2f}%", flush=True)
        print(f"  weights -> {OUTPUT_DIR / 'resnet50_mcg_best.pt'}", flush=True)
    else:
        print(f"BEST is a cell from an earlier session: lr {best['lr']:g}, "
              f"epochs {best['epochs']} (val_acc {best['acc']:.4f}).", flush=True)
        print("retraining it once to produce its weights...", flush=True)
        acc, state = train_one_config(train_loader, val_loader,
                                      best["lr"], best["epochs"], num_classes)
        if state is not None:
            path = save_best(state, classes, best["lr"], best["epochs"], acc)
            print(f"  retrained val_acc {acc * 100:.2f}%  -> {path}", flush=True)
    print("=" * 60, flush=True)
    print("\nTHE TASK HAS BEEN COMPLETED.", flush=True)


if __name__ == "__main__":
    main()


##MCG + VGGNET19 (train)

In [ ]:
#This script trains VGG19 on the 3-channel MCG dataset over a grid of
#learning rates and epoch counts. Reads DATA_DIR/<split>/<class>/*.npy, writes
#the accuracy grid to OUTPUT_DIR/results_grid.csv and the best weights to
#OUTPUT_DIR/vgg19_mcg_best.pt

import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as VT
from torchvision.models import vgg19, VGG19_Weights

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **k):
        return x


# ======================= EDIT THESE =======================
DATA_DIR = OUTPUTS / "mcg_dataset"       # folder holding <split>/<class>/*.npy (the MCG cell's OUTPUT_DIR)
OUTPUT_DIR = OUTPUTS / "vgg19_mcg"       # where the results grid CSV and best weights will be written

OPTIMIZER = "adamw"
BATCH = 64
WEIGHT_DECAY = 1e-5
NO_AUGMENTATION = True

GRID_LRS = [0.05, 0.01, 0.005, 0.001, 0.0001, 0.00001, 0.000001]
GRID_EPOCHS = [10, 20, 30, 40, 50]

# Resume point: the first grid cell that will actually run. Cells before it are
# skipped, so set these to pick up where an earlier session stopped.
START_LR = 0.05
START_EPOCHS = 10

# Accuracies from earlier sessions, e.g. {(0.05, 10): 0.7212}. Used to seed the
# best cell so a resumed session only saves weights that beat it.
PRIOR_RESULTS = {}

IMG_SIZE = 224
NUM_WORKERS = 2      # set to 0 if the DataLoader gives trouble in the notebook
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
# ==========================================================

RESULTS_CSV_NAME = "results_grid.csv"

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)


def lr_tag(lr):
    return f"{lr:.0e}"


def find_split_root(base):
    base = Path(base)
    if not base.exists():
        return None
    for d in [base] + sorted(p for p in base.iterdir() if p.is_dir()):
        if (d / "train").is_dir() and (d / "validation").is_dir() \
                and next((d / "validation").rglob("*.npy"), None) is not None:
            return d
    return None


class McgDataset(Dataset):

    def __init__(self, split_dir, class_to_idx):
        self.samples = []
        for cls, idx in class_to_idx.items():
            cdir = split_dir / cls
            if not cdir.is_dir():
                print(f"  !! {split_dir.name}: no folder for class '{cls}'", flush=True)
                continue
            for npy in sorted(cdir.glob("*.npy")):
                self.samples.append((npy, idx))
        self.resize = VT.Resize((IMG_SIZE, IMG_SIZE), antialias=True)
        self.normalize = VT.Normalize(IMAGENET_MEAN, IMAGENET_STD)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        arr = np.load(path).astype(np.float32)

        if arr.ndim == 3 and arr.shape[2] == 3 and arr.shape[0] != 3:
            arr = np.transpose(arr, (2, 0, 1))
        arr = np.ascontiguousarray(arr)
        t = torch.from_numpy(arr)

        if t.ndim == 2:
            t = t.unsqueeze(0).repeat(3, 1, 1)
        elif t.shape[0] == 1:
            t = t.repeat(3, 1, 1)

        return self.normalize(self.resize(t)), label


def build_loaders():
    root = find_split_root(DATA_DIR)
    if root is None:
        raise SystemExit(f"could not find train/validation splits under {DATA_DIR}")

    classes = sorted(d.name for d in (root / "train").iterdir() if d.is_dir())
    class_to_idx = {c: i for i, c in enumerate(classes)}

    train_ds = McgDataset(root / "train", class_to_idx)
    val_ds = McgDataset(root / "validation", class_to_idx)
    print(f"classes: {classes}", flush=True)
    print(f"train clips {len(train_ds)} | validation clips {len(val_ds)}", flush=True)

    if len(train_ds):
        probe = np.load(train_ds.samples[0][0])
        print(f"sample array shape: {probe.shape}  "
              f"(expect (3, 128, F) CHW)\n", flush=True)

    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"),
                              drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
    return train_loader, val_loader, classes


def build_model(num_classes):
    """torchvision VGG19 (no BatchNorm) with a fresh classifier head."""
    model = vgg19(weights=VGG19_Weights.IMAGENET1K_V1)
    model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
    return model.to(DEVICE)


def forward(model, x):
    return model(x)


@torch.no_grad()
def validate_accuracy(model, loader):
    model.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
            logits = forward(model, x)
        correct += (logits.argmax(1) == y).sum().item()
        total += y.numel()
    return correct / max(total, 1)


def train_one_config(train_loader, val_loader, lr, epochs, num_classes):
    model = build_model(num_classes)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()                       # no label smoothing
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

    acc, state = 0.0, None
    try:
        for epoch in range(epochs):
            model.train()
            run_loss, seen = 0.0, 0
            for x, y in train_loader:
                x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
                    loss = criterion(forward(model, x), y)
                if not torch.isfinite(loss):
                    continue
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                run_loss += loss.item() * x.size(0)
                seen += x.size(0)

            acc = validate_accuracy(model, val_loader)
            print(f"      epoch {epoch + 1:>3}/{epochs}  "
                  f"train_loss {run_loss / max(seen, 1):.4f}  val_acc {acc * 100:.2f}%",
                  flush=True)

        state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    finally:
        del model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    return acc, state


def new_grid():
    return [["" for _ in GRID_EPOCHS] for _ in GRID_LRS]


def save_grid(grid):
    """Rewrite the results CSV: one row per LR, one column per epoch count."""
    lines = ["Epochs / LR," + ",".join(str(e) for e in GRID_EPOCHS)]
    for lr, row in zip(GRID_LRS, grid):
        lines.append(f"{lr}," + ",".join(str(v) for v in row))
    (OUTPUT_DIR / RESULTS_CSV_NAME).write_text("\n".join(lines) + "\n")


def save_best(state, classes, lr, epochs, acc):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    path = OUTPUT_DIR / "vgg19_mcg_best.pt"
    torch.save({"model": "vgg19", "model_name": "torchvision vgg19 (IMAGENET1K_V1)",
                "representation": "mcg", "channels": ["mel", "cqt", "gammatone"],
                "classes": classes,
                "normalization": {"mean": IMAGENET_MEAN, "std": IMAGENET_STD},
                "hyperparameters": {"lr": lr, "epochs": epochs, "batch": BATCH,
                                    "optimizer": OPTIMIZER, "weight_decay": WEIGHT_DECAY,
                                    "augmentation": not NO_AUGMENTATION},
                "val_accuracy": acc, "model_state": state}, path)
    return path


def main():
    print(f"Device: {DEVICE}", flush=True)
    if DEVICE.type == "cuda":
        print(f"  GPU: {torch.cuda.get_device_name(0)}", flush=True)
    print("VGG19 (torchvision, ImageNet weights, no BatchNorm) + "
          "3-channel (Mel, CQT, Gamma)", flush=True)
    print(f"optimizer {OPTIMIZER} | batch {BATCH} | wd {WEIGHT_DECAY:.0e} | "
          f"no augmentation | no early stopping", flush=True)
    print("normalisation: ImageNet (matches the VGG19 mel grid)", flush=True)
    total_epochs = len(GRID_LRS) * sum(GRID_EPOCHS)
    print(f"grid: {len(GRID_LRS)} LRs x {GRID_EPOCHS} epochs = "
          f"{len(GRID_LRS) * len(GRID_EPOCHS)} cells, {total_epochs} epochs total\n",
          flush=True)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    grid = new_grid()
    for (lr, epochs), acc in PRIOR_RESULTS.items():
        if lr in GRID_LRS and epochs in GRID_EPOCHS:
            grid[GRID_LRS.index(lr)][GRID_EPOCHS.index(epochs)] = round(acc, 4)
    save_grid(grid)
    print(f"results grid -> {OUTPUT_DIR / RESULTS_CSV_NAME}\n", flush=True)

    train_loader, val_loader, classes = build_loaders()
    num_classes = len(classes)

    if PRIOR_RESULTS:
        pk = max(PRIOR_RESULTS, key=PRIOR_RESULTS.get)
        best = {"acc": PRIOR_RESULTS[pk], "lr": pk[0], "epochs": pk[1], "state": None}
        print(f"seeded best from PRIOR_RESULTS: lr {best['lr']:g}, "
              f"epochs {best['epochs']}, val_acc {best['acc']:.4f}\n", flush=True)
    else:
        best = {"acc": -1.0, "lr": None, "epochs": 0, "state": None}

    print(f"resuming at lr {START_LR:g}, epochs {START_EPOCHS}\n", flush=True)
    started = False
    grid_t0 = time.time()

    for li, lr in enumerate(GRID_LRS):
        for ej, epochs in enumerate(GRID_EPOCHS):
            if not started:
                if abs(lr - START_LR) <= 1e-12 and epochs == START_EPOCHS:
                    started = True
                else:
                    continue
            print("#" * 72, flush=True)
            print(f"lr {lr:g} | epochs {epochs}  (fresh model)", flush=True)
            print("#" * 72, flush=True)

            t0 = time.time()
            acc, state = train_one_config(train_loader, val_loader, lr, epochs, num_classes)
            mins = (time.time() - t0) / 60

            grid[li][ej] = round(acc, 4)
            save_grid(grid)
            print(f"  >> val_acc {acc * 100:.2f}%  ({mins:.1f} min)  "
                  f"-> grid row {li + 2}, col {ej + 2}", flush=True)

            if state is not None and acc > best["acc"]:
                best = {"acc": acc, "lr": lr, "epochs": epochs, "state": state}
                path = save_best(state, classes, lr, epochs, acc)
                print(f"  >> NEW BEST -> {path}", flush=True)

            elapsed = (time.time() - grid_t0) / 3600
            print(f"  >> elapsed this session: {elapsed:.2f} h\n", flush=True)

    if not started:
        print(f"!! resume point (lr {START_LR:g}, epochs {START_EPOCHS}) is not in the "
              f"grid - nothing ran", flush=True)
        return

    print("=" * 60, flush=True)
    if best["lr"] is None:
        print("no successful cell; nothing saved", flush=True)
    elif best["state"] is not None:
        print(f"BEST: lr {best['lr']:g}, epochs {best['epochs']}, "
              f"val_acc {best['acc'] * 100:.2f}%", flush=True)
        print(f"  weights -> {OUTPUT_DIR / 'vgg19_mcg_best.pt'}", flush=True)
    else:
        print(f"BEST is a cell from an earlier session: lr {best['lr']:g}, "
              f"epochs {best['epochs']} (val_acc {best['acc']:.4f}).", flush=True)
        print("retraining it once to produce its weights...", flush=True)
        acc, state = train_one_config(train_loader, val_loader,
                                      best["lr"], best["epochs"], num_classes)
        if state is not None:
            path = save_best(state, classes, best["lr"], best["epochs"], acc)
            print(f"  retrained val_acc {acc * 100:.2f}%  -> {path}", flush=True)
    print("=" * 60, flush=True)
    print("\nTHE TASK HAS BEEN COMPLETED.", flush=True)


if __name__ == "__main__":
    main()


##MCG + VIT (train)

In [ ]:
#This script trains ViT on the 3-channel MCG dataset over a grid of
#learning rates and epoch counts. Reads DATA_DIR/<split>/<class>/*.npy, writes
#the accuracy grid to OUTPUT_DIR/results_grid.csv and the best weights to
#OUTPUT_DIR/vit_mcg_best.pt

import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as VT
from transformers import ViTForImageClassification

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **k):
        return x


# ======================= EDIT THESE =======================
DATA_DIR = OUTPUTS / "mcg_dataset"       # folder holding <split>/<class>/*.npy (the MCG cell's OUTPUT_DIR)
OUTPUT_DIR = OUTPUTS / "vit_mcg"         # where the results grid CSV and best weights will be written

MODEL_NAME = "google/vit-base-patch16-224"

OPTIMIZER = "adamw"
BATCH = 64
WEIGHT_DECAY = 1e-5
NO_AUGMENTATION = True

GRID_LRS = [0.05, 0.01, 0.005, 0.001, 0.0001, 0.00001, 0.000001]
GRID_EPOCHS = [10, 20, 30, 40, 50]

# Resume point: the first grid cell that will actually run. Cells before it are
# skipped, so set these to pick up where an earlier session stopped.
START_LR = 0.05
START_EPOCHS = 10

# Accuracies from earlier sessions, e.g. {(0.05, 10): 0.7212}. Used to seed the
# best cell so a resumed session only saves weights that beat it.
PRIOR_RESULTS = {}

IMG_SIZE = 224
NUM_WORKERS = 2      # set to 0 if the DataLoader gives trouble in the notebook
VIT_MEAN, VIT_STD = [0.5, 0.5, 0.5], [0.5, 0.5, 0.5]
# ==========================================================

RESULTS_CSV_NAME = "results_grid.csv"

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)


def lr_tag(lr):
    return f"{lr:.0e}"


def find_split_root(base):
    base = Path(base)
    if not base.exists():
        return None
    for d in [base] + sorted(p for p in base.iterdir() if p.is_dir()):
        if (d / "train").is_dir() and (d / "validation").is_dir() \
                and next((d / "validation").rglob("*.npy"), None) is not None:
            return d
    return None


class McgDataset(Dataset):

    def __init__(self, split_dir, class_to_idx):
        self.samples = []
        for cls, idx in class_to_idx.items():
            cdir = split_dir / cls
            if not cdir.is_dir():
                print(f"  !! {split_dir.name}: no folder for class '{cls}'", flush=True)
                continue
            for npy in sorted(cdir.glob("*.npy")):
                self.samples.append((npy, idx))
        self.resize = VT.Resize((IMG_SIZE, IMG_SIZE), antialias=True)
        self.normalize = VT.Normalize(VIT_MEAN, VIT_STD)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        arr = np.load(path).astype(np.float32)

        if arr.ndim == 3 and arr.shape[2] == 3 and arr.shape[0] != 3:
            arr = np.transpose(arr, (2, 0, 1))
        arr = np.ascontiguousarray(arr)
        t = torch.from_numpy(arr)

        if t.ndim == 2:
            t = t.unsqueeze(0).repeat(3, 1, 1)
        elif t.shape[0] == 1:
            t = t.repeat(3, 1, 1)

        return self.normalize(self.resize(t)), label


def build_loaders():
    root = find_split_root(DATA_DIR)
    if root is None:
        raise SystemExit(f"could not find train/validation splits under {DATA_DIR}")

    classes = sorted(d.name for d in (root / "train").iterdir() if d.is_dir())
    class_to_idx = {c: i for i, c in enumerate(classes)}

    train_ds = McgDataset(root / "train", class_to_idx)
    val_ds = McgDataset(root / "validation", class_to_idx)
    print(f"classes: {classes}", flush=True)
    print(f"train clips {len(train_ds)} | validation clips {len(val_ds)}", flush=True)

    if len(train_ds):
        probe = np.load(train_ds.samples[0][0])
        print(f"sample array shape: {probe.shape}  "
              f"(expect (3, 128, F) CHW)\n", flush=True)

    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"),
                              drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
    return train_loader, val_loader, classes


def build_model(num_classes):
    model = ViTForImageClassification.from_pretrained(
        MODEL_NAME, num_labels=num_classes, ignore_mismatched_sizes=True)
    return model.to(DEVICE)


def forward(model, x):
    return model(pixel_values=x).logits


@torch.no_grad()
def validate_accuracy(model, loader):
    model.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
            logits = forward(model, x)
        correct += (logits.argmax(1) == y).sum().item()
        total += y.numel()
    return correct / max(total, 1)


def train_one_config(train_loader, val_loader, lr, epochs, num_classes):
    model = build_model(num_classes)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()                       # no label smoothing
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

    acc, state = 0.0, None
    try:
        for epoch in range(epochs):
            model.train()
            run_loss, seen = 0.0, 0
            for x, y in train_loader:
                x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
                    loss = criterion(forward(model, x), y)
                if not torch.isfinite(loss):
                    continue
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                run_loss += loss.item() * x.size(0)
                seen += x.size(0)

            acc = validate_accuracy(model, val_loader)
            print(f"      epoch {epoch + 1:>3}/{epochs}  "
                  f"train_loss {run_loss / max(seen, 1):.4f}  val_acc {acc * 100:.2f}%",
                  flush=True)

        state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    finally:
        del model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    return acc, state


def new_grid():
    return [["" for _ in GRID_EPOCHS] for _ in GRID_LRS]


def save_grid(grid):
    """Rewrite the results CSV: one row per LR, one column per epoch count."""
    lines = ["Epochs / LR," + ",".join(str(e) for e in GRID_EPOCHS)]
    for lr, row in zip(GRID_LRS, grid):
        lines.append(f"{lr}," + ",".join(str(v) for v in row))
    (OUTPUT_DIR / RESULTS_CSV_NAME).write_text("\n".join(lines) + "\n")


def save_best(state, classes, lr, epochs, acc):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    path = OUTPUT_DIR / "vit_mcg_best.pt"
    torch.save({"model": "vit", "model_name": MODEL_NAME,
                "representation": "mcg", "channels": ["mel", "cqt", "gammatone"],
                "classes": classes,
                "normalization": {"mean": VIT_MEAN, "std": VIT_STD},
                "hyperparameters": {"lr": lr, "epochs": epochs, "batch": BATCH,
                                    "optimizer": OPTIMIZER, "weight_decay": WEIGHT_DECAY,
                                    "augmentation": not NO_AUGMENTATION},
                "val_accuracy": acc, "model_state": state}, path)
    return path


def main():
    print(f"Device: {DEVICE}", flush=True)
    if DEVICE.type == "cuda":
        print(f"  GPU: {torch.cuda.get_device_name(0)}", flush=True)
    print(f"ViT ({MODEL_NAME}) + 3-channel (Mel, CQT, Gamma)", flush=True)
    print(f"optimizer {OPTIMIZER} | batch {BATCH} | wd {WEIGHT_DECAY:.0e} | "
          f"no augmentation | no early stopping", flush=True)
    print(f"normalisation: mean {VIT_MEAN[0]} / std {VIT_STD[0]} (ViT, not ImageNet)",
          flush=True)
    total_epochs = len(GRID_LRS) * sum(GRID_EPOCHS)
    print(f"grid: {len(GRID_LRS)} LRs x {GRID_EPOCHS} epochs = "
          f"{len(GRID_LRS) * len(GRID_EPOCHS)} cells, {total_epochs} epochs total\n",
          flush=True)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    grid = new_grid()
    for (lr, epochs), acc in PRIOR_RESULTS.items():
        if lr in GRID_LRS and epochs in GRID_EPOCHS:
            grid[GRID_LRS.index(lr)][GRID_EPOCHS.index(epochs)] = round(acc, 4)
    save_grid(grid)
    print(f"results grid -> {OUTPUT_DIR / RESULTS_CSV_NAME}\n", flush=True)

    train_loader, val_loader, classes = build_loaders()
    num_classes = len(classes)

    if PRIOR_RESULTS:
        pk = max(PRIOR_RESULTS, key=PRIOR_RESULTS.get)
        best = {"acc": PRIOR_RESULTS[pk], "lr": pk[0], "epochs": pk[1], "state": None}
        print(f"seeded best from PRIOR_RESULTS: lr {best['lr']:g}, "
              f"epochs {best['epochs']}, val_acc {best['acc']:.4f}\n", flush=True)
    else:
        best = {"acc": -1.0, "lr": None, "epochs": 0, "state": None}

    print(f"resuming at lr {START_LR:g}, epochs {START_EPOCHS}\n", flush=True)
    started = False
    grid_t0 = time.time()

    for li, lr in enumerate(GRID_LRS):
        for ej, epochs in enumerate(GRID_EPOCHS):
            if not started:
                if abs(lr - START_LR) <= 1e-12 and epochs == START_EPOCHS:
                    started = True
                else:
                    continue
            print("#" * 72, flush=True)
            print(f"lr {lr:g} | epochs {epochs}  (fresh model)", flush=True)
            print("#" * 72, flush=True)

            t0 = time.time()
            acc, state = train_one_config(train_loader, val_loader, lr, epochs, num_classes)
            mins = (time.time() - t0) / 60

            grid[li][ej] = round(acc, 4)
            save_grid(grid)
            print(f"  >> val_acc {acc * 100:.2f}%  ({mins:.1f} min)  "
                  f"-> grid row {li + 2}, col {ej + 2}", flush=True)

            if state is not None and acc > best["acc"]:
                best = {"acc": acc, "lr": lr, "epochs": epochs, "state": state}
                path = save_best(state, classes, lr, epochs, acc)
                print(f"  >> NEW BEST -> {path}", flush=True)

            elapsed = (time.time() - grid_t0) / 3600
            print(f"  >> elapsed this session: {elapsed:.2f} h\n", flush=True)

    if not started:
        print(f"!! resume point (lr {START_LR:g}, epochs {START_EPOCHS}) is not in the "
              f"grid - nothing ran", flush=True)
        return

    print("=" * 60, flush=True)
    if best["lr"] is None:
        print("no successful cell; nothing saved", flush=True)
    elif best["state"] is not None:
        print(f"BEST: lr {best['lr']:g}, epochs {best['epochs']}, "
              f"val_acc {best['acc'] * 100:.2f}%", flush=True)
        print(f"  weights -> {OUTPUT_DIR / 'vit_mcg_best.pt'}", flush=True)
    else:
        print(f"BEST is a cell from an earlier session: lr {best['lr']:g}, "
              f"epochs {best['epochs']} (val_acc {best['acc']:.4f}).", flush=True)
        print("retraining it once to produce its weights...", flush=True)
        acc, state = train_one_config(train_loader, val_loader,
                                      best["lr"], best["epochs"], num_classes)
        if state is not None:
            path = save_best(state, classes, best["lr"], best["epochs"], acc)
            print(f"  retrained val_acc {acc * 100:.2f}%  -> {path}", flush=True)
    print("=" * 60, flush=True)
    print("\nTHE TASK HAS BEEN COMPLETED.", flush=True)


if __name__ == "__main__":
    main()


##mel + resnet50 (train resnet50
)

In [ ]:
#This script trains ResNet50 on the mel-spectrogram dataset over a grid of
#learning rates and epoch counts. Reads DATA_DIR/<split>/<class>/*.npy, writes
#the accuracy grid to OUTPUT_DIR/results_grid.csv and the best weights to
#OUTPUT_DIR/resnet50_mel_best.pt

import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as VT
from transformers import AutoModelForImageClassification

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **k):
        return x


# ======================= EDIT THESE =======================
DATA_DIR = OUTPUTS / "mel_dataset"       # folder holding <split>/<class>/*.npy (the mel cell's OUTPUT_DIR)
OUTPUT_DIR = OUTPUTS / "resnet50_mel"    # where the results grid CSV and best weights will be written

MODEL_NAME = "microsoft/resnet-50"

OPTIMIZER = "adamw"
BATCH = 64
WEIGHT_DECAY = 1e-5
NO_AUGMENTATION = True

GRID_LRS = [0.05, 0.01, 0.005, 0.001, 0.0001, 0.00001, 0.000001]
GRID_EPOCHS = [10, 20, 30, 40, 50]

# Resume point: the first grid cell that will actually run. Cells before it are
# skipped, so set these to pick up where an earlier session stopped.
START_LR = 0.05
START_EPOCHS = 10

# Accuracies from earlier sessions, e.g. {(0.05, 10): 0.7212}. Used to seed the
# best cell so a resumed session only saves weights that beat it.
PRIOR_RESULTS = {}

IMG_SIZE = 224
NUM_WORKERS = 2      # set to 0 if the DataLoader gives trouble in the notebook
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
# ==========================================================

RESULTS_CSV_NAME = "results_grid.csv"

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)


def lr_tag(lr):
    return f"{lr:.0e}"


def find_split_root(base):
    base = Path(base)
    if not base.exists():
        return None
    for d in [base] + sorted(p for p in base.iterdir() if p.is_dir()):
        if (d / "train").is_dir() and (d / "validation").is_dir() \
                and next((d / "validation").rglob("*.npy"), None) is not None:
            return d
    return None


class MelDataset(Dataset):

    def __init__(self, split_dir, class_to_idx):
        self.samples = []
        for cls, idx in class_to_idx.items():
            cdir = split_dir / cls
            if not cdir.is_dir():
                print(f"  !! {split_dir.name}: no folder for class '{cls}'", flush=True)
                continue
            for npy in sorted(cdir.glob("*.npy")):
                self.samples.append((npy, idx))
        self.resize = VT.Resize((IMG_SIZE, IMG_SIZE), antialias=True)
        self.normalize = VT.Normalize(IMAGENET_MEAN, IMAGENET_STD)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        arr = np.load(path).astype(np.float32)
        t = torch.from_numpy(arr)

        if t.ndim == 2:
            t = t.unsqueeze(0)
        if t.shape[0] == 1:
            t = t.repeat(3, 1, 1)

        return self.normalize(self.resize(t)), label


def build_loaders():
    root = find_split_root(DATA_DIR)
    if root is None:
        raise SystemExit(f"could not find train/validation splits under {DATA_DIR}")

    classes = sorted(d.name for d in (root / "train").iterdir() if d.is_dir())
    class_to_idx = {c: i for i, c in enumerate(classes)}

    train_ds = MelDataset(root / "train", class_to_idx)
    val_ds = MelDataset(root / "validation", class_to_idx)
    print(f"classes: {classes}", flush=True)
    print(f"train clips {len(train_ds)} | validation clips {len(val_ds)}\n", flush=True)

    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"),
                              drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
    return train_loader, val_loader, classes


def build_model(num_classes):
    model = AutoModelForImageClassification.from_pretrained(
        MODEL_NAME, num_labels=num_classes, ignore_mismatched_sizes=True)
    return model.to(DEVICE)


def forward(model, x):
    return model(pixel_values=x).logits


@torch.no_grad()
def validate_accuracy(model, loader):
    model.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
            logits = forward(model, x)
        correct += (logits.argmax(1) == y).sum().item()
        total += y.numel()
    return correct / max(total, 1)


def train_one_config(train_loader, val_loader, lr, epochs, num_classes):
    model = build_model(num_classes)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()                       # no label smoothing
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

    acc, state = 0.0, None
    try:
        for epoch in range(epochs):
            model.train()
            run_loss, seen = 0.0, 0
            for x, y in train_loader:
                x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
                    loss = criterion(forward(model, x), y)
                if not torch.isfinite(loss):
                    continue
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                run_loss += loss.item() * x.size(0)
                seen += x.size(0)

            acc = validate_accuracy(model, val_loader)
            print(f"      epoch {epoch + 1:>3}/{epochs}  "
                  f"train_loss {run_loss / max(seen, 1):.4f}  val_acc {acc * 100:.2f}%",
                  flush=True)

        state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    finally:
        del model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    return acc, state


def new_grid():
    return [["" for _ in GRID_EPOCHS] for _ in GRID_LRS]


def save_grid(grid):
    """Rewrite the results CSV: one row per LR, one column per epoch count."""
    lines = ["Epochs / LR," + ",".join(str(e) for e in GRID_EPOCHS)]
    for lr, row in zip(GRID_LRS, grid):
        lines.append(f"{lr}," + ",".join(str(v) for v in row))
    (OUTPUT_DIR / RESULTS_CSV_NAME).write_text("\n".join(lines) + "\n")


def save_best(state, classes, lr, epochs, acc):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    path = OUTPUT_DIR / "resnet50_mel_best.pt"
    torch.save({"model": "resnet50", "model_name": MODEL_NAME,
                "representation": "mel", "classes": classes,
                "normalization": {"mean": IMAGENET_MEAN, "std": IMAGENET_STD},
                "hyperparameters": {"lr": lr, "epochs": epochs, "batch": BATCH,
                                    "optimizer": OPTIMIZER, "weight_decay": WEIGHT_DECAY,
                                    "augmentation": not NO_AUGMENTATION},
                "val_accuracy": acc, "model_state": state}, path)
    return path


def main():
    print(f"Device: {DEVICE}", flush=True)
    if DEVICE.type == "cuda":
        print(f"  GPU: {torch.cuda.get_device_name(0)}", flush=True)
    print(f"ResNet50 ({MODEL_NAME}) + mel-spectrogram", flush=True)
    print(f"optimizer {OPTIMIZER} | batch {BATCH} | wd {WEIGHT_DECAY:.0e} | "
          f"no augmentation | no early stopping", flush=True)
    total_epochs = len(GRID_LRS) * sum(GRID_EPOCHS)
    print(f"grid: {len(GRID_LRS)} LRs x {GRID_EPOCHS} epochs = "
          f"{len(GRID_LRS) * len(GRID_EPOCHS)} cells, {total_epochs} epochs total\n",
          flush=True)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    grid = new_grid()
    for (lr, epochs), acc in PRIOR_RESULTS.items():
        if lr in GRID_LRS and epochs in GRID_EPOCHS:
            grid[GRID_LRS.index(lr)][GRID_EPOCHS.index(epochs)] = round(acc, 4)
    save_grid(grid)
    print(f"results grid -> {OUTPUT_DIR / RESULTS_CSV_NAME}\n", flush=True)

    train_loader, val_loader, classes = build_loaders()
    num_classes = len(classes)

    if PRIOR_RESULTS:
        pk = max(PRIOR_RESULTS, key=PRIOR_RESULTS.get)
        best = {"acc": PRIOR_RESULTS[pk], "lr": pk[0], "epochs": pk[1], "state": None}
        print(f"seeded best from PRIOR_RESULTS: lr {best['lr']:g}, "
              f"epochs {best['epochs']}, val_acc {best['acc']:.4f}\n", flush=True)
    else:
        best = {"acc": -1.0, "lr": None, "epochs": 0, "state": None}

    print(f"resuming at lr {START_LR:g}, epochs {START_EPOCHS}\n", flush=True)
    started = False
    grid_t0 = time.time()

    for li, lr in enumerate(GRID_LRS):
        for ej, epochs in enumerate(GRID_EPOCHS):
            if not started:
                if abs(lr - START_LR) <= 1e-12 and epochs == START_EPOCHS:
                    started = True
                else:
                    continue
            print("#" * 72, flush=True)
            print(f"lr {lr:g} | epochs {epochs}  (fresh model)", flush=True)
            print("#" * 72, flush=True)

            t0 = time.time()
            acc, state = train_one_config(train_loader, val_loader, lr, epochs, num_classes)
            mins = (time.time() - t0) / 60

            grid[li][ej] = round(acc, 4)
            save_grid(grid)
            print(f"  >> val_acc {acc * 100:.2f}%  ({mins:.1f} min)  "
                  f"-> grid row {li + 2}, col {ej + 2}", flush=True)

            if state is not None and acc > best["acc"]:
                best = {"acc": acc, "lr": lr, "epochs": epochs, "state": state}
                path = save_best(state, classes, lr, epochs, acc)
                print(f"  >> NEW BEST -> {path}", flush=True)

            elapsed = (time.time() - grid_t0) / 3600
            print(f"  >> elapsed this session: {elapsed:.2f} h\n", flush=True)

    if not started:
        print(f"!! resume point (lr {START_LR:g}, epochs {START_EPOCHS}) is not in the "
              f"grid - nothing ran", flush=True)
        return

    print("=" * 60, flush=True)
    if best["lr"] is None:
        print("no successful cell; nothing saved", flush=True)
    elif best["state"] is not None:
        print(f"BEST: lr {best['lr']:g}, epochs {best['epochs']}, "
              f"val_acc {best['acc'] * 100:.2f}%", flush=True)
        print(f"  weights -> {OUTPUT_DIR / 'resnet50_mel_best.pt'}", flush=True)
    else:
        print(f"BEST is a cell from an earlier session: lr {best['lr']:g}, "
              f"epochs {best['epochs']} (val_acc {best['acc']:.4f}).", flush=True)
        print("retraining it once to produce its weights...", flush=True)
        acc, state = train_one_config(train_loader, val_loader,
                                      best["lr"], best["epochs"], num_classes)
        if state is not None:
            path = save_best(state, classes, best["lr"], best["epochs"], acc)
            print(f"  retrained val_acc {acc * 100:.2f}%  -> {path}", flush=True)
    print("=" * 60, flush=True)
    print("\nTHE TASK HAS BEEN COMPLETED.", flush=True)


if __name__ == "__main__":
    main()


##Mel + VGGNet19

In [ ]:
#This script trains VGG19 on the mel-spectrogram dataset over a grid of
#learning rates and epoch counts. Reads DATA_DIR/<split>/<class>/*.npy, writes
#the accuracy grid to OUTPUT_DIR/results_grid.csv and the best weights to
#OUTPUT_DIR/vgg19_mel_best.pt

import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as VT
from torchvision.models import vgg19, VGG19_Weights

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **k):
        return x


# ======================= EDIT THESE =======================
DATA_DIR = OUTPUTS / "mel_dataset"       # folder holding <split>/<class>/*.npy (the mel cell's OUTPUT_DIR)
OUTPUT_DIR = OUTPUTS / "vgg19_mel"       # where the results grid CSV and best weights will be written

OPTIMIZER = "adamw"
BATCH = 64
WEIGHT_DECAY = 1e-5
NO_AUGMENTATION = True

GRID_LRS = [0.05, 0.01, 0.005, 0.001, 0.0001, 0.00001, 0.000001]
GRID_EPOCHS = [10, 20, 30, 40, 50]

# Resume point: the first grid cell that will actually run. Cells before it are
# skipped, so set these to pick up where an earlier session stopped.
START_LR = 0.05
START_EPOCHS = 10

# Accuracies from earlier sessions, e.g. {(0.05, 10): 0.7212}. Used to seed the
# best cell so a resumed session only saves weights that beat it.
PRIOR_RESULTS = {}

IMG_SIZE = 224
NUM_WORKERS = 2      # set to 0 if the DataLoader gives trouble in the notebook
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
# ==========================================================

RESULTS_CSV_NAME = "results_grid.csv"

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)


def lr_tag(lr):
    return f"{lr:.0e}"


def find_split_root(base):
    base = Path(base)
    if not base.exists():
        return None
    for d in [base] + sorted(p for p in base.iterdir() if p.is_dir()):
        if (d / "train").is_dir() and (d / "validation").is_dir() \
                and next((d / "validation").rglob("*.npy"), None) is not None:
            return d
    return None


class MelDataset(Dataset):

    def __init__(self, split_dir, class_to_idx):
        self.samples = []
        for cls, idx in class_to_idx.items():
            cdir = split_dir / cls
            if not cdir.is_dir():
                print(f"  !! {split_dir.name}: no folder for class '{cls}'", flush=True)
                continue
            for npy in sorted(cdir.glob("*.npy")):
                self.samples.append((npy, idx))
        self.resize = VT.Resize((IMG_SIZE, IMG_SIZE), antialias=True)
        self.normalize = VT.Normalize(IMAGENET_MEAN, IMAGENET_STD)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        arr = np.load(path).astype(np.float32)
        t = torch.from_numpy(arr)

        if t.ndim == 2:
            t = t.unsqueeze(0)
        if t.shape[0] == 1:
            t = t.repeat(3, 1, 1)

        return self.normalize(self.resize(t)), label


def build_loaders():
    root = find_split_root(DATA_DIR)
    if root is None:
        raise SystemExit(f"could not find train/validation splits under {DATA_DIR}")

    classes = sorted(d.name for d in (root / "train").iterdir() if d.is_dir())
    class_to_idx = {c: i for i, c in enumerate(classes)}

    train_ds = MelDataset(root / "train", class_to_idx)
    val_ds = MelDataset(root / "validation", class_to_idx)
    print(f"classes: {classes}", flush=True)
    print(f"train clips {len(train_ds)} | validation clips {len(val_ds)}\n", flush=True)

    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"),
                              drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
    return train_loader, val_loader, classes


def build_model(num_classes):
    """torchvision VGG19 (no BatchNorm) with a fresh classifier head."""
    model = vgg19(weights=VGG19_Weights.IMAGENET1K_V1)
    model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
    return model.to(DEVICE)


def forward(model, x):
    return model(x)


@torch.no_grad()
def validate_accuracy(model, loader):
    model.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
            logits = forward(model, x)
        correct += (logits.argmax(1) == y).sum().item()
        total += y.numel()
    return correct / max(total, 1)


def train_one_config(train_loader, val_loader, lr, epochs, num_classes):
    model = build_model(num_classes)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()                       # no label smoothing
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

    acc, state = 0.0, None
    try:
        for epoch in range(epochs):
            model.train()
            run_loss, seen = 0.0, 0
            for x, y in train_loader:
                x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
                    loss = criterion(forward(model, x), y)
                if not torch.isfinite(loss):
                    continue
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                run_loss += loss.item() * x.size(0)
                seen += x.size(0)

            acc = validate_accuracy(model, val_loader)
            print(f"      epoch {epoch + 1:>3}/{epochs}  "
                  f"train_loss {run_loss / max(seen, 1):.4f}  val_acc {acc * 100:.2f}%",
                  flush=True)

        state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    finally:
        del model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    return acc, state


def new_grid():
    return [["" for _ in GRID_EPOCHS] for _ in GRID_LRS]


def save_grid(grid):
    """Rewrite the results CSV: one row per LR, one column per epoch count."""
    lines = ["Epochs / LR," + ",".join(str(e) for e in GRID_EPOCHS)]
    for lr, row in zip(GRID_LRS, grid):
        lines.append(f"{lr}," + ",".join(str(v) for v in row))
    (OUTPUT_DIR / RESULTS_CSV_NAME).write_text("\n".join(lines) + "\n")


def save_best(state, classes, lr, epochs, acc):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    path = OUTPUT_DIR / "vgg19_mel_best.pt"
    torch.save({"model": "vgg19", "model_name": "torchvision vgg19 (IMAGENET1K_V1)",
                "representation": "mel", "classes": classes,
                "normalization": {"mean": IMAGENET_MEAN, "std": IMAGENET_STD},
                "hyperparameters": {"lr": lr, "epochs": epochs, "batch": BATCH,
                                    "optimizer": OPTIMIZER, "weight_decay": WEIGHT_DECAY,
                                    "augmentation": not NO_AUGMENTATION},
                "val_accuracy": acc, "model_state": state}, path)
    return path


def main():
    print(f"Device: {DEVICE}", flush=True)
    if DEVICE.type == "cuda":
        print(f"  GPU: {torch.cuda.get_device_name(0)}", flush=True)
    print("VGG19 (torchvision, ImageNet weights) + mel-spectrogram", flush=True)
    print(f"optimizer {OPTIMIZER} | batch {BATCH} | wd {WEIGHT_DECAY:.0e} | "
          f"no augmentation | no early stopping", flush=True)
    total_epochs = len(GRID_LRS) * sum(GRID_EPOCHS)
    print(f"grid: {len(GRID_LRS)} LRs x {GRID_EPOCHS} epochs = "
          f"{len(GRID_LRS) * len(GRID_EPOCHS)} cells, {total_epochs} epochs total\n",
          flush=True)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    grid = new_grid()
    for (lr, epochs), acc in PRIOR_RESULTS.items():
        if lr in GRID_LRS and epochs in GRID_EPOCHS:
            grid[GRID_LRS.index(lr)][GRID_EPOCHS.index(epochs)] = round(acc, 4)
    save_grid(grid)
    print(f"results grid -> {OUTPUT_DIR / RESULTS_CSV_NAME}\n", flush=True)

    train_loader, val_loader, classes = build_loaders()
    num_classes = len(classes)

    if PRIOR_RESULTS:
        pk = max(PRIOR_RESULTS, key=PRIOR_RESULTS.get)
        best = {"acc": PRIOR_RESULTS[pk], "lr": pk[0], "epochs": pk[1], "state": None}
        print(f"seeded best from PRIOR_RESULTS: lr {best['lr']:g}, "
              f"epochs {best['epochs']}, val_acc {best['acc']:.4f}\n", flush=True)
    else:
        best = {"acc": -1.0, "lr": None, "epochs": 0, "state": None}

    print(f"resuming at lr {START_LR:g}, epochs {START_EPOCHS}\n", flush=True)
    started = False
    grid_t0 = time.time()

    for li, lr in enumerate(GRID_LRS):
        for ej, epochs in enumerate(GRID_EPOCHS):
            if not started:
                if abs(lr - START_LR) <= 1e-12 and epochs == START_EPOCHS:
                    started = True
                else:
                    continue
            print("#" * 72, flush=True)
            print(f"lr {lr:g} | epochs {epochs}  (fresh model)", flush=True)
            print("#" * 72, flush=True)

            t0 = time.time()
            acc, state = train_one_config(train_loader, val_loader, lr, epochs, num_classes)
            mins = (time.time() - t0) / 60

            grid[li][ej] = round(acc, 4)
            save_grid(grid)
            print(f"  >> val_acc {acc * 100:.2f}%  ({mins:.1f} min)  "
                  f"-> grid row {li + 2}, col {ej + 2}", flush=True)

            if state is not None and acc > best["acc"]:
                best = {"acc": acc, "lr": lr, "epochs": epochs, "state": state}
                path = save_best(state, classes, lr, epochs, acc)
                print(f"  >> NEW BEST -> {path}", flush=True)

            elapsed = (time.time() - grid_t0) / 3600
            print(f"  >> elapsed this session: {elapsed:.2f} h\n", flush=True)

    if not started:
        print(f"!! resume point (lr {START_LR:g}, epochs {START_EPOCHS}) is not in the "
              f"grid - nothing ran", flush=True)
        return

    print("=" * 60, flush=True)
    if best["lr"] is None:
        print("no successful cell; nothing saved", flush=True)
    elif best["state"] is not None:
        print(f"BEST: lr {best['lr']:g}, epochs {best['epochs']}, "
              f"val_acc {best['acc'] * 100:.2f}%", flush=True)
        print(f"  weights -> {OUTPUT_DIR / 'vgg19_mel_best.pt'}", flush=True)
    else:
        print(f"BEST is a cell from an earlier session: lr {best['lr']:g}, "
              f"epochs {best['epochs']} (val_acc {best['acc']:.4f}).", flush=True)
        print("retraining it once to produce its weights...", flush=True)
        acc, state = train_one_config(train_loader, val_loader,
                                      best["lr"], best["epochs"], num_classes)
        if state is not None:
            path = save_best(state, classes, best["lr"], best["epochs"], acc)
            print(f"  retrained val_acc {acc * 100:.2f}%  -> {path}", flush=True)
    print("=" * 60, flush=True)
    print("\nTHE TASK HAS BEEN COMPLETED.", flush=True)


if __name__ == "__main__":
    main()


##Mel + ViT (training)

In [ ]:
#This script trains ViT on the mel-spectrogram dataset over a grid of
#learning rates and epoch counts. Reads DATA_DIR/<split>/<class>/*.npy, writes
#the accuracy grid to OUTPUT_DIR/results_grid.csv and the best weights to
#OUTPUT_DIR/vit_mel_best.pt

import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as VT
from transformers import ViTForImageClassification

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **k):
        return x


# ======================= EDIT THESE =======================
DATA_DIR = OUTPUTS / "mel_dataset"       # folder holding <split>/<class>/*.npy (the mel cell's OUTPUT_DIR)
OUTPUT_DIR = OUTPUTS / "vit_mel"         # where the results grid CSV and best weights will be written

MODEL_NAME = "google/vit-base-patch16-224"

OPTIMIZER = "adamw"
BATCH = 64
WEIGHT_DECAY = 1e-5
NO_AUGMENTATION = True

GRID_LRS = [0.05, 0.01, 0.005, 0.001, 0.0001, 0.00001, 0.000001]
GRID_EPOCHS = [10, 20, 30, 40, 50]

# Resume point: the first grid cell that will actually run. Cells before it are
# skipped, so set these to pick up where an earlier session stopped.
START_LR = 0.05
START_EPOCHS = 10

# Accuracies from earlier sessions, e.g. {(0.05, 10): 0.7212}. Used to seed the
# best cell so a resumed session only saves weights that beat it.
PRIOR_RESULTS = {}

IMG_SIZE = 224
NUM_WORKERS = 2      # set to 0 if the DataLoader gives trouble in the notebook
VIT_MEAN, VIT_STD = [0.5, 0.5, 0.5], [0.5, 0.5, 0.5]
# ==========================================================

RESULTS_CSV_NAME = "results_grid.csv"

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)


def lr_tag(lr):
    return f"{lr:.0e}"


def find_split_root(base):
    base = Path(base)
    if not base.exists():
        return None
    for d in [base] + sorted(p for p in base.iterdir() if p.is_dir()):
        if (d / "train").is_dir() and (d / "validation").is_dir() \
                and next((d / "validation").rglob("*.npy"), None) is not None:
            return d
    return None


class MelDataset(Dataset):

    def __init__(self, split_dir, class_to_idx):
        self.samples = []
        for cls, idx in class_to_idx.items():
            cdir = split_dir / cls
            if not cdir.is_dir():
                print(f"  !! {split_dir.name}: no folder for class '{cls}'", flush=True)
                continue
            for npy in sorted(cdir.glob("*.npy")):
                self.samples.append((npy, idx))
        self.resize = VT.Resize((IMG_SIZE, IMG_SIZE), antialias=True)
        self.normalize = VT.Normalize(VIT_MEAN, VIT_STD)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        arr = np.load(path).astype(np.float32)
        t = torch.from_numpy(arr)

        if t.ndim == 2:
            t = t.unsqueeze(0)
        if t.shape[0] == 1:
            t = t.repeat(3, 1, 1)

        return self.normalize(self.resize(t)), label


def build_loaders():
    root = find_split_root(DATA_DIR)
    if root is None:
        raise SystemExit(f"could not find train/validation splits under {DATA_DIR}")

    classes = sorted(d.name for d in (root / "train").iterdir() if d.is_dir())
    class_to_idx = {c: i for i, c in enumerate(classes)}

    train_ds = MelDataset(root / "train", class_to_idx)
    val_ds = MelDataset(root / "validation", class_to_idx)
    print(f"classes: {classes}", flush=True)
    print(f"train clips {len(train_ds)} | validation clips {len(val_ds)}\n", flush=True)

    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"),
                              drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
    return train_loader, val_loader, classes


def build_model(num_classes):
    model = ViTForImageClassification.from_pretrained(
        MODEL_NAME, num_labels=num_classes, ignore_mismatched_sizes=True)
    return model.to(DEVICE)


def forward(model, x):
    return model(pixel_values=x).logits


@torch.no_grad()
def validate_accuracy(model, loader):
    model.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
            logits = forward(model, x)
        correct += (logits.argmax(1) == y).sum().item()
        total += y.numel()
    return correct / max(total, 1)


def train_one_config(train_loader, val_loader, lr, epochs, num_classes):
    model = build_model(num_classes)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()                       # no label smoothing
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

    acc, state = 0.0, None
    try:
        for epoch in range(epochs):
            model.train()
            run_loss, seen = 0.0, 0
            for x, y in train_loader:
                x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
                    loss = criterion(forward(model, x), y)
                if not torch.isfinite(loss):
                    continue
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                run_loss += loss.item() * x.size(0)
                seen += x.size(0)

            acc = validate_accuracy(model, val_loader)
            print(f"      epoch {epoch + 1:>3}/{epochs}  "
                  f"train_loss {run_loss / max(seen, 1):.4f}  val_acc {acc * 100:.2f}%",
                  flush=True)

        state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    finally:
        del model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    return acc, state


def new_grid():
    return [["" for _ in GRID_EPOCHS] for _ in GRID_LRS]


def save_grid(grid):
    """Rewrite the results CSV: one row per LR, one column per epoch count."""
    lines = ["Epochs / LR," + ",".join(str(e) for e in GRID_EPOCHS)]
    for lr, row in zip(GRID_LRS, grid):
        lines.append(f"{lr}," + ",".join(str(v) for v in row))
    (OUTPUT_DIR / RESULTS_CSV_NAME).write_text("\n".join(lines) + "\n")


def save_best(state, classes, lr, epochs, acc):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    path = OUTPUT_DIR / "vit_mel_best.pt"
    torch.save({"model": "vit", "model_name": MODEL_NAME,
                "representation": "mel", "classes": classes,
                "normalization": {"mean": VIT_MEAN, "std": VIT_STD},
                "hyperparameters": {"lr": lr, "epochs": epochs, "batch": BATCH,
                                    "optimizer": OPTIMIZER, "weight_decay": WEIGHT_DECAY,
                                    "augmentation": not NO_AUGMENTATION},
                "val_accuracy": acc, "model_state": state}, path)
    return path


def main():
    print(f"Device: {DEVICE}", flush=True)
    if DEVICE.type == "cuda":
        print(f"  GPU: {torch.cuda.get_device_name(0)}", flush=True)
    print(f"ViT ({MODEL_NAME}) + mel-spectrogram", flush=True)
    print(f"optimizer {OPTIMIZER} | batch {BATCH} | wd {WEIGHT_DECAY:.0e} | "
          f"no augmentation | no early stopping", flush=True)
    print(f"normalisation: mean {VIT_MEAN[0]} / std {VIT_STD[0]} (ViT, not ImageNet)",
          flush=True)
    total_epochs = len(GRID_LRS) * sum(GRID_EPOCHS)
    print(f"grid: {len(GRID_LRS)} LRs x {GRID_EPOCHS} epochs = "
          f"{len(GRID_LRS) * len(GRID_EPOCHS)} cells, {total_epochs} epochs total\n",
          flush=True)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    grid = new_grid()
    for (lr, epochs), acc in PRIOR_RESULTS.items():
        if lr in GRID_LRS and epochs in GRID_EPOCHS:
            grid[GRID_LRS.index(lr)][GRID_EPOCHS.index(epochs)] = round(acc, 4)
    save_grid(grid)
    print(f"results grid -> {OUTPUT_DIR / RESULTS_CSV_NAME}\n", flush=True)

    train_loader, val_loader, classes = build_loaders()
    num_classes = len(classes)

    if PRIOR_RESULTS:
        pk = max(PRIOR_RESULTS, key=PRIOR_RESULTS.get)
        best = {"acc": PRIOR_RESULTS[pk], "lr": pk[0], "epochs": pk[1], "state": None}
        print(f"seeded best from PRIOR_RESULTS: lr {best['lr']:g}, "
              f"epochs {best['epochs']}, val_acc {best['acc']:.4f}\n", flush=True)
    else:
        best = {"acc": -1.0, "lr": None, "epochs": 0, "state": None}

    print(f"resuming at lr {START_LR:g}, epochs {START_EPOCHS}\n", flush=True)
    started = False
    grid_t0 = time.time()

    for li, lr in enumerate(GRID_LRS):
        for ej, epochs in enumerate(GRID_EPOCHS):
            if not started:
                if abs(lr - START_LR) <= 1e-12 and epochs == START_EPOCHS:
                    started = True
                else:
                    continue
            print("#" * 72, flush=True)
            print(f"lr {lr:g} | epochs {epochs}  (fresh model)", flush=True)
            print("#" * 72, flush=True)

            t0 = time.time()
            acc, state = train_one_config(train_loader, val_loader, lr, epochs, num_classes)
            mins = (time.time() - t0) / 60

            grid[li][ej] = round(acc, 4)
            save_grid(grid)
            print(f"  >> val_acc {acc * 100:.2f}%  ({mins:.1f} min)  "
                  f"-> grid row {li + 2}, col {ej + 2}", flush=True)

            if state is not None and acc > best["acc"]:
                best = {"acc": acc, "lr": lr, "epochs": epochs, "state": state}
                path = save_best(state, classes, lr, epochs, acc)
                print(f"  >> NEW BEST -> {path}", flush=True)

            elapsed = (time.time() - grid_t0) / 3600
            print(f"  >> elapsed this session: {elapsed:.2f} h\n", flush=True)

    if not started:
        print(f"!! resume point (lr {START_LR:g}, epochs {START_EPOCHS}) is not in the "
              f"grid - nothing ran", flush=True)
        return

    print("=" * 60, flush=True)
    if best["lr"] is None:
        print("no successful cell; nothing saved", flush=True)
    elif best["state"] is not None:
        print(f"BEST: lr {best['lr']:g}, epochs {best['epochs']}, "
              f"val_acc {best['acc'] * 100:.2f}%", flush=True)
        print(f"  weights -> {OUTPUT_DIR / 'vit_mel_best.pt'}", flush=True)
    else:
        print(f"BEST is a cell from an earlier session: lr {best['lr']:g}, "
              f"epochs {best['epochs']} (val_acc {best['acc']:.4f}).", flush=True)
        print("retraining it once to produce its weights...", flush=True)
        acc, state = train_one_config(train_loader, val_loader,
                                      best["lr"], best["epochs"], num_classes)
        if state is not None:
            path = save_best(state, classes, best["lr"], best["epochs"], acc)
            print(f"  retrained val_acc {acc * 100:.2f}%  -> {path}", flush=True)
    print("=" * 60, flush=True)
    print("\nTHE TASK HAS BEEN COMPLETED.", flush=True)


if __name__ == "__main__":
    main()


##WAV + AST (precompute for faster training)

In [ ]:
#This script precomputes AST input features for every wav clip (all splits)
#into a folder cache, so the training and testing cells skip extraction and
#start immediately. Safe to re-run: existing cache files are skipped.

import sys
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np

# ======================= EDIT THESE =======================
WAV_DIR = OUTPUTS / "cumulative_dataset"   # folder holding <split>/<class>/*.wav
FEATURE_CACHE = OUTPUTS / "ast_features"   # folder where the AST feature cache will be written

MODEL_NAME = "MIT/ast-finetuned-speech-commands-v2"

SPLITS = ["train", "validation", "test"]

SR = 16000
CLIP_SAMPLES = SR
MAX_LENGTH = 128                  # matches config.json time_dimension
NUM_MEL_BINS = 128
CACHE_DTYPE = np.float32

THREADS = 8
SKIP_EXISTING = True              # resume a partial cache
# ==========================================================

_FE = None
_FE_LOCK = threading.Lock()


def feature_extractor():
    """Loaded once in the parent; shared by every thread."""
    global _FE
    if _FE is None:
        with _FE_LOCK:
            if _FE is None:
                from transformers import ASTFeatureExtractor
                _FE = ASTFeatureExtractor.from_pretrained(
                    MODEL_NAME, max_length=MAX_LENGTH, num_mel_bins=NUM_MEL_BINS)
    return _FE


def find_split_root(base, ext):
    base = Path(base)
    if not base.exists():
        return None
    for d in [base] + sorted(p for p in base.iterdir() if p.is_dir()):
        if (d / "train").is_dir() and (d / "validation").is_dir() \
                and next((d / "validation").rglob(f"*{ext}"), None) is not None:
            return d
    return None


def extract_one(job):
    src, dst = job
    try:
        if SKIP_EXISTING and dst.exists() and dst.stat().st_size > 0:
            return "skip", None
        import librosa
        y, _ = librosa.load(str(src), sr=SR, mono=True)
        if len(y) < CLIP_SAMPLES:
            y = np.pad(y, (0, CLIP_SAMPLES - len(y)))
        else:
            y = y[:CLIP_SAMPLES]
        feats = feature_extractor()(y, sampling_rate=SR, return_tensors="np")
        arr = feats["input_values"][0].astype(CACHE_DTYPE)
        np.save(dst, arr)
        return "ok", arr.shape
    except Exception as exc:                                # noqa: BLE001
        return "fail", f"{src}: {exc}"


def main():
    try:
        import librosa                                      # noqa: F401
        import transformers                                 # noqa: F401
    except ImportError:
        sys.exit("needs librosa and transformers:  pip install transformers librosa")

    wav_root = find_split_root(WAV_DIR, ".wav")
    if wav_root is None:
        raise SystemExit(f"no wavs under {WAV_DIR} - build the cumulative dataset first")
    print(f"wav root: {wav_root}", flush=True)

    # load the extractor up front: one HF call, not one per worker
    print(f"loading {MODEL_NAME} feature extractor "
          f"(max_length {MAX_LENGTH}, {NUM_MEL_BINS} mels)...", flush=True)
    t0 = time.time()
    feature_extractor()
    print(f"  loaded in {time.time() - t0:.1f}s\n", flush=True)

    jobs, per_split = [], {}
    for split in SPLITS:
        sdir = wav_root / split
        if not sdir.is_dir():
            continue
        counts = {}
        for cdir in sorted(p for p in sdir.iterdir() if p.is_dir()):
            out_dir = FEATURE_CACHE / split / cdir.name
            out_dir.mkdir(parents=True, exist_ok=True)      # created even if empty
            wavs = sorted(cdir.glob("*.wav"))
            for w in wavs:
                jobs.append((w, out_dir / f"{w.stem}.npy"))
            counts[cdir.name] = len(wavs)
        per_split[split] = counts

    if not jobs:
        raise SystemExit(f"no wavs found under {wav_root}")

    print("source clips:", flush=True)
    for split, counts in per_split.items():
        detail = "  ".join(f"{c}={n}" for c, n in sorted(counts.items()))
        print(f"  {split:11} {sum(counts.values()):>6}   {detail}", flush=True)
    est_gb = len(jobs) * MAX_LENGTH * NUM_MEL_BINS * np.dtype(CACHE_DTYPE).itemsize / 1e9
    print(f"  {'TOTAL':11} {len(jobs):>6}   (~{est_gb:.2f} GB cache)\n", flush=True)

    print(f"extracting on {THREADS} threads...", flush=True)
    t0 = time.time()
    ok = skip = 0
    failures, shapes = [], set()
    with ThreadPoolExecutor(max_workers=THREADS) as pool:
        futs = [pool.submit(extract_one, j) for j in jobs]
        for i, fut in enumerate(as_completed(futs), 1):
            status, info = fut.result()
            if status == "ok":
                ok += 1
                shapes.add(info)
            elif status == "skip":
                skip += 1
            else:
                failures.append(info)
            if i % 1000 == 0 or i == len(jobs):
                el = time.time() - t0
                rate = i / max(el, 1e-9)
                eta = (len(jobs) - i) / max(rate, 1e-9)
                print(f"  {i}/{len(jobs)}  ({rate:.0f} clips/s, eta {eta / 60:.1f} min)",
                      flush=True)

    print(f"\nextracted {ok}, skipped {skip} in {(time.time() - t0) / 60:.1f} min",
          flush=True)
    print(f"feature shapes: {sorted(shapes)}  "
          f"(expect ({MAX_LENGTH}, {NUM_MEL_BINS}))", flush=True)
    if len(shapes) > 1:
        print("  !! more than one shape - inspect before training", flush=True)
    if failures:
        print(f"  !! {len(failures)} failed:", flush=True)
        for f in failures[:10]:
            print(f"     {f}", flush=True)

    n = sum(1 for _ in FEATURE_CACHE.rglob("*.npy"))
    mb = sum(p.stat().st_size for p in FEATURE_CACHE.rglob("*.npy")) / 1e6
    print(f"\ncache: {n} .npy files ({mb:.0f} MB) under {FEATURE_CACHE}", flush=True)
    if n == len(jobs):
        print("complete - the AST training and testing cells will now skip extraction.",
              flush=True)
    else:
        print(f"  !! expected {len(jobs)} - re-run to fill the gaps (existing files "
              f"are skipped).", flush=True)


if __name__ == "__main__":
    main()


WAV + AST (training)

In [ ]:
#This script trains AST on the raw-wav dataset over a grid of learning rates
#and epoch counts. Reads DATA_DIR/<split>/<class>/*.wav (via the AST feature
#cache at FEATURE_CACHE, built here if the precompute cell was not run), writes
#the accuracy grid to OUTPUT_DIR/results_grid.csv and the best weights to
#OUTPUT_DIR/ast_wav_best.pt

import sys
import time
from multiprocessing import Pool, cpu_count
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import ASTFeatureExtractor, ASTForAudioClassification

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **k):
        return x


# ======================= EDIT THESE =======================
DATA_DIR = OUTPUTS / "cumulative_dataset"   # folder holding <split>/<class>/*.wav
FEATURE_CACHE = OUTPUTS / "ast_features"    # AST feature cache folder (the precompute cell's FEATURE_CACHE)
OUTPUT_DIR = OUTPUTS / "ast_wav"            # where the results grid CSV and best weights will be written

MODEL_NAME = "MIT/ast-finetuned-speech-commands-v2"

SPLITS = ["train", "validation", "test"]

SR = 16000
CLIP_SAMPLES = SR                        # clips are 1 s
MAX_LENGTH = 128                         # MUST match config.json time_dimension
NUM_MEL_BINS = 128

USE_FEATURE_CACHE = True
CACHE_DTYPE = np.float32                 # np.float16 halves the cache

OPTIMIZER = "adamw"
BATCH = 64
WEIGHT_DECAY = 1e-5
NO_AUGMENTATION = True                   # no SpecAugment, plain CE

GRID_LRS = [0.05, 0.01, 0.005, 0.001, 0.0001, 0.00001, 0.000001]   # grid rows
GRID_EPOCHS = [10, 20, 30, 40, 50]                                  # grid columns

# ---- resume point: every cell before this (row-major) is skipped ----
START_LR = 0.05
START_EPOCHS = 10

# ---- accuracies from earlier sessions, e.g. (0.05, 10): 0.2005 ----
PRIOR_RESULTS = {}

NUM_WORKERS = 2       # DataLoader workers; set to 0 if it gives trouble in the notebook
EXTRACT_WORKERS = max(1, cpu_count() - 1)   # feature extraction; set to 1 if multiprocessing gives trouble
# ==========================================================

RESULTS_CSV_NAME = "results_grid.csv"

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)


def find_split_root(base, ext):
    base = Path(base)
    if not base.exists():
        return None
    for d in [base] + sorted(p for p in base.iterdir() if p.is_dir()):
        if (d / "train").is_dir() and (d / "validation").is_dir() \
                and next((d / "validation").rglob(f"*{ext}"), None) is not None:
            return d
    return None


# ---------------------------- feature cache ---------------------------------

_FE = None


def _feature_extractor():
    """AST feature extractor pinned to MAX_LENGTH (the checkpoint omits it)."""
    global _FE
    if _FE is None:
        _FE = ASTFeatureExtractor.from_pretrained(
            MODEL_NAME, max_length=MAX_LENGTH, num_mel_bins=NUM_MEL_BINS)
    return _FE


def _extract_one(job):
    src, dst = job
    try:
        import librosa
        y, _ = librosa.load(str(src), sr=SR, mono=True)
        if len(y) < CLIP_SAMPLES:
            y = np.pad(y, (0, CLIP_SAMPLES - len(y)))
        else:
            y = y[:CLIP_SAMPLES]
        feats = _feature_extractor()(y, sampling_rate=SR, return_tensors="np")
        arr = feats["input_values"][0].astype(CACHE_DTYPE)      # (MAX_LENGTH, mels)
        np.save(dst, arr)
        return True, arr.shape
    except Exception as exc:                                    # noqa: BLE001
        return False, f"{src}: {exc}"


def build_feature_cache(wav_root):
    """Extract AST features once for every clip; return the cache root."""
    existing = find_split_root(FEATURE_CACHE, ".npy")
    if existing is not None:
        print(f"feature cache already present at {existing}", flush=True)
        return existing

    jobs, per_split = [], {}
    for split in SPLITS:
        sdir = wav_root / split
        if not sdir.is_dir():
            continue
        counts = {}
        for cdir in sorted(p for p in sdir.iterdir() if p.is_dir()):
            out_dir = FEATURE_CACHE / split / cdir.name
            out_dir.mkdir(parents=True, exist_ok=True)
            wavs = sorted(cdir.glob("*.wav"))
            for w in wavs:
                jobs.append((w, out_dir / f"{w.stem}.npy"))
            counts[cdir.name] = len(wavs)
        per_split[split] = counts

    if not jobs:
        raise SystemExit(f"no wavs found under {wav_root}")

    print("source clips:", flush=True)
    for split, counts in per_split.items():
        detail = "  ".join(f"{c}={n}" for c, n in sorted(counts.items()))
        print(f"  {split:11} {sum(counts.values()):>6}   {detail}", flush=True)

    est_gb = len(jobs) * MAX_LENGTH * NUM_MEL_BINS * np.dtype(CACHE_DTYPE).itemsize / 1e9
    print(f"\nextracting AST features for {len(jobs)} clips on {EXTRACT_WORKERS} "
          f"workers (~{est_gb:.2f} GB, once)...", flush=True)

    t0, done, failed, shapes = time.time(), 0, [], set()

    def handle(ok, info):
        if ok:
            shapes.add(info)
        else:
            failed.append(info)

    if EXTRACT_WORKERS == 1:
        for job in jobs:
            ok, info = _extract_one(job)
            handle(ok, info)
            done += 1
            if done % 2000 == 0 or done == len(jobs):
                print(f"  {done}/{len(jobs)}  ({done / max(time.time() - t0, 1e-9):.0f}/s)",
                      flush=True)
    else:
        with Pool(EXTRACT_WORKERS) as pool:
            for ok, info in pool.imap_unordered(_extract_one, jobs, chunksize=32):
                handle(ok, info)
                done += 1
                if done % 2000 == 0 or done == len(jobs):
                    print(f"  {done}/{len(jobs)}  "
                          f"({done / max(time.time() - t0, 1e-9):.0f}/s)", flush=True)
    print(f"  done in {(time.time() - t0) / 60:.1f} min", flush=True)
    print(f"  feature shapes: {sorted(shapes)}  "
          f"(expect ({MAX_LENGTH}, {NUM_MEL_BINS}))", flush=True)
    if len(shapes) > 1:
        print("  !! more than one shape - inspect before training", flush=True)
    if failed:
        print(f"  !! {len(failed)} clip(s) failed:", flush=True)
        for msg in failed[:10]:
            print(f"     {msg}", flush=True)

    root = find_split_root(FEATURE_CACHE, ".npy")
    if root is None:
        raise SystemExit("feature cache build produced no usable splits")
    return root


# ---------------------------- datasets --------------------------------------

class CachedFeatureDataset(Dataset):
    """Precomputed AST features (MAX_LENGTH, mels). Already normalised."""

    def __init__(self, split_dir, class_to_idx):
        self.samples = []
        for cls, idx in class_to_idx.items():
            cdir = split_dir / cls
            if not cdir.is_dir():
                print(f"  !! {split_dir.name}: no folder for class '{cls}'", flush=True)
                continue
            for npy in sorted(cdir.glob("*.npy")):
                self.samples.append((npy, idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        return torch.from_numpy(np.load(path).astype(np.float32)), label


class WavDataset(Dataset):
    """Fallback: extract AST features from the wav on the fly (slow)."""

    def __init__(self, split_dir, class_to_idx):
        self.samples = []
        for cls, idx in class_to_idx.items():
            cdir = split_dir / cls
            if not cdir.is_dir():
                print(f"  !! {split_dir.name}: no folder for class '{cls}'", flush=True)
                continue
            for w in sorted(cdir.glob("*.wav")):
                self.samples.append((w, idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        import librosa
        path, label = self.samples[i]
        y, _ = librosa.load(str(path), sr=SR, mono=True)
        if len(y) < CLIP_SAMPLES:
            y = np.pad(y, (0, CLIP_SAMPLES - len(y)))
        else:
            y = y[:CLIP_SAMPLES]
        feats = _feature_extractor()(y, sampling_rate=SR, return_tensors="pt")
        return feats["input_values"].squeeze(0), label


def build_loaders():
    wav_root = find_split_root(DATA_DIR, ".wav")
    if wav_root is None:
        raise SystemExit(f"could not find train/validation wav splits under {DATA_DIR}")
    if USE_FEATURE_CACHE:
        root = build_feature_cache(wav_root)
        make = CachedFeatureDataset
    else:
        root, make = wav_root, WavDataset

    classes = sorted(d.name for d in (root / "train").iterdir() if d.is_dir())
    class_to_idx = {c: i for i, c in enumerate(classes)}

    train_ds = make(root / "train", class_to_idx)
    val_ds = make(root / "validation", class_to_idx)
    print(f"\nclasses: {classes}", flush=True)
    print(f"train clips {len(train_ds)} | validation clips {len(val_ds)}", flush=True)
    if len(train_ds):
        probe, _ = train_ds[0]
        print(f"sample input shape: {tuple(probe.shape)}  "
              f"(expect ({MAX_LENGTH}, {NUM_MEL_BINS}))\n", flush=True)

    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"),
                              drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
    return train_loader, val_loader, classes


# ---------------------------- model -----------------------------------------

def build_model(num_classes):
    model = ASTForAudioClassification.from_pretrained(
        MODEL_NAME, num_labels=num_classes, ignore_mismatched_sizes=True)
    return model.to(DEVICE)


def forward(model, x):
    return model(input_values=x).logits


@torch.no_grad()
def validate_accuracy(model, loader):
    model.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
            logits = forward(model, x)
        correct += (logits.argmax(1) == y).sum().item()
        total += y.numel()
    return correct / max(total, 1)


def train_one_config(train_loader, val_loader, lr, epochs, num_classes):
    """Train from scratch for exactly `epochs` epochs. Returns (val_acc, state_dict)."""
    model = build_model(num_classes)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()                       # no label smoothing
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

    acc, state = 0.0, None
    try:
        for epoch in range(epochs):
            model.train()
            run_loss, seen = 0.0, 0
            for x, y in train_loader:
                x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
                    loss = criterion(forward(model, x), y)
                if not torch.isfinite(loss):
                    continue
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                run_loss += loss.item() * x.size(0)
                seen += x.size(0)

            acc = validate_accuracy(model, val_loader)
            print(f"      epoch {epoch + 1:>3}/{epochs}  "
                  f"train_loss {run_loss / max(seen, 1):.4f}  val_acc {acc * 100:.2f}%",
                  flush=True)

        state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    finally:
        del model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    return acc, state


# ---------------------------- results ---------------------------------------

def new_grid():
    return [["" for _ in GRID_EPOCHS] for _ in GRID_LRS]


def save_grid(grid):
    """Rewrite the results CSV: one row per LR, one column per epoch count."""
    lines = ["Epochs / LR," + ",".join(str(e) for e in GRID_EPOCHS)]
    for lr, row in zip(GRID_LRS, grid):
        lines.append(f"{lr}," + ",".join(str(v) for v in row))
    (OUTPUT_DIR / RESULTS_CSV_NAME).write_text("\n".join(lines) + "\n")


def save_best(state, classes, lr, epochs, acc):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    path = OUTPUT_DIR / "ast_wav_best.pt"
    torch.save({"model": "ast", "model_name": MODEL_NAME,
                "representation": "wav", "classes": classes,
                "feature_extraction": {"sampling_rate": SR, "max_length": MAX_LENGTH,
                                       "num_mel_bins": NUM_MEL_BINS},
                "hyperparameters": {"lr": lr, "epochs": epochs, "batch": BATCH,
                                    "optimizer": OPTIMIZER, "weight_decay": WEIGHT_DECAY,
                                    "augmentation": not NO_AUGMENTATION},
                "val_accuracy": acc, "model_state": state}, path)
    return path


# ---------------------------- main ------------------------------------------

def main():
    try:
        import librosa                                  # noqa: F401
    except ImportError:
        sys.exit("this script needs librosa:  pip install librosa")

    print(f"Device: {DEVICE}", flush=True)
    if DEVICE.type == "cuda":
        print(f"  GPU: {torch.cuda.get_device_name(0)}", flush=True)
    print(f"AST ({MODEL_NAME}) + raw WAV", flush=True)
    print(f"optimizer {OPTIMIZER} | batch {BATCH} | wd {WEIGHT_DECAY:.0e} | "
          f"no augmentation | no early stopping", flush=True)
    print(f"features: max_length {MAX_LENGTH} x {NUM_MEL_BINS} mels @ {SR} Hz "
          f"(max_length pinned to config.json time_dimension)", flush=True)
    print(f"feature cache: {'on' if USE_FEATURE_CACHE else 'OFF (slow)'}", flush=True)
    total_epochs = len(GRID_LRS) * sum(GRID_EPOCHS)
    print(f"grid: {len(GRID_LRS)} LRs x {GRID_EPOCHS} epochs = "
          f"{len(GRID_LRS) * len(GRID_EPOCHS)} cells, {total_epochs} epochs total\n",
          flush=True)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    grid = new_grid()
    for (lr, epochs), acc in PRIOR_RESULTS.items():
        if lr in GRID_LRS and epochs in GRID_EPOCHS:
            grid[GRID_LRS.index(lr)][GRID_EPOCHS.index(epochs)] = round(acc, 4)
    save_grid(grid)
    print(f"results grid -> {OUTPUT_DIR / RESULTS_CSV_NAME}\n", flush=True)

    train_loader, val_loader, classes = build_loaders()
    num_classes = len(classes)

    # seed the running best from cells finished in earlier sessions
    if PRIOR_RESULTS:
        pk = max(PRIOR_RESULTS, key=PRIOR_RESULTS.get)
        best = {"acc": PRIOR_RESULTS[pk], "lr": pk[0], "epochs": pk[1], "state": None}
        print(f"seeded best from PRIOR_RESULTS: lr {best['lr']:g}, "
              f"epochs {best['epochs']}, val_acc {best['acc']:.4f}\n", flush=True)
    else:
        best = {"acc": -1.0, "lr": None, "epochs": 0, "state": None}

    print(f"resuming at lr {START_LR:g}, epochs {START_EPOCHS}\n", flush=True)
    started = False
    grid_t0 = time.time()

    for li, lr in enumerate(GRID_LRS):
        for ej, epochs in enumerate(GRID_EPOCHS):
            if not started:
                if abs(lr - START_LR) <= 1e-12 and epochs == START_EPOCHS:
                    started = True
                else:
                    continue
            print("#" * 72, flush=True)
            print(f"lr {lr:g} | epochs {epochs}  (fresh model)", flush=True)
            print("#" * 72, flush=True)

            t0 = time.time()
            acc, state = train_one_config(train_loader, val_loader, lr, epochs, num_classes)
            mins = (time.time() - t0) / 60

            grid[li][ej] = round(acc, 4)
            save_grid(grid)
            print(f"  >> val_acc {acc * 100:.2f}%  ({mins:.1f} min)  "
                  f"-> grid row {li + 2}, col {ej + 2}", flush=True)

            if state is not None and acc > best["acc"]:
                best = {"acc": acc, "lr": lr, "epochs": epochs, "state": state}
                path = save_best(state, classes, lr, epochs, acc)   # save immediately
                print(f"  >> NEW BEST -> {path}", flush=True)

            elapsed = (time.time() - grid_t0) / 3600
            print(f"  >> elapsed this session: {elapsed:.2f} h\n", flush=True)

    if not started:
        print(f"!! resume point (lr {START_LR:g}, epochs {START_EPOCHS}) is not in the "
              f"grid - nothing ran", flush=True)
        return

    print("=" * 60, flush=True)
    if best["lr"] is None:
        print("no successful cell; nothing saved", flush=True)
    elif best["state"] is not None:
        print(f"BEST: lr {best['lr']:g}, epochs {best['epochs']}, "
              f"val_acc {best['acc'] * 100:.2f}%", flush=True)
        print(f"  weights -> {OUTPUT_DIR / 'ast_wav_best.pt'}", flush=True)
    else:
        # winner came from a previous session -> retrain it once so weights exist
        print(f"BEST is a cell from an earlier session: lr {best['lr']:g}, "
              f"epochs {best['epochs']} (val_acc {best['acc']:.4f}).", flush=True)
        print("retraining it once to produce its weights...", flush=True)
        acc, state = train_one_config(train_loader, val_loader,
                                      best["lr"], best["epochs"], num_classes)
        if state is not None:
            path = save_best(state, classes, best["lr"], best["epochs"], acc)
            print(f"  retrained val_acc {acc * 100:.2f}%  -> {path}", flush=True)
    print("=" * 60, flush=True)
    print("\nTHE TASK HAS BEEN COMPLETED.", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
##MFCC + KNN

In [ ]:
#This script sweeps k for K-Nearest Neighbors on the MFCC features. Reads the
#X/y arrays from DATA_DIR, writes the validation-accuracy sweep to
#OUTPUT_DIR/results.csv and the best model to OUTPUT_DIR/knn_mfcc_best.joblib

import json
import sys
import time
from pathlib import Path

import numpy as np

# ======================= EDIT THESE =======================
DATA_DIR = OUTPUTS / "mfcc_dataset"   # folder holding X_<split>.npy / y_<split>.npy (the MFCC cell's OUTPUT_DIR)
OUTPUT_DIR = OUTPUTS / "knn_mfcc"     # where the results CSV and best model will be written

NEIGHBORS = list(range(1, 15))

WEIGHTS = "distance"
METRIC = "minkowski"
P_NORM = 2
ALGORITHM = "auto"
N_JOBS = -1
# ==========================================================

RESULTS_CSV_NAME = "results.csv"


def find_data_root(base):
    """Locate the folder holding X_train.npy / X_validation.npy."""
    base = Path(base)
    if not base.exists():
        return None
    for d in [base] + sorted(p for p in base.iterdir() if p.is_dir()):
        if (d / "X_train.npy").is_file() and (d / "X_validation.npy").is_file():
            return d
    return None


def load_split(root, split):
    X = np.load(root / f"X_{split}.npy").astype(np.float32)
    y = np.load(root / f"y_{split}.npy")
    return X, y


def save_results_csv(results):
    """Rewrite the results CSV: one row per k, validation accuracy beside it."""
    lines = ["Neighbors,Accuracy"]
    for r in results:
        lines.append(f"{r['k']},{r['accuracy']:.4f}")
    (OUTPUT_DIR / RESULTS_CSV_NAME).write_text("\n".join(lines) + "\n")


def main():
    try:
        from sklearn.neighbors import KNeighborsClassifier
        import joblib
    except ImportError:
        sys.exit("this script needs scikit-learn and joblib:  pip install scikit-learn joblib")

    print("K-Nearest Neighbors + MFCC (mean+std aggregated)", flush=True)
    print(f"weights {WEIGHTS} | metric {METRIC} (p={P_NORM}) | algorithm {ALGORITHM}",
          flush=True)
    print(f"neighbor sweep: k = {NEIGHBORS[0]}..{NEIGHBORS[-1]}\n", flush=True)

    root = find_data_root(DATA_DIR)
    if root is None:
        raise SystemExit(f"could not find X_train.npy / X_validation.npy under {DATA_DIR}")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"results -> {OUTPUT_DIR / RESULTS_CSV_NAME}\n", flush=True)

    Xtr, ytr = load_split(root, "train")
    Xva, yva = load_split(root, "validation")
    classes = json.loads((root / "classes.json").read_text()) \
        if (root / "classes.json").is_file() else None

    print(f"train {Xtr.shape}  validation {Xva.shape}", flush=True)
    print(f"classes: {classes}", flush=True)
    print(f"feature check: train mean {Xtr.mean():+.4f}, std {Xtr.std():.4f} "
          f"(already standardized -> not re-scaled here)\n", flush=True)

    best = {"acc": -1.0, "k": None}
    results = []
    t_all = time.time()

    for i, k in enumerate(NEIGHBORS):
        t0 = time.time()
        model = KNeighborsClassifier(
            n_neighbors=k, weights=WEIGHTS, metric=METRIC, p=P_NORM,
            algorithm=ALGORITHM, n_jobs=N_JOBS)
        model.fit(Xtr, ytr)
        acc = float(model.score(Xva, yva))
        secs = time.time() - t0

        results.append({"k": k, "accuracy": acc, "seconds": round(secs, 2)})
        save_results_csv(results)
        parity = "even" if k % 2 == 0 else "odd"
        print(f"  k {k:>3} ({parity:<4}) -> val accuracy {acc * 100:6.2f}%   "
              f"({secs:.1f}s)", flush=True)

        if acc > best["acc"]:
            best = {"acc": acc, "k": k}
            joblib.dump(model, OUTPUT_DIR / "knn_mfcc_best.joblib")
            print(f"    >> NEW BEST (k={k})", flush=True)

    print("\n" + "=" * 52, flush=True)
    print(f"{'k':>4}{'val acc':>11}{'parity':>9}", flush=True)
    for r in results:
        mark = "  <- best" if r["k"] == best["k"] else ""
        print(f"{r['k']:>4}{r['accuracy'] * 100:>10.2f}%"
              f"{('even' if r['k'] % 2 == 0 else 'odd'):>9}{mark}", flush=True)
    print("=" * 52, flush=True)

    odd = [r["accuracy"] for r in results if r["k"] % 2 == 1]
    even = [r["accuracy"] for r in results if r["k"] % 2 == 0]
    if odd and even:
        print(f"\nmean accuracy  odd k {np.mean(odd) * 100:.2f}%  |  "
              f"even k {np.mean(even) * 100:.2f}%", flush=True)
        if np.mean(odd) > np.mean(even):
            print("  odd k scoring higher is consistent with tie-breaking losses at even k;", flush=True)
            print("  WEIGHTS='distance' would remove ties if you want to test that.", flush=True)

    if best["k"] == NEIGHBORS[-1]:
        print(f"\nnote: the best k is the largest tested ({NEIGHBORS[-1]}) - the curve may", flush=True)
        print("still be rising, so extending the sweep could find a better value.", flush=True)
    if best["k"] == 1:
        print("\nnote: k=1 winning usually means the classes are tightly clustered, but it", flush=True)
        print("is also the most overfit-prone setting - check it holds up on test.", flush=True)

    print(f"\nBEST: k {best['k']}, val accuracy {best['acc'] * 100:.2f}%", flush=True)
    print(f"  model -> {OUTPUT_DIR / 'knn_mfcc_best.joblib'}", flush=True)
    print(f"  sweep completed in {time.time() - t_all:.0f}s", flush=True)

    (OUTPUT_DIR / "sweep_results.json").write_text(json.dumps({
        "model": "knn", "representation": "mfcc",
        "weights": WEIGHTS, "metric": METRIC, "p": P_NORM, "algorithm": ALGORITHM,
        "classes": classes, "n_features": int(Xtr.shape[1]),
        "n_train": int(Xtr.shape[0]), "n_validation": int(Xva.shape[0]),
        "best": {"k": best["k"], "val_accuracy": best["acc"]},
        "results": results,
    }, indent=2))
    print(f"  sweep log -> {OUTPUT_DIR / 'sweep_results.json'}", flush=True)
    print("\nTHE TASK HAS BEEN COMPLETED.", flush=True)


if __name__ == "__main__":
    main()


##MFCC + RANDOM FOREST

In [ ]:
#This script sweeps a Random Forest grid (n_estimators x max_depth) on the MFCC
#features. Reads the X/y arrays from DATA_DIR, writes the validation-accuracy
#grid to OUTPUT_DIR/results_grid.csv and the best model to
#OUTPUT_DIR/rf_mfcc_best.joblib

import json
import sys
import time
from pathlib import Path

import numpy as np

# ======================= EDIT THESE =======================
DATA_DIR = OUTPUTS / "mfcc_dataset"   # folder holding X_<split>.npy / y_<split>.npy (the MFCC cell's OUTPUT_DIR)
OUTPUT_DIR = OUTPUTS / "rf_mfcc"      # where the results grid CSV and best model will be written

GRID_ESTIMATORS = [1, 2, 3, 4, 5, 6, 7]
GRID_MAX_DEPTHS = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]

CRITERION = "gini"
MAX_FEATURES = "sqrt"
RANDOM_STATE = 42
N_JOBS = -1
# ==========================================================

RESULTS_CSV_NAME = "results_grid.csv"


def find_data_root(base):
    """Locate the folder holding X_train.npy / X_validation.npy."""
    base = Path(base)
    if not base.exists():
        return None
    for d in [base] + sorted(p for p in base.iterdir() if p.is_dir()):
        if (d / "X_train.npy").is_file() and (d / "X_validation.npy").is_file():
            return d
    return None


def load_split(root, split):
    X = np.load(root / f"X_{split}.npy").astype(np.float32)
    y = np.load(root / f"y_{split}.npy")
    return X, y


def save_grid_csv(grid):
    """Rewrite the grid CSV: one row per n_estimators, one column per max_depth."""
    lines = ["estimators/max_depth," + ",".join(str(d) for d in GRID_MAX_DEPTHS)]
    for n_est, row in zip(GRID_ESTIMATORS, grid):
        cells = ",".join("" if np.isnan(v) else f"{v:.4f}" for v in row)
        lines.append(f"{n_est},{cells}")
    (OUTPUT_DIR / RESULTS_CSV_NAME).write_text("\n".join(lines) + "\n")


def main():
    try:
        from sklearn.ensemble import RandomForestClassifier
        import joblib
    except ImportError:
        sys.exit("this script needs scikit-learn and joblib:  pip install scikit-learn joblib")

    print("Random Forest + MFCC (mean+std aggregated)", flush=True)
    print(f"criterion {CRITERION} | max_features {MAX_FEATURES} | "
          f"random_state {RANDOM_STATE}", flush=True)
    print(f"grid: n_estimators {GRID_ESTIMATORS} x max_depth {GRID_MAX_DEPTHS} = "
          f"{len(GRID_ESTIMATORS) * len(GRID_MAX_DEPTHS)} cells\n", flush=True)

    root = find_data_root(DATA_DIR)
    if root is None:
        raise SystemExit(f"could not find X_train.npy / X_validation.npy under {DATA_DIR}")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"results grid -> {OUTPUT_DIR / RESULTS_CSV_NAME}\n", flush=True)

    Xtr, ytr = load_split(root, "train")
    Xva, yva = load_split(root, "validation")
    classes = json.loads((root / "classes.json").read_text()) \
        if (root / "classes.json").is_file() else None

    print(f"train {Xtr.shape}  validation {Xva.shape}", flush=True)
    print(f"classes: {classes}\n", flush=True)

    probe = RandomForestClassifier(n_estimators=10, max_depth=None,
                                   random_state=RANDOM_STATE, n_jobs=N_JOBS).fit(Xtr, ytr)
    depths = [t.get_depth() for t in probe.estimators_]
    sat = int(np.max(depths))
    print(f"unconstrained tree depth on this data: min {min(depths)}, "
          f"mean {np.mean(depths):.1f}, max {sat}", flush=True)
    print(f"  -> max_depth >= {sat} is equivalent to unlimited; those columns will "
          f"repeat\n", flush=True)
    del probe

    best = {"acc": -1.0, "n_estimators": None, "max_depth": None}
    results = []
    grid = np.full((len(GRID_ESTIMATORS), len(GRID_MAX_DEPTHS)), np.nan)
    grid_t0 = time.time()

    for i, n_est in enumerate(GRID_ESTIMATORS):
        for j, max_depth in enumerate(GRID_MAX_DEPTHS):
            t0 = time.time()
            model = RandomForestClassifier(
                n_estimators=n_est, max_depth=max_depth, criterion=CRITERION,
                max_features=MAX_FEATURES, random_state=RANDOM_STATE, n_jobs=N_JOBS)
            model.fit(Xtr, ytr)
            acc = float(model.score(Xva, yva))
            mean_depth = float(np.mean([t.get_depth() for t in model.estimators_]))
            secs = time.time() - t0

            grid[i, j] = acc
            save_grid_csv(grid)
            note = "  (depth saturated)" if mean_depth < max_depth - 0.5 else ""
            print(f"  n_est {n_est:>2}  max_depth {max_depth:>4} -> acc {acc * 100:6.2f}%   "
                  f"realised depth {mean_depth:5.1f}{note}   ({secs:.1f}s)", flush=True)

            results.append({"n_estimators": n_est, "max_depth": max_depth,
                            "accuracy": acc, "mean_tree_depth": round(mean_depth, 2),
                            "seconds": round(secs, 2)})

            if acc > best["acc"]:
                best = {"acc": acc, "n_estimators": n_est, "max_depth": max_depth}
                joblib.dump(model, OUTPUT_DIR / "rf_mfcc_best.joblib")
                np.save(OUTPUT_DIR / "rf_mfcc_best_importances.npy",
                        model.feature_importances_)
                print(f"    >> NEW BEST (n_est {n_est}, max_depth {max_depth})", flush=True)
        print("", flush=True)

    print("=" * 78, flush=True)
    print("validation accuracy grid (rows = n_estimators, columns = max_depth)", flush=True)
    print(f"  {'n_est':<7}" + "".join(f"{d:>7}" for d in GRID_MAX_DEPTHS), flush=True)
    for i, n_est in enumerate(GRID_ESTIMATORS):
        row = "".join(f"{grid[i, j] * 100:>7.2f}" for j in range(len(GRID_MAX_DEPTHS)))
        flat = "  (flat)" if np.ptp(grid[i]) < 1e-12 else ""
        print(f"  {n_est:<7}{row}{flat}", flush=True)
    print("=" * 78, flush=True)

    same_as_last = [j for j in range(1, len(GRID_MAX_DEPTHS))
                    if np.allclose(grid[:, j], grid[:, j - 1])]
    if same_as_last:
        first = GRID_MAX_DEPTHS[same_as_last[0]]
        print(f"\ncolumns from max_depth {first} onward duplicate the previous column -", flush=True)
        print(f"trees had already stopped growing, so the depth cap was not binding.", flush=True)

    print(f"\nbest accuracy per n_estimators:", flush=True)
    for n_est, a in zip(GRID_ESTIMATORS, grid.max(axis=1)):
        print(f"  n_est {n_est:<3} {a * 100:6.2f}%", flush=True)

    print(f"\nBEST: n_estimators {best['n_estimators']}, max_depth {best['max_depth']}, "
          f"val accuracy {best['acc'] * 100:.2f}%", flush=True)
    print(f"  model -> {OUTPUT_DIR / 'rf_mfcc_best.joblib'}", flush=True)
    print(f"  grid completed in {(time.time() - grid_t0) / 60:.1f} min", flush=True)

    (OUTPUT_DIR / "grid_results.json").write_text(json.dumps({
        "model": "random_forest", "representation": "mfcc",
        "criterion": CRITERION, "max_features": MAX_FEATURES,
        "random_state": RANDOM_STATE, "classes": classes,
        "n_features": int(Xtr.shape[1]),
        "n_train": int(Xtr.shape[0]), "n_validation": int(Xva.shape[0]),
        "unconstrained_depth": {"min": int(min(depths)), "max": sat,
                                "mean": round(float(np.mean(depths)), 2)},
        "n_estimators_values": GRID_ESTIMATORS,
        "max_depth_values": GRID_MAX_DEPTHS,
        "best": {"n_estimators": best["n_estimators"], "max_depth": best["max_depth"],
                 "val_accuracy": best["acc"]},
        "results": results,
    }, indent=2))
    print(f"  grid log -> {OUTPUT_DIR / 'grid_results.json'}", flush=True)
    print("\nTHE TASK HAS BEEN COMPLETED.", flush=True)


if __name__ == "__main__":
    main()


##MFCC + LOGISTIC REGRESSION

In [ ]:
#This script sweeps a Logistic Regression grid (C x max_iter) on the MFCC
#features. Reads the X/y arrays from DATA_DIR, writes the validation-accuracy
#grid to OUTPUT_DIR/results_grid.csv and the best model to
#OUTPUT_DIR/logreg_mfcc_best.joblib

import json
import sys
import time
import warnings
from pathlib import Path

import numpy as np

# ======================= EDIT THESE =======================
DATA_DIR = OUTPUTS / "mfcc_dataset"   # folder holding X_<split>.npy / y_<split>.npy (the MFCC cell's OUTPUT_DIR)
OUTPUT_DIR = OUTPUTS / "logreg_mfcc"  # where the results grid CSV and best model will be written

GRID_CS = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
GRID_ITERS = [50, 100, 150, 200, 250, 300]

SOLVER = "lbfgs"
PENALTY = "l2"
RANDOM_STATE = 42
N_JOBS = -1
# ==========================================================

RESULTS_CSV_NAME = "results_grid.csv"


def find_data_root(base):
    """Locate the folder holding X_train.npy / X_validation.npy."""
    base = Path(base)
    if not base.exists():
        return None
    for d in [base] + sorted(p for p in base.iterdir() if p.is_dir()):
        if (d / "X_train.npy").is_file() and (d / "X_validation.npy").is_file():
            return d
    return None


def load_split(root, split):
    X = np.load(root / f"X_{split}.npy").astype(np.float64)
    y = np.load(root / f"y_{split}.npy")
    return X, y


def save_grid_csv(grid):
    """Rewrite the grid CSV: one row per C, one column per max_iter."""
    lines = ["Iterations/C," + ",".join(str(it) for it in GRID_ITERS)]
    for C, row in zip(GRID_CS, grid):
        cells = ",".join("" if np.isnan(v) else f"{v:.4f}" for v in row)
        lines.append(f"{C},{cells}")
    (OUTPUT_DIR / RESULTS_CSV_NAME).write_text("\n".join(lines) + "\n")


def main():
    try:
        from sklearn.linear_model import LogisticRegression
        from sklearn.exceptions import ConvergenceWarning
        import joblib
    except ImportError:
        sys.exit("this script needs scikit-learn and joblib:  pip install scikit-learn joblib")

    print("Logistic Regression + MFCC (mean+std aggregated)", flush=True)
    print(f"solver {SOLVER} | penalty {PENALTY} | multinomial", flush=True)
    print(f"grid: C {GRID_CS} x max_iter {GRID_ITERS} = "
          f"{len(GRID_CS) * len(GRID_ITERS)} cells\n", flush=True)

    root = find_data_root(DATA_DIR)
    if root is None:
        raise SystemExit(f"could not find X_train.npy / X_validation.npy under {DATA_DIR}")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"results grid -> {OUTPUT_DIR / RESULTS_CSV_NAME}\n", flush=True)

    Xtr, ytr = load_split(root, "train")
    Xva, yva = load_split(root, "validation")
    classes = json.loads((root / "classes.json").read_text()) \
        if (root / "classes.json").is_file() else None

    print(f"train {Xtr.shape}  validation {Xva.shape}", flush=True)
    print(f"classes: {classes}", flush=True)
    print(f"feature check: train mean {Xtr.mean():+.4f}, std {Xtr.std():.4f} "
          f"(already standardized -> not re-scaled here)\n", flush=True)

    best = {"acc": -1.0, "C": None, "iters": None}
    results = []
    grid = np.full((len(GRID_CS), len(GRID_ITERS)), np.nan)
    grid_t0 = time.time()

    for i, C in enumerate(GRID_CS):
        for j, max_iter in enumerate(GRID_ITERS):
            t0 = time.time()
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always", ConvergenceWarning)
                model = LogisticRegression(
                    C=C, max_iter=max_iter, solver=SOLVER, penalty=PENALTY,
                    random_state=RANDOM_STATE, n_jobs=N_JOBS)
                model.fit(Xtr, ytr)
                hit_warning = any(issubclass(w.category, ConvergenceWarning)
                                  for w in caught)

            acc = float(model.score(Xva, yva))
            n_iter = int(np.max(model.n_iter_))
            converged = (n_iter < max_iter) and not hit_warning
            secs = time.time() - t0

            grid[i, j] = acc
            save_grid_csv(grid)
            flag = "converged" if converged else "HIT CAP"
            print(f"  C {C:<8g} max_iter {max_iter:<4} -> acc {acc * 100:6.2f}%   "
                  f"n_iter {n_iter:>4}/{max_iter:<4} {flag:<10} ({secs:.1f}s)", flush=True)

            results.append({"C": C, "max_iter": max_iter, "accuracy": acc,
                            "n_iter": n_iter, "converged": converged,
                            "seconds": round(secs, 2)})

            if acc > best["acc"]:
                best = {"acc": acc, "C": C, "iters": max_iter}
                joblib.dump(model, OUTPUT_DIR / "logreg_mfcc_best.joblib")
                np.save(OUTPUT_DIR / "logreg_mfcc_best_coef.npy", model.coef_)
                np.save(OUTPUT_DIR / "logreg_mfcc_best_intercept.npy", model.intercept_)
                print(f"    >> NEW BEST (C {C:g}, max_iter {max_iter})", flush=True)
        print("", flush=True)

    print("=" * 68, flush=True)
    print("validation accuracy grid (rows = C, columns = max_iter)", flush=True)
    print(f"  {'C':<10}" + "".join(f"{it:>9}" for it in GRID_ITERS), flush=True)
    for i, C in enumerate(GRID_CS):
        row = "".join(f"{grid[i, j] * 100:>9.2f}" for j in range(len(GRID_ITERS)))
        flat = "  (flat)" if np.ptp(grid[i]) < 1e-12 else ""
        print(f"  {C:<10g}{row}{flat}", flush=True)
    print("=" * 68, flush=True)

    n_flat = sum(1 for i in range(len(GRID_CS)) if np.ptp(grid[i]) < 1e-12)
    print(f"\n{n_flat}/{len(GRID_CS)} C-rows are flat across max_iter - in those the "
          f"solver converged\nbefore the smallest cap, so only C affected the result.",
          flush=True)
    best_c_curve = grid.max(axis=1)
    print(f"\nbest accuracy per C:", flush=True)
    for C, a in zip(GRID_CS, best_c_curve):
        print(f"  C {C:<10g} {a * 100:6.2f}%", flush=True)

    print(f"\nBEST: C {best['C']:g}, max_iter {best['iters']}, "
          f"val accuracy {best['acc'] * 100:.2f}%", flush=True)
    print(f"  model -> {OUTPUT_DIR / 'logreg_mfcc_best.joblib'}", flush=True)
    print(f"  grid completed in {(time.time() - grid_t0) / 60:.1f} min", flush=True)

    (OUTPUT_DIR / "grid_results.json").write_text(json.dumps({
        "model": "logistic_regression", "representation": "mfcc",
        "solver": SOLVER, "penalty": PENALTY, "random_state": RANDOM_STATE,
        "classes": classes, "n_features": int(Xtr.shape[1]),
        "n_train": int(Xtr.shape[0]), "n_validation": int(Xva.shape[0]),
        "C_values": GRID_CS, "max_iter_values": GRID_ITERS,
        "best": {"C": best["C"], "max_iter": best["iters"], "val_accuracy": best["acc"]},
        "results": results,
    }, indent=2))
    print(f"  grid log -> {OUTPUT_DIR / 'grid_results.json'}", flush=True)
    print("\nTHE TASK HAS BEEN COMPLETED.", flush=True)


if __name__ == "__main__":
    main()


##MCG + RESNET50 (testing)

In [ ]:
#This script evaluates the best ResNet50 + MCG checkpoint on the test split.
#Reads DATA_DIR/<split>/<class>/*.npy and the checkpoint from CKPT_PATH, and
#prints the test accuracy, classification report and confusion matrix to the
#terminal. Nothing is written to disk.

import time
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as VT
from transformers import AutoModelForImageClassification

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **k):
        return x


# ======================= EDIT THESE =======================
DATA_DIR = OUTPUTS / "mcg_dataset"                        # folder holding <split>/<class>/*.npy (the MCG cell's OUTPUT_DIR)
CKPT_PATH = OUTPUTS / "resnet50_mcg" / "resnet50_mcg_best.pt" # best checkpoint saved by the training cell

MODEL_NAME = "microsoft/resnet-50"
TEST_SPLIT = "test"

IMG_SIZE = 224
BATCH = 64
NUM_WORKERS = 2      # set to 0 if the DataLoader gives trouble in the notebook

IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
# ==========================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)


def find_split_root(base):
    base = Path(base)
    if not base.exists():
        return None
    for d in [base] + sorted(p for p in base.iterdir() if p.is_dir()):
        if (d / TEST_SPLIT).is_dir() and (d / "train").is_dir() \
                and next((d / TEST_SPLIT).rglob("*.npy"), None) is not None:
            return d
    return None


class McgDataset(Dataset):
    """(3, 128, F) mel/CQT/gamma .npy -> 224x224, normalised as the model was trained."""

    def __init__(self, split_dir, class_to_idx, mean, std):
        self.samples = []
        for cls, idx in class_to_idx.items():
            cdir = split_dir / cls
            if not cdir.is_dir():
                print(f"  !! {split_dir.name}: no folder for class '{cls}'", flush=True)
                continue
            for npy in sorted(cdir.glob("*.npy")):
                self.samples.append((npy, idx))
        self.resize = VT.Resize((IMG_SIZE, IMG_SIZE), antialias=True)
        self.normalize = VT.Normalize(mean, std)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        arr = np.load(path).astype(np.float32)
        if arr.ndim == 3 and arr.shape[2] == 3 and arr.shape[0] != 3:
            arr = np.transpose(arr, (2, 0, 1))
        arr = np.ascontiguousarray(arr)
        t = torch.from_numpy(arr)
        if t.ndim == 2:
            t = t.unsqueeze(0).repeat(3, 1, 1)
        elif t.shape[0] == 1:
            t = t.repeat(3, 1, 1)
        return self.normalize(self.resize(t)), label


def confusion_matrix(preds, labels, n_classes):
    cm = np.zeros((n_classes, n_classes), dtype=np.int64)
    for t, p in zip(labels, preds):
        cm[t, p] += 1
    return cm


def report_rows(cm, classes):
    support = cm.sum(axis=1).astype(float)
    pred_tot = cm.sum(axis=0).astype(float)
    tp = np.diag(cm).astype(float)

    precision = np.divide(tp, pred_tot, out=np.zeros_like(tp), where=pred_tot > 0)
    recall = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
    denom = precision + recall
    f1 = np.divide(2 * precision * recall, denom, out=np.zeros_like(tp), where=denom > 0)

    total = support.sum()
    accuracy = tp.sum() / total if total > 0 else 0.0
    w = support / total if total > 0 else np.zeros_like(support)

    per_class = [{"name": c, "precision": precision[i], "recall": recall[i],
                  "f1": f1[i], "support": int(support[i])}
                 for i, c in enumerate(classes)]
    summary = {
        "accuracy": accuracy,
        "macro": {"precision": precision.mean(), "recall": recall.mean(),
                  "f1": f1.mean(), "support": int(total)},
        "weighted": {"precision": float((precision * w).sum()),
                     "recall": float((recall * w).sum()),
                     "f1": float((f1 * w).sum()), "support": int(total)},
    }
    return per_class, summary


def format_report(per_class, summary):
    name_w = max(14, max(len(r["name"]) for r in per_class) + 2)
    out = [f"{'':<{name_w}}{'precision':>10}{'recall':>10}{'f1-score':>10}{'support':>10}", ""]
    for r in per_class:
        out.append(f"{r['name']:<{name_w}}{r['precision']:>10.4f}{r['recall']:>10.4f}"
                   f"{r['f1']:>10.4f}{r['support']:>10d}")
    out.append("")
    s = summary
    out.append(f"{'accuracy':<{name_w}}{'':>10}{'':>10}{s['accuracy']:>10.4f}"
               f"{s['macro']['support']:>10d}")
    for key, label in (("macro", "macro avg"), ("weighted", "weighted avg")):
        m = s[key]
        out.append(f"{label:<{name_w}}{m['precision']:>10.4f}{m['recall']:>10.4f}"
                   f"{m['f1']:>10.4f}{m['support']:>10d}")
    return "\n".join(out)


def format_confusion(cm, classes):
    name_w = max(12, max(len(c) for c in classes) + 2)
    col_w = max(8, max(len(c) for c in classes) + 2)
    out = ["confusion matrix (rows = true class, columns = predicted):",
           f"{'':<{name_w}}" + "".join(f"{c:>{col_w}}" for c in classes)]
    for i, c in enumerate(classes):
        out.append(f"{c:<{name_w}}" + "".join(f"{cm[i, j]:>{col_w}d}"
                                              for j in range(len(classes))))
    return "\n".join(out)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    preds, labels = [], []
    for x, y in tqdm(loader, desc="test"):
        x = x.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
            logits = model(pixel_values=x).logits
        preds.append(logits.argmax(1).cpu())
        labels.append(y)
    return torch.cat(preds).numpy(), torch.cat(labels).numpy()


def main():
    print(f"Device: {DEVICE}", flush=True)
    if not CKPT_PATH.is_file():
        raise SystemExit(f"checkpoint not found: {CKPT_PATH}")

    ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
    classes = ckpt["classes"]
    state = ckpt["model_state"]
    hp = ckpt.get("hyperparameters", {})
    norm = ckpt.get("normalization", {})
    mean = norm.get("mean", IMAGENET_MEAN)
    std = norm.get("std", IMAGENET_STD)

    print(f"\nloaded {CKPT_PATH.name}", flush=True)
    print(f"  model {ckpt.get('model')} ({ckpt.get('model_name')})", flush=True)
    print(f"  representation {ckpt.get('representation')} "
          f"{ckpt.get('channels', '')}", flush=True)
    print(f"  classes {classes}", flush=True)
    print(f"  config: lr {hp.get('lr')}, epochs {hp.get('epochs')}, "
          f"batch {hp.get('batch')}, {hp.get('optimizer')}", flush=True)
    print(f"  normalisation mean {mean} std {std}  (taken from the checkpoint)", flush=True)
    if "val_accuracy" in ckpt:
        print(f"  validation accuracy at selection: "
              f"{ckpt['val_accuracy'] * 100:.2f}%", flush=True)

    root = find_split_root(DATA_DIR)
    if root is None:
        raise SystemExit(f"could not find a '{TEST_SPLIT}' split under {DATA_DIR}")
    class_to_idx = {c: i for i, c in enumerate(classes)}
    ds = McgDataset(root / TEST_SPLIT, class_to_idx, mean, std)
    if len(ds) == 0:
        raise SystemExit(f"no .npy files found in {root / TEST_SPLIT}")
    loader = DataLoader(ds, batch_size=BATCH, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
    probe, _ = ds[0]
    print(f"\ntest clips {len(ds)}  |  input shape {tuple(probe.shape)}\n", flush=True)

    model = AutoModelForImageClassification.from_pretrained(
        MODEL_NAME, num_labels=len(classes), ignore_mismatched_sizes=True)
    model.load_state_dict(state)
    model = model.to(DEVICE)

    preds, labels = predict(model, loader)
    cm = confusion_matrix(preds, labels, len(classes))
    per_class, summary = report_rows(cm, classes)

    header = "Classification report - ResNet50 + 3-channel (Mel, CQT, Gamma) - test set"
    print("\n" + "=" * len(header), flush=True)
    print(header, flush=True)
    print("=" * len(header), flush=True)
    print(format_report(per_class, summary), flush=True)
    print("=" * len(header), flush=True)

    print("\n" + format_confusion(cm, classes), flush=True)

    if "val_accuracy" in ckpt:
        gap = ckpt["val_accuracy"] - summary["accuracy"]
        print(f"\nvalidation {ckpt['val_accuracy'] * 100:.2f}%  ->  "
              f"test {summary['accuracy'] * 100:.2f}%   (gap {gap * 100:+.2f} pts)",
              flush=True)
        print("  a modest drop is normal: the configuration was chosen on validation.",
              flush=True)

    w = summary["weighted"]
    print(f"\nTable 3 row:  accuracy {summary['accuracy'] * 100:.2f}  "
          f"precision {w['precision'] * 100:.2f}  recall {w['recall'] * 100:.2f}  "
          f"F1 {w['f1'] * 100:.2f}", flush=True)

    print(f"\nTEST ACCURACY: {summary['accuracy'] * 100:.2f}%", flush=True)


if __name__ == "__main__":
    main()


##MCG + VGGNet19 (testing)

In [ ]:
#This script evaluates the best VGG19 + MCG checkpoint on the test split.
#Reads DATA_DIR/<split>/<class>/*.npy and the checkpoint from CKPT_PATH, and
#prints the test accuracy, classification report and confusion matrix to the
#terminal. Nothing is written to disk.

import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as VT
from torchvision.models import vgg19

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **k):
        return x


# ======================= EDIT THESE =======================
DATA_DIR = OUTPUTS / "mcg_dataset"                        # folder holding <split>/<class>/*.npy (the MCG cell's OUTPUT_DIR)
CKPT_PATH = OUTPUTS / "vgg19_mcg" / "vgg19_mcg_best.pt"       # best checkpoint saved by the training cell

TEST_SPLIT = "test"

IMG_SIZE = 224
BATCH = 64
NUM_WORKERS = 2      # set to 0 if the DataLoader gives trouble in the notebook

IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
# ==========================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)


def find_split_root(base):
    base = Path(base)
    if not base.exists():
        return None
    for d in [base] + sorted(p for p in base.iterdir() if p.is_dir()):
        if (d / TEST_SPLIT).is_dir() and (d / "train").is_dir() \
                and next((d / TEST_SPLIT).rglob("*.npy"), None) is not None:
            return d
    return None


class McgDataset(Dataset):
    """(3, 128, F) mel/CQT/gamma .npy -> 224x224, normalised as the model was trained."""

    def __init__(self, split_dir, class_to_idx, mean, std):
        self.samples = []
        for cls, idx in class_to_idx.items():
            cdir = split_dir / cls
            if not cdir.is_dir():
                print(f"  !! {split_dir.name}: no folder for class '{cls}'", flush=True)
                continue
            for npy in sorted(cdir.glob("*.npy")):
                self.samples.append((npy, idx))
        self.resize = VT.Resize((IMG_SIZE, IMG_SIZE), antialias=True)
        self.normalize = VT.Normalize(mean, std)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        arr = np.load(path).astype(np.float32)
        if arr.ndim == 3 and arr.shape[2] == 3 and arr.shape[0] != 3:
            arr = np.transpose(arr, (2, 0, 1))
        arr = np.ascontiguousarray(arr)
        t = torch.from_numpy(arr)
        if t.ndim == 2:
            t = t.unsqueeze(0).repeat(3, 1, 1)
        elif t.shape[0] == 1:
            t = t.repeat(3, 1, 1)
        return self.normalize(self.resize(t)), label


def confusion_matrix(preds, labels, n_classes):
    cm = np.zeros((n_classes, n_classes), dtype=np.int64)
    for t, p in zip(labels, preds):
        cm[t, p] += 1
    return cm


def report_rows(cm, classes):
    support = cm.sum(axis=1).astype(float)
    pred_tot = cm.sum(axis=0).astype(float)
    tp = np.diag(cm).astype(float)

    precision = np.divide(tp, pred_tot, out=np.zeros_like(tp), where=pred_tot > 0)
    recall = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
    denom = precision + recall
    f1 = np.divide(2 * precision * recall, denom, out=np.zeros_like(tp), where=denom > 0)

    total = support.sum()
    accuracy = tp.sum() / total if total > 0 else 0.0
    w = support / total if total > 0 else np.zeros_like(support)

    per_class = [{"name": c, "precision": precision[i], "recall": recall[i],
                  "f1": f1[i], "support": int(support[i])}
                 for i, c in enumerate(classes)]
    summary = {
        "accuracy": accuracy,
        "macro": {"precision": precision.mean(), "recall": recall.mean(),
                  "f1": f1.mean(), "support": int(total)},
        "weighted": {"precision": float((precision * w).sum()),
                     "recall": float((recall * w).sum()),
                     "f1": float((f1 * w).sum()), "support": int(total)},
    }
    return per_class, summary


def format_report(per_class, summary):
    name_w = max(14, max(len(r["name"]) for r in per_class) + 2)
    out = [f"{'':<{name_w}}{'precision':>10}{'recall':>10}{'f1-score':>10}{'support':>10}", ""]
    for r in per_class:
        out.append(f"{r['name']:<{name_w}}{r['precision']:>10.4f}{r['recall']:>10.4f}"
                   f"{r['f1']:>10.4f}{r['support']:>10d}")
    out.append("")
    s = summary
    out.append(f"{'accuracy':<{name_w}}{'':>10}{'':>10}{s['accuracy']:>10.4f}"
               f"{s['macro']['support']:>10d}")
    for key, label in (("macro", "macro avg"), ("weighted", "weighted avg")):
        m = s[key]
        out.append(f"{label:<{name_w}}{m['precision']:>10.4f}{m['recall']:>10.4f}"
                   f"{m['f1']:>10.4f}{m['support']:>10d}")
    return "\n".join(out)


def format_confusion(cm, classes):
    name_w = max(12, max(len(c) for c in classes) + 2)
    col_w = max(8, max(len(c) for c in classes) + 2)
    out = ["confusion matrix (rows = true class, columns = predicted):",
           f"{'':<{name_w}}" + "".join(f"{c:>{col_w}}" for c in classes)]
    for i, c in enumerate(classes):
        out.append(f"{c:<{name_w}}" + "".join(f"{cm[i, j]:>{col_w}d}"
                                              for j in range(len(classes))))
    return "\n".join(out)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    preds, labels = [], []
    for x, y in tqdm(loader, desc="test"):
        x = x.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
            logits = model(x)                    # torchvision returns logits
        preds.append(logits.argmax(1).cpu())
        labels.append(y)
    return torch.cat(preds).numpy(), torch.cat(labels).numpy()


def main():
    print(f"Device: {DEVICE}", flush=True)
    if not CKPT_PATH.is_file():
        raise SystemExit(f"checkpoint not found: {CKPT_PATH}")

    ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
    classes = ckpt["classes"]
    state = ckpt["model_state"]
    hp = ckpt.get("hyperparameters", {})
    norm = ckpt.get("normalization", {})
    mean = norm.get("mean", IMAGENET_MEAN)
    std = norm.get("std", IMAGENET_STD)

    print(f"\nloaded {CKPT_PATH.name}", flush=True)
    print(f"  model {ckpt.get('model')} ({ckpt.get('model_name')})", flush=True)
    print(f"  representation {ckpt.get('representation')} "
          f"{ckpt.get('channels', '')}", flush=True)
    print(f"  classes {classes}", flush=True)
    print(f"  config: lr {hp.get('lr')}, epochs {hp.get('epochs')}, "
          f"batch {hp.get('batch')}, {hp.get('optimizer')}", flush=True)
    print(f"  normalisation mean {mean} std {std}  (taken from the checkpoint)", flush=True)
    if "val_accuracy" in ckpt:
        print(f"  validation accuracy at selection: "
              f"{ckpt['val_accuracy'] * 100:.2f}%", flush=True)

    root = find_split_root(DATA_DIR)
    if root is None:
        raise SystemExit(f"could not find a '{TEST_SPLIT}' split under {DATA_DIR}")
    class_to_idx = {c: i for i, c in enumerate(classes)}
    ds = McgDataset(root / TEST_SPLIT, class_to_idx, mean, std)
    if len(ds) == 0:
        raise SystemExit(f"no .npy files found in {root / TEST_SPLIT}")
    loader = DataLoader(ds, batch_size=BATCH, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
    probe, _ = ds[0]
    print(f"\ntest clips {len(ds)}  |  input shape {tuple(probe.shape)}\n", flush=True)

    model = vgg19(weights=None)
    model.classifier[6] = nn.Linear(model.classifier[6].in_features, len(classes))
    model.load_state_dict(state)
    model = model.to(DEVICE)

    preds, labels = predict(model, loader)
    cm = confusion_matrix(preds, labels, len(classes))
    per_class, summary = report_rows(cm, classes)

    header = "Classification report - VGG19 + 3-channel (Mel, CQT, Gamma) - test set"
    print("\n" + "=" * len(header), flush=True)
    print(header, flush=True)
    print("=" * len(header), flush=True)
    print(format_report(per_class, summary), flush=True)
    print("=" * len(header), flush=True)

    print("\n" + format_confusion(cm, classes), flush=True)

    if "val_accuracy" in ckpt:
        gap = ckpt["val_accuracy"] - summary["accuracy"]
        print(f"\nvalidation {ckpt['val_accuracy'] * 100:.2f}%  ->  "
              f"test {summary['accuracy'] * 100:.2f}%   (gap {gap * 100:+.2f} pts)",
              flush=True)
        print("  a modest drop is normal: the configuration was chosen on validation.",
              flush=True)

    w = summary["weighted"]
    print(f"\nTable 3 row:  accuracy {summary['accuracy'] * 100:.2f}  "
          f"precision {w['precision'] * 100:.2f}  recall {w['recall'] * 100:.2f}  "
          f"F1 {w['f1'] * 100:.2f}", flush=True)

    print(f"\nTEST ACCURACY: {summary['accuracy'] * 100:.2f}%", flush=True)


if __name__ == "__main__":
    main()


##MCG + VIT (testing)

In [ ]:
#This script evaluates the best ViT + MCG checkpoint on the test split.
#Reads DATA_DIR/<split>/<class>/*.npy and the checkpoint from CKPT_PATH, and
#prints the test accuracy, classification report and confusion matrix to the
#terminal. Nothing is written to disk.

import time
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as VT
from transformers import ViTForImageClassification

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **k):
        return x


# ======================= EDIT THESE =======================
DATA_DIR = OUTPUTS / "mcg_dataset"                        # folder holding <split>/<class>/*.npy (the MCG cell's OUTPUT_DIR)
CKPT_PATH = OUTPUTS / "vit_mcg" / "vit_mcg_best.pt"           # best checkpoint saved by the training cell

MODEL_NAME = "google/vit-base-patch16-224"
TEST_SPLIT = "test"

IMG_SIZE = 224
BATCH = 64
NUM_WORKERS = 2      # set to 0 if the DataLoader gives trouble in the notebook

VIT_MEAN, VIT_STD = [0.5, 0.5, 0.5], [0.5, 0.5, 0.5]
# ==========================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)


def find_split_root(base):
    base = Path(base)
    if not base.exists():
        return None
    for d in [base] + sorted(p for p in base.iterdir() if p.is_dir()):
        if (d / TEST_SPLIT).is_dir() and (d / "train").is_dir() \
                and next((d / TEST_SPLIT).rglob("*.npy"), None) is not None:
            return d
    return None


class McgDataset(Dataset):
    """(3, 128, F) mel/CQT/gamma .npy -> 224x224, normalised as the model was trained."""

    def __init__(self, split_dir, class_to_idx, mean, std):
        self.samples = []
        for cls, idx in class_to_idx.items():
            cdir = split_dir / cls
            if not cdir.is_dir():
                print(f"  !! {split_dir.name}: no folder for class '{cls}'", flush=True)
                continue
            for npy in sorted(cdir.glob("*.npy")):
                self.samples.append((npy, idx))
        self.resize = VT.Resize((IMG_SIZE, IMG_SIZE), antialias=True)
        self.normalize = VT.Normalize(mean, std)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        arr = np.load(path).astype(np.float32)
        if arr.ndim == 3 and arr.shape[2] == 3 and arr.shape[0] != 3:
            arr = np.transpose(arr, (2, 0, 1))
        arr = np.ascontiguousarray(arr)
        t = torch.from_numpy(arr)
        if t.ndim == 2:
            t = t.unsqueeze(0).repeat(3, 1, 1)
        elif t.shape[0] == 1:
            t = t.repeat(3, 1, 1)
        return self.normalize(self.resize(t)), label


def confusion_matrix(preds, labels, n_classes):
    cm = np.zeros((n_classes, n_classes), dtype=np.int64)
    for t, p in zip(labels, preds):
        cm[t, p] += 1
    return cm


def report_rows(cm, classes):
    support = cm.sum(axis=1).astype(float)
    pred_tot = cm.sum(axis=0).astype(float)
    tp = np.diag(cm).astype(float)

    precision = np.divide(tp, pred_tot, out=np.zeros_like(tp), where=pred_tot > 0)
    recall = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
    denom = precision + recall
    f1 = np.divide(2 * precision * recall, denom, out=np.zeros_like(tp), where=denom > 0)

    total = support.sum()
    accuracy = tp.sum() / total if total > 0 else 0.0
    w = support / total if total > 0 else np.zeros_like(support)

    per_class = [{"name": c, "precision": precision[i], "recall": recall[i],
                  "f1": f1[i], "support": int(support[i])}
                 for i, c in enumerate(classes)]
    summary = {
        "accuracy": accuracy,
        "macro": {"precision": precision.mean(), "recall": recall.mean(),
                  "f1": f1.mean(), "support": int(total)},
        "weighted": {"precision": float((precision * w).sum()),
                     "recall": float((recall * w).sum()),
                     "f1": float((f1 * w).sum()), "support": int(total)},
    }
    return per_class, summary


def format_report(per_class, summary):
    name_w = max(14, max(len(r["name"]) for r in per_class) + 2)
    out = [f"{'':<{name_w}}{'precision':>10}{'recall':>10}{'f1-score':>10}{'support':>10}", ""]
    for r in per_class:
        out.append(f"{r['name']:<{name_w}}{r['precision']:>10.4f}{r['recall']:>10.4f}"
                   f"{r['f1']:>10.4f}{r['support']:>10d}")
    out.append("")
    s = summary
    out.append(f"{'accuracy':<{name_w}}{'':>10}{'':>10}{s['accuracy']:>10.4f}"
               f"{s['macro']['support']:>10d}")
    for key, label in (("macro", "macro avg"), ("weighted", "weighted avg")):
        m = s[key]
        out.append(f"{label:<{name_w}}{m['precision']:>10.4f}{m['recall']:>10.4f}"
                   f"{m['f1']:>10.4f}{m['support']:>10d}")
    return "\n".join(out)


def format_confusion(cm, classes):
    name_w = max(12, max(len(c) for c in classes) + 2)
    col_w = max(8, max(len(c) for c in classes) + 2)
    out = ["confusion matrix (rows = true class, columns = predicted):",
           f"{'':<{name_w}}" + "".join(f"{c:>{col_w}}" for c in classes)]
    for i, c in enumerate(classes):
        out.append(f"{c:<{name_w}}" + "".join(f"{cm[i, j]:>{col_w}d}"
                                              for j in range(len(classes))))
    return "\n".join(out)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    preds, labels = [], []
    for x, y in tqdm(loader, desc="test"):
        x = x.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
            logits = model(pixel_values=x).logits
        preds.append(logits.argmax(1).cpu())
        labels.append(y)
    return torch.cat(preds).numpy(), torch.cat(labels).numpy()


def main():
    print(f"Device: {DEVICE}", flush=True)
    if not CKPT_PATH.is_file():
        raise SystemExit(f"checkpoint not found: {CKPT_PATH}")

    ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
    classes = ckpt["classes"]
    state = ckpt["model_state"]
    hp = ckpt.get("hyperparameters", {})
    norm = ckpt.get("normalization", {})
    mean = norm.get("mean", VIT_MEAN)
    std = norm.get("std", VIT_STD)

    print(f"\nloaded {CKPT_PATH.name}", flush=True)
    print(f"  model {ckpt.get('model')} ({ckpt.get('model_name')})", flush=True)
    print(f"  representation {ckpt.get('representation')} "
          f"{ckpt.get('channels', '')}", flush=True)
    print(f"  classes {classes}", flush=True)
    print(f"  config: lr {hp.get('lr')}, epochs {hp.get('epochs')}, "
          f"batch {hp.get('batch')}, {hp.get('optimizer')}", flush=True)
    print(f"  normalisation mean {mean} std {std}  (taken from the checkpoint)", flush=True)
    if "val_accuracy" in ckpt:
        print(f"  validation accuracy at selection: "
              f"{ckpt['val_accuracy'] * 100:.2f}%", flush=True)

    root = find_split_root(DATA_DIR)
    if root is None:
        raise SystemExit(f"could not find a '{TEST_SPLIT}' split under {DATA_DIR}")
    class_to_idx = {c: i for i, c in enumerate(classes)}
    ds = McgDataset(root / TEST_SPLIT, class_to_idx, mean, std)
    if len(ds) == 0:
        raise SystemExit(f"no .npy files found in {root / TEST_SPLIT}")
    loader = DataLoader(ds, batch_size=BATCH, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
    probe, _ = ds[0]
    print(f"\ntest clips {len(ds)}  |  input shape {tuple(probe.shape)}\n", flush=True)

    model = ViTForImageClassification.from_pretrained(
        MODEL_NAME, num_labels=len(classes), ignore_mismatched_sizes=True)
    model.load_state_dict(state)
    model = model.to(DEVICE)

    preds, labels = predict(model, loader)
    cm = confusion_matrix(preds, labels, len(classes))
    per_class, summary = report_rows(cm, classes)

    header = "Classification report - ViT + 3-channel (Mel, CQT, Gamma) - test set"
    print("\n" + "=" * len(header), flush=True)
    print(header, flush=True)
    print("=" * len(header), flush=True)
    print(format_report(per_class, summary), flush=True)
    print("=" * len(header), flush=True)

    print("\n" + format_confusion(cm, classes), flush=True)

    if "val_accuracy" in ckpt:
        gap = ckpt["val_accuracy"] - summary["accuracy"]
        print(f"\nvalidation {ckpt['val_accuracy'] * 100:.2f}%  ->  "
              f"test {summary['accuracy'] * 100:.2f}%   (gap {gap * 100:+.2f} pts)",
              flush=True)
        print("  a modest drop is normal: the configuration was chosen on validation.",
              flush=True)

    w = summary["weighted"]
    print(f"\nTable 3 row:  accuracy {summary['accuracy'] * 100:.2f}  "
          f"precision {w['precision'] * 100:.2f}  recall {w['recall'] * 100:.2f}  "
          f"F1 {w['f1'] * 100:.2f}", flush=True)

    print(f"\nTEST ACCURACY: {summary['accuracy'] * 100:.2f}%", flush=True)


if __name__ == "__main__":
    main()


##Mel + Resnet50

In [ ]:
#This script evaluates the best ResNet50 + mel checkpoint on the test split.
#Reads DATA_DIR/<split>/<class>/*.npy and the checkpoint from CKPT_PATH, and
#prints the test accuracy, classification report and confusion matrix to the
#terminal. Nothing is written to disk.

import time
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as VT
from transformers import AutoModelForImageClassification

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **k):
        return x


# ======================= EDIT THESE =======================
DATA_DIR = OUTPUTS / "mel_dataset"                        # folder holding <split>/<class>/*.npy (the mel cell's OUTPUT_DIR)
CKPT_PATH = OUTPUTS / "resnet50_mel" / "resnet50_mel_best.pt" # best checkpoint saved by the training cell

MODEL_NAME = "microsoft/resnet-50"
TEST_SPLIT = "test"

IMG_SIZE = 224
BATCH = 64
NUM_WORKERS = 2      # set to 0 if the DataLoader gives trouble in the notebook

IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
# ==========================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)


def find_split_root(base):
    base = Path(base)
    if not base.exists():
        return None
    for d in [base] + sorted(p for p in base.iterdir() if p.is_dir()):
        if (d / TEST_SPLIT).is_dir() and (d / "train").is_dir() \
                and next((d / TEST_SPLIT).rglob("*.npy"), None) is not None:
            return d
    return None


class MelDataset(Dataset):
    """(128, F) mel .npy -> 3 channels, 224x224, normalised as the model was trained."""

    def __init__(self, split_dir, class_to_idx, mean, std):
        self.samples = []
        for cls, idx in class_to_idx.items():
            cdir = split_dir / cls
            if not cdir.is_dir():
                print(f"  !! {split_dir.name}: no folder for class '{cls}'", flush=True)
                continue
            for npy in sorted(cdir.glob("*.npy")):
                self.samples.append((npy, idx))
        self.resize = VT.Resize((IMG_SIZE, IMG_SIZE), antialias=True)
        self.normalize = VT.Normalize(mean, std)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        arr = np.load(path).astype(np.float32)
        t = torch.from_numpy(np.ascontiguousarray(arr))
        if t.ndim == 2:
            t = t.unsqueeze(0)
        if t.shape[0] == 1:
            t = t.repeat(3, 1, 1)
        return self.normalize(self.resize(t)), label


def confusion_matrix(preds, labels, n_classes):
    cm = np.zeros((n_classes, n_classes), dtype=np.int64)
    for t, p in zip(labels, preds):
        cm[t, p] += 1
    return cm


def report_rows(cm, classes):
    support = cm.sum(axis=1).astype(float)
    pred_tot = cm.sum(axis=0).astype(float)
    tp = np.diag(cm).astype(float)

    precision = np.divide(tp, pred_tot, out=np.zeros_like(tp), where=pred_tot > 0)
    recall = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
    denom = precision + recall
    f1 = np.divide(2 * precision * recall, denom, out=np.zeros_like(tp), where=denom > 0)

    total = support.sum()
    accuracy = tp.sum() / total if total > 0 else 0.0
    w = support / total if total > 0 else np.zeros_like(support)

    per_class = [{"name": c, "precision": precision[i], "recall": recall[i],
                  "f1": f1[i], "support": int(support[i])}
                 for i, c in enumerate(classes)]
    summary = {
        "accuracy": accuracy,
        "macro": {"precision": precision.mean(), "recall": recall.mean(),
                  "f1": f1.mean(), "support": int(total)},
        "weighted": {"precision": float((precision * w).sum()),
                     "recall": float((recall * w).sum()),
                     "f1": float((f1 * w).sum()), "support": int(total)},
    }
    return per_class, summary


def format_report(per_class, summary):
    name_w = max(14, max(len(r["name"]) for r in per_class) + 2)
    out = [f"{'':<{name_w}}{'precision':>10}{'recall':>10}{'f1-score':>10}{'support':>10}", ""]
    for r in per_class:
        out.append(f"{r['name']:<{name_w}}{r['precision']:>10.4f}{r['recall']:>10.4f}"
                   f"{r['f1']:>10.4f}{r['support']:>10d}")
    out.append("")
    s = summary
    out.append(f"{'accuracy':<{name_w}}{'':>10}{'':>10}{s['accuracy']:>10.4f}"
               f"{s['macro']['support']:>10d}")
    for key, label in (("macro", "macro avg"), ("weighted", "weighted avg")):
        m = s[key]
        out.append(f"{label:<{name_w}}{m['precision']:>10.4f}{m['recall']:>10.4f}"
                   f"{m['f1']:>10.4f}{m['support']:>10d}")
    return "\n".join(out)


def format_confusion(cm, classes):
    name_w = max(12, max(len(c) for c in classes) + 2)
    col_w = max(8, max(len(c) for c in classes) + 2)
    out = ["confusion matrix (rows = true class, columns = predicted):",
           f"{'':<{name_w}}" + "".join(f"{c:>{col_w}}" for c in classes)]
    for i, c in enumerate(classes):
        out.append(f"{c:<{name_w}}" + "".join(f"{cm[i, j]:>{col_w}d}"
                                              for j in range(len(classes))))
    return "\n".join(out)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    preds, labels = [], []
    for x, y in tqdm(loader, desc="test"):
        x = x.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
            logits = model(pixel_values=x).logits
        preds.append(logits.argmax(1).cpu())
        labels.append(y)
    return torch.cat(preds).numpy(), torch.cat(labels).numpy()


def main():
    print(f"Device: {DEVICE}", flush=True)
    if not CKPT_PATH.is_file():
        raise SystemExit(f"checkpoint not found: {CKPT_PATH}")

    ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
    classes = ckpt["classes"]
    state = ckpt["model_state"]
    hp = ckpt.get("hyperparameters", {})
    norm = ckpt.get("normalization", {})
    mean = norm.get("mean", IMAGENET_MEAN)
    std = norm.get("std", IMAGENET_STD)

    print(f"\nloaded {CKPT_PATH.name}", flush=True)
    print(f"  model {ckpt.get('model')} ({ckpt.get('model_name')})", flush=True)
    print(f"  representation {ckpt.get('representation')}", flush=True)
    print(f"  classes {classes}", flush=True)
    print(f"  config: lr {hp.get('lr')}, epochs {hp.get('epochs')}, "
          f"batch {hp.get('batch')}, {hp.get('optimizer')}", flush=True)
    print(f"  normalisation mean {mean} std {std}  (taken from the checkpoint)", flush=True)
    if "val_accuracy" in ckpt:
        print(f"  validation accuracy at selection: "
              f"{ckpt['val_accuracy'] * 100:.2f}%", flush=True)

    root = find_split_root(DATA_DIR)
    if root is None:
        raise SystemExit(f"could not find a '{TEST_SPLIT}' split under {DATA_DIR}")
    class_to_idx = {c: i for i, c in enumerate(classes)}
    ds = MelDataset(root / TEST_SPLIT, class_to_idx, mean, std)
    if len(ds) == 0:
        raise SystemExit(f"no .npy files found in {root / TEST_SPLIT}")
    loader = DataLoader(ds, batch_size=BATCH, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
    probe, _ = ds[0]
    print(f"\ntest clips {len(ds)}  |  input shape {tuple(probe.shape)}\n", flush=True)

    model = AutoModelForImageClassification.from_pretrained(
        MODEL_NAME, num_labels=len(classes), ignore_mismatched_sizes=True)
    model.load_state_dict(state)
    model = model.to(DEVICE)

    preds, labels = predict(model, loader)
    cm = confusion_matrix(preds, labels, len(classes))
    per_class, summary = report_rows(cm, classes)

    header = "Classification report - ResNet50 + Mel spectrogram - test set"
    print("\n" + "=" * len(header), flush=True)
    print(header, flush=True)
    print("=" * len(header), flush=True)
    print(format_report(per_class, summary), flush=True)
    print("=" * len(header), flush=True)

    print("\n" + format_confusion(cm, classes), flush=True)

    if "val_accuracy" in ckpt:
        gap = ckpt["val_accuracy"] - summary["accuracy"]
        print(f"\nvalidation {ckpt['val_accuracy'] * 100:.2f}%  ->  "
              f"test {summary['accuracy'] * 100:.2f}%   (gap {gap * 100:+.2f} pts)",
              flush=True)
        print("  a modest drop is normal: the configuration was chosen on validation.",
              flush=True)

    w = summary["weighted"]
    print(f"\nTable 3 row:  accuracy {summary['accuracy'] * 100:.2f}  "
          f"precision {w['precision'] * 100:.2f}  recall {w['recall'] * 100:.2f}  "
          f"F1 {w['f1'] * 100:.2f}", flush=True)

    print(f"\nTEST ACCURACY: {summary['accuracy'] * 100:.2f}%", flush=True)


if __name__ == "__main__":
    main()


##MEL + VGGNET19

In [ ]:
#This script evaluates the best VGG19 + mel checkpoint on the test split.
#Reads DATA_DIR/<split>/<class>/*.npy and the checkpoint from CKPT_PATH, and
#prints the test accuracy, classification report and confusion matrix to the
#terminal. Nothing is written to disk.

import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as VT
from torchvision.models import vgg19

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **k):
        return x


# ======================= EDIT THESE =======================
DATA_DIR = OUTPUTS / "mel_dataset"                        # folder holding <split>/<class>/*.npy (the mel cell's OUTPUT_DIR)
CKPT_PATH = OUTPUTS / "vgg19_mel" / "vgg19_mel_best.pt"       # best checkpoint saved by the training cell

TEST_SPLIT = "test"

IMG_SIZE = 224
BATCH = 64
NUM_WORKERS = 2      # set to 0 if the DataLoader gives trouble in the notebook

IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
# ==========================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)


def find_split_root(base):
    base = Path(base)
    if not base.exists():
        return None
    for d in [base] + sorted(p for p in base.iterdir() if p.is_dir()):
        if (d / TEST_SPLIT).is_dir() and (d / "train").is_dir() \
                and next((d / TEST_SPLIT).rglob("*.npy"), None) is not None:
            return d
    return None


class MelDataset(Dataset):
    """(128, F) mel .npy -> 3 channels, 224x224, normalised as the model was trained."""

    def __init__(self, split_dir, class_to_idx, mean, std):
        self.samples = []
        for cls, idx in class_to_idx.items():
            cdir = split_dir / cls
            if not cdir.is_dir():
                print(f"  !! {split_dir.name}: no folder for class '{cls}'", flush=True)
                continue
            for npy in sorted(cdir.glob("*.npy")):
                self.samples.append((npy, idx))
        self.resize = VT.Resize((IMG_SIZE, IMG_SIZE), antialias=True)
        self.normalize = VT.Normalize(mean, std)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        arr = np.load(path).astype(np.float32)
        t = torch.from_numpy(np.ascontiguousarray(arr))
        if t.ndim == 2:
            t = t.unsqueeze(0)
        if t.shape[0] == 1:
            t = t.repeat(3, 1, 1)
        return self.normalize(self.resize(t)), label


def confusion_matrix(preds, labels, n_classes):
    cm = np.zeros((n_classes, n_classes), dtype=np.int64)
    for t, p in zip(labels, preds):
        cm[t, p] += 1
    return cm


def report_rows(cm, classes):
    support = cm.sum(axis=1).astype(float)
    pred_tot = cm.sum(axis=0).astype(float)
    tp = np.diag(cm).astype(float)

    precision = np.divide(tp, pred_tot, out=np.zeros_like(tp), where=pred_tot > 0)
    recall = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
    denom = precision + recall
    f1 = np.divide(2 * precision * recall, denom, out=np.zeros_like(tp), where=denom > 0)

    total = support.sum()
    accuracy = tp.sum() / total if total > 0 else 0.0
    w = support / total if total > 0 else np.zeros_like(support)

    per_class = [{"name": c, "precision": precision[i], "recall": recall[i],
                  "f1": f1[i], "support": int(support[i])}
                 for i, c in enumerate(classes)]
    summary = {
        "accuracy": accuracy,
        "macro": {"precision": precision.mean(), "recall": recall.mean(),
                  "f1": f1.mean(), "support": int(total)},
        "weighted": {"precision": float((precision * w).sum()),
                     "recall": float((recall * w).sum()),
                     "f1": float((f1 * w).sum()), "support": int(total)},
    }
    return per_class, summary


def format_report(per_class, summary):
    name_w = max(14, max(len(r["name"]) for r in per_class) + 2)
    out = [f"{'':<{name_w}}{'precision':>10}{'recall':>10}{'f1-score':>10}{'support':>10}", ""]
    for r in per_class:
        out.append(f"{r['name']:<{name_w}}{r['precision']:>10.4f}{r['recall']:>10.4f}"
                   f"{r['f1']:>10.4f}{r['support']:>10d}")
    out.append("")
    s = summary
    out.append(f"{'accuracy':<{name_w}}{'':>10}{'':>10}{s['accuracy']:>10.4f}"
               f"{s['macro']['support']:>10d}")
    for key, label in (("macro", "macro avg"), ("weighted", "weighted avg")):
        m = s[key]
        out.append(f"{label:<{name_w}}{m['precision']:>10.4f}{m['recall']:>10.4f}"
                   f"{m['f1']:>10.4f}{m['support']:>10d}")
    return "\n".join(out)


def format_confusion(cm, classes):
    name_w = max(12, max(len(c) for c in classes) + 2)
    col_w = max(8, max(len(c) for c in classes) + 2)
    out = ["confusion matrix (rows = true class, columns = predicted):",
           f"{'':<{name_w}}" + "".join(f"{c:>{col_w}}" for c in classes)]
    for i, c in enumerate(classes):
        out.append(f"{c:<{name_w}}" + "".join(f"{cm[i, j]:>{col_w}d}"
                                              for j in range(len(classes))))
    return "\n".join(out)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    preds, labels = [], []
    for x, y in tqdm(loader, desc="test"):
        x = x.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
            logits = model(x)                    # torchvision returns logits
        preds.append(logits.argmax(1).cpu())
        labels.append(y)
    return torch.cat(preds).numpy(), torch.cat(labels).numpy()


def main():
    print(f"Device: {DEVICE}", flush=True)
    if not CKPT_PATH.is_file():
        raise SystemExit(f"checkpoint not found: {CKPT_PATH}")

    ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
    classes = ckpt["classes"]
    state = ckpt["model_state"]
    hp = ckpt.get("hyperparameters", {})
    norm = ckpt.get("normalization", {})
    mean = norm.get("mean", IMAGENET_MEAN)
    std = norm.get("std", IMAGENET_STD)

    print(f"\nloaded {CKPT_PATH.name}", flush=True)
    print(f"  model {ckpt.get('model')} ({ckpt.get('model_name')})", flush=True)
    print(f"  representation {ckpt.get('representation')}", flush=True)
    print(f"  classes {classes}", flush=True)
    print(f"  config: lr {hp.get('lr')}, epochs {hp.get('epochs')}, "
          f"batch {hp.get('batch')}, {hp.get('optimizer')}", flush=True)
    print(f"  normalisation mean {mean} std {std}  (taken from the checkpoint)", flush=True)
    if "val_accuracy" in ckpt:
        print(f"  validation accuracy at selection: "
              f"{ckpt['val_accuracy'] * 100:.2f}%", flush=True)

    root = find_split_root(DATA_DIR)
    if root is None:
        raise SystemExit(f"could not find a '{TEST_SPLIT}' split under {DATA_DIR}")
    class_to_idx = {c: i for i, c in enumerate(classes)}
    ds = MelDataset(root / TEST_SPLIT, class_to_idx, mean, std)
    if len(ds) == 0:
        raise SystemExit(f"no .npy files found in {root / TEST_SPLIT}")
    loader = DataLoader(ds, batch_size=BATCH, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
    probe, _ = ds[0]
    print(f"\ntest clips {len(ds)}  |  input shape {tuple(probe.shape)}\n", flush=True)

    model = vgg19(weights=None)
    model.classifier[6] = nn.Linear(model.classifier[6].in_features, len(classes))
    model.load_state_dict(state)
    model = model.to(DEVICE)

    preds, labels = predict(model, loader)
    cm = confusion_matrix(preds, labels, len(classes))
    per_class, summary = report_rows(cm, classes)

    header = "Classification report - VGG19 + Mel spectrogram - test set"
    print("\n" + "=" * len(header), flush=True)
    print(header, flush=True)
    print("=" * len(header), flush=True)
    print(format_report(per_class, summary), flush=True)
    print("=" * len(header), flush=True)

    print("\n" + format_confusion(cm, classes), flush=True)

    if "val_accuracy" in ckpt:
        gap = ckpt["val_accuracy"] - summary["accuracy"]
        print(f"\nvalidation {ckpt['val_accuracy'] * 100:.2f}%  ->  "
              f"test {summary['accuracy'] * 100:.2f}%   (gap {gap * 100:+.2f} pts)",
              flush=True)
        print("  a modest drop is normal: the configuration was chosen on validation.",
              flush=True)

    w = summary["weighted"]
    print(f"\nTable 3 row:  accuracy {summary['accuracy'] * 100:.2f}  "
          f"precision {w['precision'] * 100:.2f}  recall {w['recall'] * 100:.2f}  "
          f"F1 {w['f1'] * 100:.2f}", flush=True)

    print(f"\nTEST ACCURACY: {summary['accuracy'] * 100:.2f}%", flush=True)


if __name__ == "__main__":
    main()


##MEL + VIT (testing)

In [ ]:
#This script evaluates the best ViT + mel checkpoint on the test split.
#Reads DATA_DIR/<split>/<class>/*.npy and the checkpoint from CKPT_PATH, and
#prints the test accuracy, classification report and confusion matrix to the
#terminal. Nothing is written to disk.

import time
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as VT
from transformers import ViTForImageClassification

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **k):
        return x


# ======================= EDIT THESE =======================
DATA_DIR = OUTPUTS / "mel_dataset"                        # folder holding <split>/<class>/*.npy (the mel cell's OUTPUT_DIR)
CKPT_PATH = OUTPUTS / "vit_mel" / "vit_mel_best.pt"           # best checkpoint saved by the training cell

MODEL_NAME = "google/vit-base-patch16-224"
TEST_SPLIT = "test"

IMG_SIZE = 224
BATCH = 64
NUM_WORKERS = 2      # set to 0 if the DataLoader gives trouble in the notebook

VIT_MEAN, VIT_STD = [0.5, 0.5, 0.5], [0.5, 0.5, 0.5]
# ==========================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)


def find_split_root(base):
    base = Path(base)
    if not base.exists():
        return None
    for d in [base] + sorted(p for p in base.iterdir() if p.is_dir()):
        if (d / TEST_SPLIT).is_dir() and (d / "train").is_dir() \
                and next((d / TEST_SPLIT).rglob("*.npy"), None) is not None:
            return d
    return None


class MelDataset(Dataset):
    """(128, F) mel .npy -> 3 channels, 224x224, normalised as the model was trained."""

    def __init__(self, split_dir, class_to_idx, mean, std):
        self.samples = []
        for cls, idx in class_to_idx.items():
            cdir = split_dir / cls
            if not cdir.is_dir():
                print(f"  !! {split_dir.name}: no folder for class '{cls}'", flush=True)
                continue
            for npy in sorted(cdir.glob("*.npy")):
                self.samples.append((npy, idx))
        self.resize = VT.Resize((IMG_SIZE, IMG_SIZE), antialias=True)
        self.normalize = VT.Normalize(mean, std)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        arr = np.load(path).astype(np.float32)
        t = torch.from_numpy(np.ascontiguousarray(arr))
        if t.ndim == 2:
            t = t.unsqueeze(0)
        if t.shape[0] == 1:
            t = t.repeat(3, 1, 1)
        return self.normalize(self.resize(t)), label


def confusion_matrix(preds, labels, n_classes):
    cm = np.zeros((n_classes, n_classes), dtype=np.int64)
    for t, p in zip(labels, preds):
        cm[t, p] += 1
    return cm


def report_rows(cm, classes):
    support = cm.sum(axis=1).astype(float)
    pred_tot = cm.sum(axis=0).astype(float)
    tp = np.diag(cm).astype(float)

    precision = np.divide(tp, pred_tot, out=np.zeros_like(tp), where=pred_tot > 0)
    recall = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
    denom = precision + recall
    f1 = np.divide(2 * precision * recall, denom, out=np.zeros_like(tp), where=denom > 0)

    total = support.sum()
    accuracy = tp.sum() / total if total > 0 else 0.0
    w = support / total if total > 0 else np.zeros_like(support)

    per_class = [{"name": c, "precision": precision[i], "recall": recall[i],
                  "f1": f1[i], "support": int(support[i])}
                 for i, c in enumerate(classes)]
    summary = {
        "accuracy": accuracy,
        "macro": {"precision": precision.mean(), "recall": recall.mean(),
                  "f1": f1.mean(), "support": int(total)},
        "weighted": {"precision": float((precision * w).sum()),
                     "recall": float((recall * w).sum()),
                     "f1": float((f1 * w).sum()), "support": int(total)},
    }
    return per_class, summary


def format_report(per_class, summary):
    name_w = max(14, max(len(r["name"]) for r in per_class) + 2)
    out = [f"{'':<{name_w}}{'precision':>10}{'recall':>10}{'f1-score':>10}{'support':>10}", ""]
    for r in per_class:
        out.append(f"{r['name']:<{name_w}}{r['precision']:>10.4f}{r['recall']:>10.4f}"
                   f"{r['f1']:>10.4f}{r['support']:>10d}")
    out.append("")
    s = summary
    out.append(f"{'accuracy':<{name_w}}{'':>10}{'':>10}{s['accuracy']:>10.4f}"
               f"{s['macro']['support']:>10d}")
    for key, label in (("macro", "macro avg"), ("weighted", "weighted avg")):
        m = s[key]
        out.append(f"{label:<{name_w}}{m['precision']:>10.4f}{m['recall']:>10.4f}"
                   f"{m['f1']:>10.4f}{m['support']:>10d}")
    return "\n".join(out)


def format_confusion(cm, classes):
    name_w = max(12, max(len(c) for c in classes) + 2)
    col_w = max(8, max(len(c) for c in classes) + 2)
    out = ["confusion matrix (rows = true class, columns = predicted):",
           f"{'':<{name_w}}" + "".join(f"{c:>{col_w}}" for c in classes)]
    for i, c in enumerate(classes):
        out.append(f"{c:<{name_w}}" + "".join(f"{cm[i, j]:>{col_w}d}"
                                              for j in range(len(classes))))
    return "\n".join(out)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    preds, labels = [], []
    for x, y in tqdm(loader, desc="test"):
        x = x.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
            logits = model(pixel_values=x).logits
        preds.append(logits.argmax(1).cpu())
        labels.append(y)
    return torch.cat(preds).numpy(), torch.cat(labels).numpy()


def main():
    print(f"Device: {DEVICE}", flush=True)
    if not CKPT_PATH.is_file():
        raise SystemExit(f"checkpoint not found: {CKPT_PATH}")

    ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
    classes = ckpt["classes"]
    state = ckpt["model_state"]
    hp = ckpt.get("hyperparameters", {})
    norm = ckpt.get("normalization", {})
    mean = norm.get("mean", VIT_MEAN)
    std = norm.get("std", VIT_STD)

    print(f"\nloaded {CKPT_PATH.name}", flush=True)
    print(f"  model {ckpt.get('model')} ({ckpt.get('model_name')})", flush=True)
    print(f"  representation {ckpt.get('representation')}", flush=True)
    print(f"  classes {classes}", flush=True)
    print(f"  config: lr {hp.get('lr')}, epochs {hp.get('epochs')}, "
          f"batch {hp.get('batch')}, {hp.get('optimizer')}", flush=True)
    print(f"  normalisation mean {mean} std {std}  (taken from the checkpoint)", flush=True)
    if "val_accuracy" in ckpt:
        print(f"  validation accuracy at selection: "
              f"{ckpt['val_accuracy'] * 100:.2f}%", flush=True)

    root = find_split_root(DATA_DIR)
    if root is None:
        raise SystemExit(f"could not find a '{TEST_SPLIT}' split under {DATA_DIR}")
    class_to_idx = {c: i for i, c in enumerate(classes)}
    ds = MelDataset(root / TEST_SPLIT, class_to_idx, mean, std)
    if len(ds) == 0:
        raise SystemExit(f"no .npy files found in {root / TEST_SPLIT}")
    loader = DataLoader(ds, batch_size=BATCH, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
    probe, _ = ds[0]
    print(f"\ntest clips {len(ds)}  |  input shape {tuple(probe.shape)}\n", flush=True)

    model = ViTForImageClassification.from_pretrained(
        MODEL_NAME, num_labels=len(classes), ignore_mismatched_sizes=True)
    model.load_state_dict(state)
    model = model.to(DEVICE)

    preds, labels = predict(model, loader)
    cm = confusion_matrix(preds, labels, len(classes))
    per_class, summary = report_rows(cm, classes)

    header = "Classification report - ViT + Mel spectrogram - test set"
    print("\n" + "=" * len(header), flush=True)
    print(header, flush=True)
    print("=" * len(header), flush=True)
    print(format_report(per_class, summary), flush=True)
    print("=" * len(header), flush=True)

    print("\n" + format_confusion(cm, classes), flush=True)

    if "val_accuracy" in ckpt:
        gap = ckpt["val_accuracy"] - summary["accuracy"]
        print(f"\nvalidation {ckpt['val_accuracy'] * 100:.2f}%  ->  "
              f"test {summary['accuracy'] * 100:.2f}%   (gap {gap * 100:+.2f} pts)",
              flush=True)
        print("  a modest drop is normal: the configuration was chosen on validation.",
              flush=True)

    w = summary["weighted"]
    print(f"\nTable 3 row:  accuracy {summary['accuracy'] * 100:.2f}  "
          f"precision {w['precision'] * 100:.2f}  recall {w['recall'] * 100:.2f}  "
          f"F1 {w['f1'] * 100:.2f}", flush=True)

    print(f"\nTEST ACCURACY: {summary['accuracy'] * 100:.2f}%", flush=True)


if __name__ == "__main__":
    main()


##MFCC + KNN (testing)

In [ ]:
#This script evaluates the best KNN + MFCC model on the test split. Reads the
#X/y arrays from DATA_DIR and the model from MODEL_PATH, and prints the test
#accuracy, classification report and confusion matrix to the terminal. Nothing
#is written to disk.

import json
import sys
import time
from pathlib import Path

import numpy as np

# ======================= EDIT THESE =======================
DATA_DIR = OUTPUTS / "mfcc_dataset"                       # folder holding X_<split>.npy / y_<split>.npy (the MFCC cell's OUTPUT_DIR)
MODEL_PATH = OUTPUTS / "knn_mfcc" / "knn_mfcc_best.joblib"    # best model saved by the training cell
SWEEP_LOG = OUTPUTS / "knn_mfcc" / "sweep_results.json"       # optional, only used for the validation -> test comparison

TEST_SPLIT = "test"
# ==========================================================


def find_data_root(base):
    base = Path(base)
    if not base.exists():
        return None
    for d in [base] + sorted(p for p in base.iterdir() if p.is_dir()):
        if (d / f"X_{TEST_SPLIT}.npy").is_file() and (d / f"y_{TEST_SPLIT}.npy").is_file():
            return d
    return None


def confusion_matrix(preds, labels, n_classes):
    cm = np.zeros((n_classes, n_classes), dtype=np.int64)
    for t, p in zip(labels, preds):
        cm[t, p] += 1
    return cm


def report_rows(cm, classes):
    support = cm.sum(axis=1).astype(float)
    pred_tot = cm.sum(axis=0).astype(float)
    tp = np.diag(cm).astype(float)

    precision = np.divide(tp, pred_tot, out=np.zeros_like(tp), where=pred_tot > 0)
    recall = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
    denom = precision + recall
    f1 = np.divide(2 * precision * recall, denom, out=np.zeros_like(tp), where=denom > 0)

    total = support.sum()
    accuracy = tp.sum() / total if total > 0 else 0.0
    w = support / total if total > 0 else np.zeros_like(support)

    per_class = [{"name": c, "precision": precision[i], "recall": recall[i],
                  "f1": f1[i], "support": int(support[i])}
                 for i, c in enumerate(classes)]
    summary = {
        "accuracy": accuracy,
        "macro": {"precision": precision.mean(), "recall": recall.mean(),
                  "f1": f1.mean(), "support": int(total)},
        "weighted": {"precision": float((precision * w).sum()),
                     "recall": float((recall * w).sum()),
                     "f1": float((f1 * w).sum()), "support": int(total)},
    }
    return per_class, summary


def format_report(per_class, summary):
    name_w = max(14, max(len(r["name"]) for r in per_class) + 2)
    out = [f"{'':<{name_w}}{'precision':>10}{'recall':>10}{'f1-score':>10}{'support':>10}", ""]
    for r in per_class:
        out.append(f"{r['name']:<{name_w}}{r['precision']:>10.4f}{r['recall']:>10.4f}"
                   f"{r['f1']:>10.4f}{r['support']:>10d}")
    out.append("")
    s = summary
    out.append(f"{'accuracy':<{name_w}}{'':>10}{'':>10}{s['accuracy']:>10.4f}"
               f"{s['macro']['support']:>10d}")
    for key, label in (("macro", "macro avg"), ("weighted", "weighted avg")):
        m = s[key]
        out.append(f"{label:<{name_w}}{m['precision']:>10.4f}{m['recall']:>10.4f}"
                   f"{m['f1']:>10.4f}{m['support']:>10d}")
    return "\n".join(out)


def format_confusion(cm, classes):
    name_w = max(12, max(len(c) for c in classes) + 2)
    col_w = max(8, max(len(c) for c in classes) + 2)
    out = ["confusion matrix (rows = true class, columns = predicted):",
           f"{'':<{name_w}}" + "".join(f"{c:>{col_w}}" for c in classes)]
    for i, c in enumerate(classes):
        out.append(f"{c:<{name_w}}" + "".join(f"{cm[i, j]:>{col_w}d}"
                                              for j in range(len(classes))))
    return "\n".join(out)


def main():
    try:
        import joblib
        import sklearn                                   # noqa: F401
    except ImportError:
        sys.exit("this script needs scikit-learn and joblib:  pip install scikit-learn joblib")

    if not MODEL_PATH.is_file():
        raise SystemExit(f"model not found: {MODEL_PATH}")

    root = find_data_root(DATA_DIR)
    if root is None:
        raise SystemExit(f"could not find X_{TEST_SPLIT}.npy under {DATA_DIR}")
    Xte = np.load(root / f"X_{TEST_SPLIT}.npy").astype(np.float32)
    yte = np.load(root / f"y_{TEST_SPLIT}.npy")

    classes = json.loads((root / "classes.json").read_text()) \
        if (root / "classes.json").is_file() else [str(i) for i in range(int(yte.max()) + 1)]

    print(f"loaded {MODEL_PATH.name} ({MODEL_PATH.stat().st_size / 1e6:.1f} MB)", flush=True)
    model = joblib.load(MODEL_PATH)
    print(f"  {type(model).__name__}: k={getattr(model, 'n_neighbors', '?')}, "
          f"weights={getattr(model, 'weights', '?')}, "
          f"metric={getattr(model, 'metric', '?')}", flush=True)
    print(f"  (a KNN file stores the training set, not learned weights)", flush=True)

    val_acc = None
    if SWEEP_LOG.is_file():
        log = json.loads(SWEEP_LOG.read_text())
        best = log.get("best", {})
        val_acc = best.get("val_accuracy")
        if val_acc is not None:
            print(f"  selected on validation: k={best.get('k')}, "
                  f"val_accuracy={val_acc * 100:.2f}%", flush=True)

    print(f"\nclasses: {classes}", flush=True)
    print(f"test {Xte.shape}  labels {yte.shape}", flush=True)
    print(f"feature check: mean {Xte.mean():+.4f}, std {Xte.std():.4f} "
          f"(standardised in the builder -> not re-scaled here)", flush=True)

    if hasattr(model, "classes_"):
        model_labels = set(int(c) for c in model.classes_)
        data_labels = set(int(c) for c in np.unique(yte))
        if not data_labels.issubset(model_labels):
            print(f"  !! label mismatch - model knows {sorted(model_labels)}, "
                  f"test contains {sorted(data_labels)}", flush=True)

    t0 = time.time()
    preds = model.predict(Xte)
    print(f"\npredicted {len(preds)} clips in {time.time() - t0:.1f}s", flush=True)

    cm = confusion_matrix(preds, yte, len(classes))
    per_class, summary = report_rows(cm, classes)

    header = "Classification report - KNN + MFCC - test set"
    print("\n" + "=" * len(header), flush=True)
    print(header, flush=True)
    print("=" * len(header), flush=True)
    print(format_report(per_class, summary), flush=True)
    print("=" * len(header), flush=True)

    print("\n" + format_confusion(cm, classes), flush=True)

    if val_acc is not None:
        gap = val_acc - summary["accuracy"]
        print(f"\nvalidation {val_acc * 100:.2f}%  ->  test {summary['accuracy'] * 100:.2f}%"
              f"   (gap {gap * 100:+.2f} pts)", flush=True)
        print("  a modest drop is normal: k was chosen on validation.", flush=True)

    w = summary["weighted"]
    print(f"\nTable 3 row:  accuracy {summary['accuracy'] * 100:.2f}  "
          f"precision {w['precision'] * 100:.2f}  recall {w['recall'] * 100:.2f}  "
          f"F1 {w['f1'] * 100:.2f}", flush=True)

    print(f"\nTEST ACCURACY: {summary['accuracy'] * 100:.2f}%", flush=True)


if __name__ == "__main__":
    main()


##MFCC + LOGISTIC REGRESSION (testing)

In [ ]:
#This script evaluates the best Logistic Regression + MFCC model on the test
#split, using the saved coefficient/intercept arrays directly (no sklearn
#needed). Prints the test accuracy, classification report and confusion matrix
#to the terminal. Nothing is written to disk.

import json
import time
from pathlib import Path

import numpy as np

# ======================= EDIT THESE =======================
DATA_DIR = OUTPUTS / "mfcc_dataset"                       # folder holding X_<split>.npy / y_<split>.npy (the MFCC cell's OUTPUT_DIR)
COEF_PATH = OUTPUTS / "logreg_mfcc" / "logreg_mfcc_best_coef.npy"            # coefficients saved by the training cell
INTERCEPT_PATH = OUTPUTS / "logreg_mfcc" / "logreg_mfcc_best_intercept.npy"  # intercepts saved by the training cell
GRID_LOG = OUTPUTS / "logreg_mfcc" / "grid_results.json"      # optional, only used for the validation -> test comparison

TEST_SPLIT = "test"
# ==========================================================


def predict(X, coef, intercept):
    scores = X @ coef.T + intercept
    return scores.argmax(axis=1)


def find_data_root(base):
    base = Path(base)
    if not base.exists():
        return None
    for d in [base] + sorted(p for p in base.iterdir() if p.is_dir()):
        if (d / f"X_{TEST_SPLIT}.npy").is_file() and (d / f"y_{TEST_SPLIT}.npy").is_file():
            return d
    return None


def confusion_matrix(preds, labels, n_classes):
    cm = np.zeros((n_classes, n_classes), dtype=np.int64)
    for t, p in zip(labels, preds):
        cm[t, p] += 1
    return cm


def report_rows(cm, classes):
    support = cm.sum(axis=1).astype(float)
    pred_tot = cm.sum(axis=0).astype(float)
    tp = np.diag(cm).astype(float)

    precision = np.divide(tp, pred_tot, out=np.zeros_like(tp), where=pred_tot > 0)
    recall = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
    denom = precision + recall
    f1 = np.divide(2 * precision * recall, denom, out=np.zeros_like(tp), where=denom > 0)

    total = support.sum()
    accuracy = tp.sum() / total if total > 0 else 0.0
    w = support / total if total > 0 else np.zeros_like(support)

    per_class = [{"name": c, "precision": precision[i], "recall": recall[i],
                  "f1": f1[i], "support": int(support[i])}
                 for i, c in enumerate(classes)]
    summary = {
        "accuracy": accuracy,
        "macro": {"precision": precision.mean(), "recall": recall.mean(),
                  "f1": f1.mean(), "support": int(total)},
        "weighted": {"precision": float((precision * w).sum()),
                     "recall": float((recall * w).sum()),
                     "f1": float((f1 * w).sum()), "support": int(total)},
    }
    return per_class, summary


def format_report(per_class, summary):
    name_w = max(14, max(len(r["name"]) for r in per_class) + 2)
    out = [f"{'':<{name_w}}{'precision':>10}{'recall':>10}{'f1-score':>10}{'support':>10}", ""]
    for r in per_class:
        out.append(f"{r['name']:<{name_w}}{r['precision']:>10.4f}{r['recall']:>10.4f}"
                   f"{r['f1']:>10.4f}{r['support']:>10d}")
    out.append("")
    s = summary
    out.append(f"{'accuracy':<{name_w}}{'':>10}{'':>10}{s['accuracy']:>10.4f}"
               f"{s['macro']['support']:>10d}")
    for key, label in (("macro", "macro avg"), ("weighted", "weighted avg")):
        m = s[key]
        out.append(f"{label:<{name_w}}{m['precision']:>10.4f}{m['recall']:>10.4f}"
                   f"{m['f1']:>10.4f}{m['support']:>10d}")
    return "\n".join(out)


def format_confusion(cm, classes):
    name_w = max(12, max(len(c) for c in classes) + 2)
    col_w = max(8, max(len(c) for c in classes) + 2)
    out = ["confusion matrix (rows = true class, columns = predicted):",
           f"{'':<{name_w}}" + "".join(f"{c:>{col_w}}" for c in classes)]
    for i, c in enumerate(classes):
        out.append(f"{c:<{name_w}}" + "".join(f"{cm[i, j]:>{col_w}d}"
                                              for j in range(len(classes))))
    return "\n".join(out)


def main():
    for p in (COEF_PATH, INTERCEPT_PATH):
        if not p.is_file():
            raise SystemExit(f"weights not found: {p}")

    coef = np.load(COEF_PATH).astype(np.float64)
    intercept = np.load(INTERCEPT_PATH).astype(np.float64)
    print(f"loaded coef {coef.shape}  intercept {intercept.shape}", flush=True)

    root = find_data_root(DATA_DIR)
    if root is None:
        raise SystemExit(f"could not find X_{TEST_SPLIT}.npy under {DATA_DIR}")
    Xte = np.load(root / f"X_{TEST_SPLIT}.npy").astype(np.float64)
    yte = np.load(root / f"y_{TEST_SPLIT}.npy")

    classes = json.loads((root / "classes.json").read_text()) \
        if (root / "classes.json").is_file() else [str(i) for i in range(coef.shape[0])]

    if coef.shape[0] != len(classes):
        raise SystemExit(f"coef has {coef.shape[0]} rows but there are "
                         f"{len(classes)} classes")
    if coef.shape[1] != Xte.shape[1]:
        raise SystemExit(f"coef expects {coef.shape[1]} features but the test "
                         f"matrix has {Xte.shape[1]}")

    val_acc = None
    if GRID_LOG.is_file():
        log = json.loads(GRID_LOG.read_text())
        best = log.get("best", {})
        val_acc = best.get("val_accuracy")
        if val_acc is not None:
            print(f"selected on validation: C={best.get('C')}, "
                  f"max_iter={best.get('max_iter')}, "
                  f"val_accuracy={val_acc * 100:.2f}%", flush=True)

    print(f"\nclasses: {classes}", flush=True)
    print(f"test {Xte.shape}  labels {yte.shape}", flush=True)
    print(f"feature check: mean {Xte.mean():+.4f}, std {Xte.std():.4f}", flush=True)

    t0 = time.time()
    preds = predict(Xte, coef, intercept)
    print(f"\npredicted {len(preds)} clips in {time.time() - t0:.2f}s", flush=True)

    cm = confusion_matrix(preds, yte, len(classes))
    per_class, summary = report_rows(cm, classes)

    header = "Classification report - Logistic Regression + MFCC - test set"
    print("\n" + "=" * len(header), flush=True)
    print(header, flush=True)
    print("=" * len(header), flush=True)
    print(format_report(per_class, summary), flush=True)
    print("=" * len(header), flush=True)

    print("\n" + format_confusion(cm, classes), flush=True)

    if val_acc is not None:
        gap = val_acc - summary["accuracy"]
        print(f"\nvalidation {val_acc * 100:.2f}%  ->  test {summary['accuracy'] * 100:.2f}%"
              f"   (gap {gap * 100:+.2f} pts)", flush=True)

    w = summary["weighted"]
    print(f"\nTable 3 row:  accuracy {summary['accuracy'] * 100:.2f}  "
          f"precision {w['precision'] * 100:.2f}  recall {w['recall'] * 100:.2f}  "
          f"F1 {w['f1'] * 100:.2f}", flush=True)

    print(f"\nTEST ACCURACY: {summary['accuracy'] * 100:.2f}%", flush=True)


if __name__ == "__main__":
    main()


##MFCC + RANDOM FOREST (testing)

In [ ]:
#This script evaluates the best Random Forest + MFCC model on the test split.
#Reads the X/y arrays from DATA_DIR and the model from MODEL_PATH, and prints
#the test accuracy, classification report, confusion matrix and feature
#importances to the terminal. Nothing is written to disk.

import json
import sys
import time
from pathlib import Path

import numpy as np

# ======================= EDIT THESE =======================
DATA_DIR = OUTPUTS / "mfcc_dataset"                       # folder holding X_<split>.npy / y_<split>.npy (the MFCC cell's OUTPUT_DIR)
MODEL_PATH = OUTPUTS / "rf_mfcc" / "rf_mfcc_best.joblib"      # best model saved by the training cell
IMPORTANCES_PATH = OUTPUTS / "rf_mfcc" / "rf_mfcc_best_importances.npy"  # optional fallback for feature importances
GRID_LOG = OUTPUTS / "rf_mfcc" / "grid_results.json"          # optional, only used for the validation -> test comparison

TEST_SPLIT = "test"
TOP_FEATURES = 10
# ==========================================================


def find_data_root(base):
    base = Path(base)
    if not base.exists():
        return None
    for d in [base] + sorted(p for p in base.iterdir() if p.is_dir()):
        if (d / f"X_{TEST_SPLIT}.npy").is_file() and (d / f"y_{TEST_SPLIT}.npy").is_file():
            return d
    return None


def confusion_matrix(preds, labels, n_classes):
    cm = np.zeros((n_classes, n_classes), dtype=np.int64)
    for t, p in zip(labels, preds):
        cm[t, p] += 1
    return cm


def report_rows(cm, classes):
    support = cm.sum(axis=1).astype(float)
    pred_tot = cm.sum(axis=0).astype(float)
    tp = np.diag(cm).astype(float)

    precision = np.divide(tp, pred_tot, out=np.zeros_like(tp), where=pred_tot > 0)
    recall = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
    denom = precision + recall
    f1 = np.divide(2 * precision * recall, denom, out=np.zeros_like(tp), where=denom > 0)

    total = support.sum()
    accuracy = tp.sum() / total if total > 0 else 0.0
    w = support / total if total > 0 else np.zeros_like(support)

    per_class = [{"name": c, "precision": precision[i], "recall": recall[i],
                  "f1": f1[i], "support": int(support[i])}
                 for i, c in enumerate(classes)]
    summary = {
        "accuracy": accuracy,
        "macro": {"precision": precision.mean(), "recall": recall.mean(),
                  "f1": f1.mean(), "support": int(total)},
        "weighted": {"precision": float((precision * w).sum()),
                     "recall": float((recall * w).sum()),
                     "f1": float((f1 * w).sum()), "support": int(total)},
    }
    return per_class, summary


def format_report(per_class, summary):
    name_w = max(14, max(len(r["name"]) for r in per_class) + 2)
    out = [f"{'':<{name_w}}{'precision':>10}{'recall':>10}{'f1-score':>10}{'support':>10}", ""]
    for r in per_class:
        out.append(f"{r['name']:<{name_w}}{r['precision']:>10.4f}{r['recall']:>10.4f}"
                   f"{r['f1']:>10.4f}{r['support']:>10d}")
    out.append("")
    s = summary
    out.append(f"{'accuracy':<{name_w}}{'':>10}{'':>10}{s['accuracy']:>10.4f}"
               f"{s['macro']['support']:>10d}")
    for key, label in (("macro", "macro avg"), ("weighted", "weighted avg")):
        m = s[key]
        out.append(f"{label:<{name_w}}{m['precision']:>10.4f}{m['recall']:>10.4f}"
                   f"{m['f1']:>10.4f}{m['support']:>10d}")
    return "\n".join(out)


def format_confusion(cm, classes):
    name_w = max(12, max(len(c) for c in classes) + 2)
    col_w = max(8, max(len(c) for c in classes) + 2)
    out = ["confusion matrix (rows = true class, columns = predicted):",
           f"{'':<{name_w}}" + "".join(f"{c:>{col_w}}" for c in classes)]
    for i, c in enumerate(classes):
        out.append(f"{c:<{name_w}}" + "".join(f"{cm[i, j]:>{col_w}d}"
                                              for j in range(len(classes))))
    return "\n".join(out)


def main():
    try:
        import joblib
        import sklearn                                   # noqa: F401
    except ImportError:
        sys.exit("this script needs scikit-learn and joblib:  pip install scikit-learn joblib")

    if not MODEL_PATH.is_file():
        raise SystemExit(f"model not found: {MODEL_PATH}")

    root = find_data_root(DATA_DIR)
    if root is None:
        raise SystemExit(f"could not find X_{TEST_SPLIT}.npy under {DATA_DIR}")
    Xte = np.load(root / f"X_{TEST_SPLIT}.npy").astype(np.float32)
    yte = np.load(root / f"y_{TEST_SPLIT}.npy")

    classes = json.loads((root / "classes.json").read_text()) \
        if (root / "classes.json").is_file() else [str(i) for i in range(int(yte.max()) + 1)]
    feature_names = None
    if (root / "metadata.json").is_file():
        feature_names = json.loads((root / "metadata.json").read_text()).get("feature_names")

    print(f"loaded {MODEL_PATH.name} ({MODEL_PATH.stat().st_size / 1e6:.1f} MB)", flush=True)
    model = joblib.load(MODEL_PATH)
    n_est = getattr(model, "n_estimators", "?")
    max_d = getattr(model, "max_depth", "?")
    print(f"  {type(model).__name__}: n_estimators={n_est}, max_depth={max_d}, "
          f"criterion={getattr(model, 'criterion', '?')}", flush=True)
    if hasattr(model, "estimators_"):
        depths = [t.get_depth() for t in model.estimators_]
        print(f"  realised tree depth: min {min(depths)}, "
              f"mean {np.mean(depths):.1f}, max {max(depths)}", flush=True)

    val_acc = None
    if GRID_LOG.is_file():
        log = json.loads(GRID_LOG.read_text())
        best = log.get("best", {})
        val_acc = best.get("val_accuracy")
        if val_acc is not None:
            print(f"  selected on validation: n_estimators={best.get('n_estimators')}, "
                  f"max_depth={best.get('max_depth')}, "
                  f"val_accuracy={val_acc * 100:.2f}%", flush=True)

    print(f"\nclasses: {classes}", flush=True)
    print(f"test {Xte.shape}  labels {yte.shape}", flush=True)

    if hasattr(model, "n_features_in_") and model.n_features_in_ != Xte.shape[1]:
        raise SystemExit(f"model expects {model.n_features_in_} features but the "
                         f"test matrix has {Xte.shape[1]}")

    t0 = time.time()
    preds = model.predict(Xte)
    print(f"\npredicted {len(preds)} clips in {time.time() - t0:.2f}s", flush=True)

    cm = confusion_matrix(preds, yte, len(classes))
    per_class, summary = report_rows(cm, classes)

    header = "Classification report - Random Forest + MFCC - test set"
    print("\n" + "=" * len(header), flush=True)
    print(header, flush=True)
    print("=" * len(header), flush=True)
    print(format_report(per_class, summary), flush=True)
    print("=" * len(header), flush=True)

    print("\n" + format_confusion(cm, classes), flush=True)

    if val_acc is not None:
        gap = val_acc - summary["accuracy"]
        print(f"\nvalidation {val_acc * 100:.2f}%  ->  test {summary['accuracy'] * 100:.2f}%"
              f"   (gap {gap * 100:+.2f} pts)", flush=True)

    importances = None
    if hasattr(model, "feature_importances_"):
        importances = model.feature_importances_
    elif IMPORTANCES_PATH.is_file():
        importances = np.load(IMPORTANCES_PATH)
    if importances is not None:
        names = feature_names if feature_names and len(feature_names) == len(importances) \
            else [f"f{i}" for i in range(len(importances))]
        order = np.argsort(importances)[::-1][:TOP_FEATURES]
        print(f"\ntop {TOP_FEATURES} features by importance:", flush=True)
        for rank, i in enumerate(order, 1):
            print(f"  {rank:>2}. {names[i]:<16} {importances[i]:.4f}", flush=True)
        mean_share = importances[:len(importances) // 2].sum()
        print(f"\n  mean-statistic features carry {mean_share * 100:.1f}% of total "
              f"importance, std features {(1 - mean_share) * 100:.1f}%", flush=True)

    w = summary["weighted"]
    print(f"\nTable 3 row:  accuracy {summary['accuracy'] * 100:.2f}  "
          f"precision {w['precision'] * 100:.2f}  recall {w['recall'] * 100:.2f}  "
          f"F1 {w['f1'] * 100:.2f}", flush=True)

    print(f"\nTEST ACCURACY: {summary['accuracy'] * 100:.2f}%", flush=True)


if __name__ == "__main__":
    main()


##WAV + AST (testing)

In [ ]:
#This script evaluates the best AST checkpoint on the raw-wav test split. Reads
#DATA_DIR/<split>/<class>/*.wav (extracting AST features once into a folder
#cache at FEATURE_CACHE), and prints the test accuracy, classification report
#and confusion matrix to the terminal. No result files are written.

import sys
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import ASTFeatureExtractor, ASTForAudioClassification

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **k):
        return x


# ======================= EDIT THESE =======================
DATA_DIR = OUTPUTS / "cumulative_dataset"     # folder holding <split>/<class>/*.wav
CKPT_PATH = OUTPUTS / "ast_wav" / "ast_wav_best.pt"  # best checkpoint saved by the AST training script
FEATURE_CACHE = OUTPUTS / "ast_features"      # folder where the extracted AST features are cached

MODEL_NAME = "MIT/ast-finetuned-speech-commands-v2"
TEST_SPLIT = "test"

BATCH = 64
NUM_WORKERS = 2      # set to 0 if the DataLoader gives trouble in the notebook
EXTRACT_THREADS = 8

SR = 16000
MAX_LENGTH = 128
NUM_MEL_BINS = 128
# ==========================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

_FE = None
_FE_LOCK = threading.Lock()


def feature_extractor(max_length, num_mel_bins):
    global _FE
    if _FE is None:
        with _FE_LOCK:
            if _FE is None:
                _FE = ASTFeatureExtractor.from_pretrained(
                    MODEL_NAME, max_length=max_length, num_mel_bins=num_mel_bins)
    return _FE


def find_dir_with(base, ext, split):
    base = Path(base)
    if not base.exists():
        return None
    for d in [base] + sorted(p for p in base.iterdir() if p.is_dir()):
        if (d / split).is_dir() and next((d / split).rglob(f"*{ext}"), None) is not None:
            return d
    return None


def build_test_features(classes, sr, max_length, num_mel_bins):
    cached = find_dir_with(FEATURE_CACHE, ".npy", TEST_SPLIT)
    if cached is not None:
        print(f"reusing feature cache at {cached / TEST_SPLIT}", flush=True)
        root = cached
    else:
        print("no feature cache found - extracting the test split", flush=True)
        wav_root = find_dir_with(DATA_DIR, ".wav", TEST_SPLIT)
        if wav_root is None:
            raise SystemExit(f"no '{TEST_SPLIT}' wav split under {DATA_DIR}")

        fe = feature_extractor(max_length, num_mel_bins)
        jobs = []
        for cls in classes:
            cdir = wav_root / TEST_SPLIT / cls
            out_dir = FEATURE_CACHE / TEST_SPLIT / cls
            out_dir.mkdir(parents=True, exist_ok=True)
            if not cdir.is_dir():
                print(f"  !! {TEST_SPLIT}: no folder for class '{cls}'", flush=True)
                continue
            for w in sorted(cdir.glob("*.wav")):
                jobs.append((w, out_dir / f"{w.stem}.npy"))

        def one(job):
            src, dst = job
            if dst.exists() and dst.stat().st_size > 0:
                return
            import librosa
            y, _ = librosa.load(str(src), sr=sr, mono=True)
            y = np.pad(y, (0, sr - len(y))) if len(y) < sr else y[:sr]
            arr = fe(y, sampling_rate=sr, return_tensors="np")["input_values"][0]
            np.save(dst, arr.astype(np.float32))

        print(f"  extracting {len(jobs)} clips on {EXTRACT_THREADS} threads...", flush=True)
        t0 = time.time()
        with ThreadPoolExecutor(max_workers=EXTRACT_THREADS) as pool:
            list(tqdm(pool.map(one, jobs), total=len(jobs), desc="extract"))
        print(f"  done in {time.time() - t0:.0f}s", flush=True)
        root = FEATURE_CACHE

    samples = []
    for idx, cls in enumerate(classes):
        cdir = root / TEST_SPLIT / cls
        if not cdir.is_dir():
            print(f"  !! {TEST_SPLIT}: no folder for class '{cls}'", flush=True)
            continue
        for npy in sorted(cdir.glob("*.npy")):
            samples.append((npy, idx))
    return samples


class FeatureDataset(Dataset):

    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        return torch.from_numpy(np.load(path).astype(np.float32)), label


def confusion_matrix(preds, labels, n_classes):
    cm = np.zeros((n_classes, n_classes), dtype=np.int64)
    for t, p in zip(labels, preds):
        cm[t, p] += 1
    return cm


def report_rows(cm, classes):
    support = cm.sum(axis=1).astype(float)
    pred_tot = cm.sum(axis=0).astype(float)
    tp = np.diag(cm).astype(float)

    precision = np.divide(tp, pred_tot, out=np.zeros_like(tp), where=pred_tot > 0)
    recall = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
    denom = precision + recall
    f1 = np.divide(2 * precision * recall, denom, out=np.zeros_like(tp), where=denom > 0)

    total = support.sum()
    accuracy = tp.sum() / total if total > 0 else 0.0
    w = support / total if total > 0 else np.zeros_like(support)

    per_class = [{"name": c, "precision": precision[i], "recall": recall[i],
                  "f1": f1[i], "support": int(support[i])}
                 for i, c in enumerate(classes)]
    summary = {
        "accuracy": accuracy,
        "macro": {"precision": precision.mean(), "recall": recall.mean(),
                  "f1": f1.mean(), "support": int(total)},
        "weighted": {"precision": float((precision * w).sum()),
                     "recall": float((recall * w).sum()),
                     "f1": float((f1 * w).sum()), "support": int(total)},
    }
    return per_class, summary


def format_report(per_class, summary):
    name_w = max(14, max(len(r["name"]) for r in per_class) + 2)
    out = [f"{'':<{name_w}}{'precision':>10}{'recall':>10}{'f1-score':>10}{'support':>10}", ""]
    for r in per_class:
        out.append(f"{r['name']:<{name_w}}{r['precision']:>10.4f}{r['recall']:>10.4f}"
                   f"{r['f1']:>10.4f}{r['support']:>10d}")
    out.append("")
    s = summary
    out.append(f"{'accuracy':<{name_w}}{'':>10}{'':>10}{s['accuracy']:>10.4f}"
               f"{s['macro']['support']:>10d}")
    for key, label in (("macro", "macro avg"), ("weighted", "weighted avg")):
        m = s[key]
        out.append(f"{label:<{name_w}}{m['precision']:>10.4f}{m['recall']:>10.4f}"
                   f"{m['f1']:>10.4f}{m['support']:>10d}")
    return "\n".join(out)


def format_confusion(cm, classes):
    name_w = max(12, max(len(c) for c in classes) + 2)
    col_w = max(8, max(len(c) for c in classes) + 2)
    out = ["confusion matrix (rows = true class, columns = predicted):",
           f"{'':<{name_w}}" + "".join(f"{c:>{col_w}}" for c in classes)]
    for i, c in enumerate(classes):
        out.append(f"{c:<{name_w}}" + "".join(f"{cm[i, j]:>{col_w}d}"
                                              for j in range(len(classes))))
    return "\n".join(out)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    preds, labels = [], []
    for x, y in tqdm(loader, desc="test"):
        x = x.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
            logits = model(input_values=x).logits
        preds.append(logits.argmax(1).cpu())
        labels.append(y)
    return torch.cat(preds).numpy(), torch.cat(labels).numpy()


def main():
    try:
        import librosa                                   # noqa: F401
    except ImportError:
        sys.exit("this script needs librosa:  pip install librosa")

    print(f"Device: {DEVICE}", flush=True)
    if not CKPT_PATH.is_file():
        raise SystemExit(f"checkpoint not found: {CKPT_PATH}")

    ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
    classes = ckpt["classes"]
    state = ckpt["model_state"]
    hp = ckpt.get("hyperparameters", {})
    fx = ckpt.get("feature_extraction", {})
    sr = fx.get("sampling_rate", SR)
    max_length = fx.get("max_length", MAX_LENGTH)
    num_mel_bins = fx.get("num_mel_bins", NUM_MEL_BINS)

    print(f"\nloaded {CKPT_PATH.name}", flush=True)
    print(f"  model {ckpt.get('model')} ({ckpt.get('model_name')})", flush=True)
    print(f"  representation {ckpt.get('representation')}", flush=True)
    print(f"  classes {classes}", flush=True)
    print(f"  config: lr {hp.get('lr')}, epochs {hp.get('epochs')}, "
          f"batch {hp.get('batch')}, {hp.get('optimizer')}", flush=True)
    print(f"  features: {max_length} x {num_mel_bins} @ {sr} Hz  "
          f"(read from the checkpoint)", flush=True)
    if "val_accuracy" in ckpt:
        print(f"  validation accuracy at selection: "
              f"{ckpt['val_accuracy'] * 100:.2f}%", flush=True)
    print("", flush=True)

    samples = build_test_features(classes, sr, max_length, num_mel_bins)
    if not samples:
        raise SystemExit("no test features found")
    ds = FeatureDataset(samples)
    loader = DataLoader(ds, batch_size=BATCH, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
    probe, _ = ds[0]
    print(f"\ntest clips {len(ds)}  |  input shape {tuple(probe.shape)}  "
          f"(expect ({max_length}, {num_mel_bins}))\n", flush=True)

    model = ASTForAudioClassification.from_pretrained(
        MODEL_NAME, num_labels=len(classes), ignore_mismatched_sizes=True)
    model.load_state_dict(state)
    model = model.to(DEVICE)

    preds, labels = predict(model, loader)
    cm = confusion_matrix(preds, labels, len(classes))
    per_class, summary = report_rows(cm, classes)

    header = "Classification report - AST + raw WAV - test set"
    print("\n" + "=" * len(header), flush=True)
    print(header, flush=True)
    print("=" * len(header), flush=True)
    print(format_report(per_class, summary), flush=True)
    print("=" * len(header), flush=True)

    print("\n" + format_confusion(cm, classes), flush=True)

    if "val_accuracy" in ckpt:
        gap = ckpt["val_accuracy"] - summary["accuracy"]
        print(f"\nvalidation {ckpt['val_accuracy'] * 100:.2f}%  ->  "
              f"test {summary['accuracy'] * 100:.2f}%   (gap {gap * 100:+.2f} pts)",
              flush=True)
        print("  a modest drop is normal: the configuration was chosen on validation.",
              flush=True)

    w = summary["weighted"]
    print(f"\nTable 3 row:  accuracy {summary['accuracy'] * 100:.2f}  "
          f"precision {w['precision'] * 100:.2f}  recall {w['recall'] * 100:.2f}  "
          f"F1 {w['f1'] * 100:.2f}", flush=True)

    print(f"\nTEST ACCURACY: {summary['accuracy'] * 100:.2f}%", flush=True)


if __name__ == "__main__":
    main()
